# Multi-Modal Document Intelligence Agent
### Invoice & Form Processing — LangGraph Multi-Agent System

**Complete self-contained Colab** — no GitHub required. All source files are embedded.

---

## Pipeline
```
PDF / Image
   → Ingestion (pdf2image + OpenCV deskew)
   → Extraction Agent (VLM: Gemini / Claude)
       ↓ low confidence?
   → OCR Fallback (Tesseract)
   → Validation Agent (math + date + IBAN + vendor fuzzy-match)
       ↓ failed fields?
   → Auto-Correction Agent (crop & re-query VLM, max 2×)
       ↓ still failing / low confidence?
   → Human Review Queue
   → SQLite Storage + JSON/CSV Export
```

## Quick Start
1. **Runtime → Run all** (or run cells top-to-bottom)
2. Add your API key in **Step 2** (Gemini is free — recommended)
3. Upload an invoice in **Step 5**
4. (Optional) Launch the full Streamlit UI in **Step 7**


## Step 1 — Install Dependencies


In [ ]:
# System packages: Tesseract OCR + Poppler (PDF rendering)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils libgl1
!tesseract --version 2>&1 | head -1
print('System dependencies ready.')


In [ ]:
# Python packages — core stack
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    pydantic>=2.0.0 \
    python-dotenv \
    streamlit>=1.35.0 \
    pdf2image \
    Pillow \
    pytesseract \
    google-genai>=1.0.0 \
    anthropic>=0.40.0 \
    opencv-python-headless \
    rapidfuzz \
    pandas \
    einops

# chromadb is installed separately — it can conflict with pre-installed
# Colab packages and needs to be visible if it fails
!pip install chromadb

# Verify critical packages imported correctly
import importlib, sys
missing = []
for pkg in ['chromadb', 'rapidfuzz', 'langgraph', 'pytesseract', 'pdf2image', 'google.genai']:
    try:
        importlib.import_module(pkg.split('.')[0])
    except ImportError:
        missing.append(pkg)
if missing:
    print(f'WARNING — failed to import: {missing}')
    print('Run:  !pip install ' + ' '.join(missing))
else:
    print('All packages installed and importable.')


## Step 2 — API Key Configuration

Choose **one** backend:

| Backend | Key | Cost |
|---------|-----|------|
| **Gemini** *(recommended)* | Free at [aistudio.google.com](https://aistudio.google.com/app/apikey) | Free (30 RPM) |
| **Claude** | [console.anthropic.com](https://console.anthropic.com/) | Paid |

> Add your key via the **Colab Secrets panel** (🔑 in left sidebar) as `GEMINI_API_KEY`
> or `ANTHROPIC_API_KEY`, then run this cell.


In [ ]:
import os, getpass

# ── Choose backend ────────────────────────────────────────────────────────
VLM_BACKEND = 'gemini'   # Change to 'claude' if you have an Anthropic key

def _load_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ''
    except Exception:
        return ''

if VLM_BACKEND == 'gemini':
    GEMINI_API_KEY = _load_secret('GEMINI_API_KEY') or getpass.getpass('Gemini API key: ')
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    os.environ['VLM_BACKEND']    = 'gemini'
    os.environ['GEMINI_MODEL']   = 'gemini-2.5-flash'
    print(f'Gemini backend configured. Key: ...{GEMINI_API_KEY[-6:]}')
else:
    ANTHROPIC_API_KEY = _load_secret('ANTHROPIC_API_KEY') or getpass.getpass('Anthropic API key: ')
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
    os.environ['VLM_BACKEND']       = 'claude'
    os.environ['CLAUDE_MODEL']      = 'claude-sonnet-4-6'
    print(f'Claude backend configured. Key: ...{ANTHROPIC_API_KEY[-6:]}')

# Shared config
os.environ['SQLITE_PATH']       = '/content/capstone/data/invoices.db'
os.environ['CHROMA_PERSIST_DIR'] = '/content/capstone/data/chroma'
print('Configuration set.')


## Step 3 — Write Project Source Files

All source files are embedded in this notebook. This cell extracts them to `/content/capstone/`.


In [ ]:
import base64, pathlib, os

FILES = {
    "/content/capstone/config.py": "aW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSBkb3RlbnYgaW1wb3J0IGxvYWRfZG90ZW52Cgpsb2FkX2RvdGVudigpCgpCQVNFX0RJUiA9IFBhdGgoX19maWxlX18pLnBhcmVudAoKIyDilIDilIAgVkxNIEJhY2tlbmQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgImdlbWluaSIgIOKGkiBHb29nbGUgR2VtaW5pIChGUkVFIOKAlCByZWNvbW1lbmRlZCBmb3Igc3R1ZGVudHMsIGp1c3QgbmVlZHMgR29vZ2xlIGFjY291bnQpCiMgImNsYXVkZSIgIOKGkiBBbnRocm9waWMgQ2xhdWRlIChwYWlkIEFQSSBrZXkgcmVxdWlyZWQpCiMgImludGVybnZsIiAvICJsbGF2YSIg4oaSIGxvY2FsIEdQVSBtb2RlbCAoSHVnZ2luZ0ZhY2UsIG5vIEFQSSBrZXksIG5lZWRzIEdQVSkKVkxNX0JBQ0tFTkQgPSBvcy5nZXRlbnYoIlZMTV9CQUNLRU5EIiwgImdlbWluaSIpCgojIOKUgOKUgCBHb29nbGUgR2VtaW5pIChGcmVlIFRpZXIpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIEdldCB5b3VyIGZyZWUgQVBJIGtleSBhdDogaHR0cHM6Ly9haXN0dWRpby5nb29nbGUuY29tL2FwcC9hcGlrZXkKIyBObyBjcmVkaXQgY2FyZCByZXF1aXJlZCDigJQganVzdCBhIEdvb2dsZSBhY2NvdW50CkdFTUlOSV9BUElfS0VZID0gb3MuZ2V0ZW52KCJHRU1JTklfQVBJX0tFWSIsICIiKQpHRU1JTklfTU9ERUwgICA9IG9zLmdldGVudigiR0VNSU5JX01PREVMIiwgImdlbWluaS0yLjUtZmxhc2giKQoKIyDilIDilIAgQW50aHJvcGljIENsYXVkZSAoT3B0aW9uYWwpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApBTlRIUk9QSUNfQVBJX0tFWSA9IG9zLmdldGVudigiQU5USFJPUElDX0FQSV9LRVkiLCAiIikKQ0xBVURFX01PREVMICAgICAgPSBvcy5nZXRlbnYoIkNMQVVERV9NT0RFTCIsICJjbGF1ZGUtc29ubmV0LTQtNiIpCgojIOKUgOKUgCBMb2NhbCBWTE0gKE9wdGlvbmFsIOKAlCBuZWVkcyBHUFUpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApMT0NBTF9WTE1fTU9ERUwgID0gb3MuZ2V0ZW52KCJMT0NBTF9WTE1fTU9ERUwiLCAiSW50ZXJuVkwyLThCIikKTE9DQUxfVkxNX0RFVklDRSA9IG9zLmdldGVudigiTE9DQUxfVkxNX0RFVklDRSIsICJjdWRhIikKCiMg4pSA4pSAIFRlc3NlcmFjdCBPQ1Ig4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAClRFU1NFUkFDVF9DTUQgPSBvcy5nZXRlbnYoIlRFU1NFUkFDVF9DTUQiLCAidGVzc2VyYWN0IikKT0NSX0RQSSAgICAgICA9IGludChvcy5nZXRlbnYoIk9DUl9EUEkiLCAiMzAwIikpCgojIOKUgOKUgCBDaHJvbWFEQiAodmVuZG9yIG1hdGNoaW5nKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKQ0hST01BX1BFUlNJU1RfRElSID0gb3MuZ2V0ZW52KCJDSFJPTUFfUEVSU0lTVF9ESVIiLCBzdHIoQkFTRV9ESVIgLyAiZGF0YSIgLyAiY2hyb21hIikpClZFTkRPUl9DT0xMRUNUSU9OICA9ICJ2ZW5kb3JzIgoKIyDilIDilIAgU1FMaXRlIHN0b3JhZ2Ug4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAClNRTElURV9QQVRIID0gb3MuZ2V0ZW52KCJTUUxJVEVfUEFUSCIsIHN0cihCQVNFX0RJUiAvICJkYXRhIiAvICJpbnZvaWNlcy5kYiIpKQoKIyDilIDilIAgVmFsaWRhdGlvbiB0aHJlc2hvbGRzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApUT1RBTF9UT0xFUkFOQ0UgICAgICAgICAgPSBmbG9hdChvcy5nZXRlbnYoIlRPVEFMX1RPTEVSQU5DRSIsICIwLjAxIikpCkZVWlpZX1ZFTkRPUl9USFJFU0hPTEQgICA9IGludChvcy5nZXRlbnYoIkZVWlpZX1ZFTkRPUl9USFJFU0hPTEQiLCAiODAiKSkKRVhUUkFDVElPTl9DT05GX1RIUkVTSE9MRCA9IGZsb2F0KG9zLmdldGVudigiRVhUUkFDVElPTl9DT05GX1RIUkVTSE9MRCIsICIwLjcwIikpCkFVVE9fQVBQUk9WRV9USFJFU0hPTEQgICA9IGZsb2F0KG9zLmdldGVudigiQVVUT19BUFBST1ZFX1RIUkVTSE9MRCIsICIwLjg1IikpCgojIOKUgOKUgCBBdXRvLWNvcnJlY3Rpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACk1BWF9DT1JSRUNUSU9OX0FUVEVNUFRTID0gaW50KG9zLmdldGVudigiTUFYX0NPUlJFQ1RJT05fQVRURU1QVFMiLCAiMiIpKQpDUk9QX1BBRERJTkdfUFggICAgICAgICA9IGludChvcy5nZXRlbnYoIkNST1BfUEFERElOR19QWCIsICIyMCIpKQoKCiMg4pSA4pSAIFJ1bnRpbWUgdXBkYXRlIGhlbHBlcnMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgYXBwbHkodXBkYXRlczogZGljdCkgLT4gTm9uZToKICAgICIiIlVwZGF0ZSBjb25maWcgdmFsdWVzIGluIG1lbW9yeSBBTkQgcGVyc2lzdCB0byAuZW52IOKAlCBubyByZXN0YXJ0IG5lZWRlZC4iIiIKICAgIGltcG9ydCBzeXMKICAgIG1vZHVsZSA9IHN5cy5tb2R1bGVzW19fbmFtZV9fXQoKICAgICMgMS4gVXBkYXRlIG1vZHVsZS1sZXZlbCBhdHRyaWJ1dGVzIHNvIGFsbCBpbXBvcnRlcnMgc2VlIHRoZSBuZXcgdmFsdWVzCiAgICBmb3Iga2V5LCB2YWx1ZSBpbiB1cGRhdGVzLml0ZW1zKCk6CiAgICAgICAgb3MuZW52aXJvbltrZXldID0gc3RyKHZhbHVlKQogICAgICAgIGlmIGhhc2F0dHIobW9kdWxlLCBrZXkpOgogICAgICAgICAgICBjdXJyZW50ID0gZ2V0YXR0cihtb2R1bGUsIGtleSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2V0YXR0cihtb2R1bGUsIGtleSwgdHlwZShjdXJyZW50KSh2YWx1ZSkpCiAgICAgICAgICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgIHNldGF0dHIobW9kdWxlLCBrZXksIHZhbHVlKQoKICAgICMgMi4gUGVyc2lzdCB0byAuZW52IHNvIHZhbHVlcyBzdXJ2aXZlIGEgcmVzdGFydAogICAgZW52X3BhdGggPSBCQVNFX0RJUiAvICIuZW52IgogICAgdHJ5OgogICAgICAgIGV4aXN0aW5nOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICAgICAgaWYgZW52X3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGVudl9wYXRoLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKToKICAgICAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICAgICAgICAgIGlmIGxpbmUgYW5kIG5vdCBsaW5lLnN0YXJ0c3dpdGgoIiMiKSBhbmQgIj0iIGluIGxpbmU6CiAgICAgICAgICAgICAgICAgICAgaywgXywgdiA9IGxpbmUucGFydGl0aW9uKCI9IikKICAgICAgICAgICAgICAgICAgICBleGlzdGluZ1trLnN0cmlwKCldID0gdi5zdHJpcCgpCiAgICAgICAgZXhpc3RpbmcudXBkYXRlKHtrOiBzdHIodikgZm9yIGssIHYgaW4gdXBkYXRlcy5pdGVtcygpfSkKICAgICAgICBlbnZfcGF0aC53cml0ZV90ZXh0KAogICAgICAgICAgICAiXG4iLmpvaW4oZiJ7a309e3Z9IiBmb3IgaywgdiBpbiBleGlzdGluZy5pdGVtcygpKSArICJcbiIKICAgICAgICApCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MgICAjIHdyaXRlIGZhaWx1cmUgaXMgbm9uLWZhdGFsIOKAlCBpbi1tZW1vcnkgdXBkYXRlIHN0aWxsIHdvcmtzCg==",
    "/content/capstone/models/__init__.py": "",
    "/content/capstone/models/schemas.py": "IiIiUHlkYW50aWMgbW9kZWxzIGZvciBEb2MgQWdlbnQg4oCUIGZsZXhpYmxlIGRvY3VtZW50LXR5cGUtYWdub3N0aWMgZXh0cmFjdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZW51bSBpbXBvcnQgRW51bQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBPcHRpb25hbAoKZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBGaWVsZAoKCiMg4pSA4pSAIERvY3VtZW50IHR5cGVzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgRG9jVHlwZShzdHIsIEVudW0pOgogICAgSU5WT0lDRSAgICAgICAgID0gImludm9pY2UiCiAgICBSRUNFSVBUICAgICAgICAgPSAicmVjZWlwdCIKICAgIFBVUkNIQVNFX09SREVSICA9ICJwdXJjaGFzZV9vcmRlciIKICAgIEJBTktfU1RBVEVNRU5UICA9ICJiYW5rX3N0YXRlbWVudCIKICAgIEVYUEVOU0VfUkVQT1JUICA9ICJleHBlbnNlX3JlcG9ydCIKICAgIFFVT1RFICAgICAgICAgICA9ICJxdW90ZSIKICAgIERFTElWRVJZX05PVEUgICA9ICJkZWxpdmVyeV9ub3RlIgogICAgQ09OVFJBQ1QgICAgICAgID0gImNvbnRyYWN0IgogICAgRk9STSAgICAgICAgICAgID0gImZvcm0iCiAgICBPVEhFUiAgICAgICAgICAgPSAib3RoZXIiCiAgICBVTktOT1dOICAgICAgICAgPSAidW5rbm93biIKCgpjbGFzcyBWYWxpZGF0aW9uU3RhdHVzKHN0ciwgRW51bSk6CiAgICBWQUxJRCAgICA9ICJ2YWxpZCIKICAgIEZBSUxFRCAgID0gImZhaWxlZCIKICAgIENPUlJFQ1RFRD0gImNvcnJlY3RlZCIKICAgIFBFTkRJTkcgID0gInBlbmRpbmdfcmV2aWV3IgogICAgUkVKRUNURUQgPSAicmVqZWN0ZWQiCgoKIyDilIDilIAgQ29yZSBmbGV4aWJsZSBkYXRhIG1vZGVsIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgRG9jdW1lbnREYXRhKEJhc2VNb2RlbCk6CiAgICAiIiJGbGV4aWJsZSwgc2NoZW1hLWZyZWUgY29udGFpbmVyIGZvciBhbnkgZG9jdW1lbnQgdHlwZS4KCiAgICBSYXRoZXIgdGhhbiBoYXJkY29kaW5nIGZpZWxkIG5hbWVzLCB0aGUgVkxNIGV4dHJhY3RzIGV2ZXJ5dGhpbmcgaXQgZmluZHMKICAgIGludG8gYGZpZWxkc2AgYXMgYSBwbGFpbiBkaWN0LiAgSGVscGVyIHByb3BlcnRpZXMgcHJvdmlkZSBjb252ZW5pZW50IGFjY2VzcwogICAgdG8gdGhlIG1vc3QgY29tbW9uIGZpbmFuY2lhbCAvIGlkZW50aWZpY2F0aW9uIGZpZWxkcyByZWdhcmRsZXNzIG9mIHdoYXQKICAgIGV4YWN0IGtleSB0aGUgbW9kZWwgdXNlZC4KICAgICIiIgoKICAgIGRvY190eXBlOiAgICAgICAgIHN0ciAgICAgICAgICAgID0gInVua25vd24iCiAgICBkb2Nfc3VidHlwZTogICAgICBPcHRpb25hbFtzdHJdICA9IE5vbmUgICAgICAgICAgIyBlLmcuICJ0YXhfaW52b2ljZSIsICJwcm9fZm9ybWEiCiAgICBmaWVsZHM6ICAgICAgICAgICBkaWN0ICAgICAgICAgICA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgbGluZV9pdGVtczogICAgICAgbGlzdFtkaWN0XSAgICAgPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGV4dHJhY3Rpb25fbm90ZXM6IE9wdGlvbmFsW3N0cl0gID0gTm9uZQoKICAgICMg4pSA4pSAIENvbnZlbmllbmNlIGFjY2Vzc29ycyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiAgICBkZWYgZ2V0KHNlbGYsICpuYW1lczogc3RyLCBkZWZhdWx0OiBBbnkgPSBOb25lKSAtPiBBbnk6CiAgICAgICAgIiIiUmV0dXJuIGZpcnN0IG5vbi1Ob25lIHZhbHVlIG1hdGNoaW5nIGFueSBvZiB0aGUgZ2l2ZW4gZmllbGQgbmFtZXMuIiIiCiAgICAgICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICAgICAgICAgIHYgPSBzZWxmLmZpZWxkcy5nZXQobmFtZSkKICAgICAgICAgICAgaWYgdiBpcyBub3QgTm9uZSBhbmQgdiAhPSAiIjoKICAgICAgICAgICAgICAgIHJldHVybiB2CiAgICAgICAgcmV0dXJuIGRlZmF1bHQKCiAgICBkZWYgX3RvX2Zsb2F0KHNlbGYsIHY6IEFueSkgLT4gT3B0aW9uYWxbZmxvYXRdOgogICAgICAgIGlmIHYgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBmbG9hdChzdHIodikucmVwbGFjZSgiLCIsICIiKS5yZXBsYWNlKCIkIiwgIiIpLnJlcGxhY2UoIuKCrCIsICIiKS5yZXBsYWNlKCLCoyIsICIiKS5zdHJpcCgpKQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVHlwZUVycm9yKToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCiAgICAjIENvbW1vbiBpZGVudGlmaWNhdGlvbgogICAgQHByb3BlcnR5CiAgICBkZWYgZG9jdW1lbnRfbnVtYmVyKHNlbGYpIC0+IE9wdGlvbmFsW3N0cl06CiAgICAgICAgcmV0dXJuIHNlbGYuZ2V0KCJpbnZvaWNlX251bWJlciIsICJyZWNlaXB0X251bWJlciIsICJvcmRlcl9udW1iZXIiLAogICAgICAgICAgICAgICAgICAgICAgICAiZG9jdW1lbnRfbnVtYmVyIiwgInJlZmVyZW5jZV9udW1iZXIiLCAicG9fbnVtYmVyIiwgImJpbGxfbnVtYmVyIikKCiAgICBAcHJvcGVydHkKICAgIGRlZiB2ZW5kb3JfbmFtZShzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmdldCgidmVuZG9yX25hbWUiLCAidmVuZG9yIiwgInN1cHBsaWVyX25hbWUiLCAic3VwcGxpZXIiLAogICAgICAgICAgICAgICAgICAgICAgICAibWVyY2hhbnQiLCAiY29tcGFueV9uYW1lIiwgImlzc3VlZF9ieSIsICJzZWxsZXIiLCAiZnJvbSIpCgogICAgQHByb3BlcnR5CiAgICBkZWYgY3VzdG9tZXJfbmFtZShzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmdldCgiY3VzdG9tZXJfbmFtZSIsICJjdXN0b21lciIsICJjbGllbnRfbmFtZSIsICJjbGllbnQiLAogICAgICAgICAgICAgICAgICAgICAgICAiYmlsbGVkX3RvIiwgImJpbGxfdG8iLCAiYnV5ZXIiLCAic2hpcF90byIpCgogICAgQHByb3BlcnR5CiAgICBkZWYgZG9jdW1lbnRfZGF0ZShzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmdldCgiaW52b2ljZV9kYXRlIiwgInJlY2VpcHRfZGF0ZSIsICJvcmRlcl9kYXRlIiwgImlzc3VlX2RhdGUiLAogICAgICAgICAgICAgICAgICAgICAgICAiZGF0ZSIsICJ0cmFuc2FjdGlvbl9kYXRlIiwgInN0YXRlbWVudF9kYXRlIikKCiAgICBAcHJvcGVydHkKICAgIGRlZiBkdWVfZGF0ZShzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmdldCgiZHVlX2RhdGUiLCAicGF5bWVudF9kdWUiLCAicGF5bWVudF9kdWVfZGF0ZSIsICJleHBpcnlfZGF0ZSIpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdG90YWxfYW1vdW50KHNlbGYpIC0+IE9wdGlvbmFsW2Zsb2F0XToKICAgICAgICByZXR1cm4gc2VsZi5fdG9fZmxvYXQoCiAgICAgICAgICAgIHNlbGYuZ2V0KCJ0b3RhbF9hbW91bnQiLCAidG90YWwiLCAiYW1vdW50X2R1ZSIsICJiYWxhbmNlX2R1ZSIsCiAgICAgICAgICAgICAgICAgICAgICJncmFuZF90b3RhbCIsICJpbnZvaWNlX3RvdGFsIiwgImFtb3VudF9wYXlhYmxlIikKICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgc3VidG90YWwoc2VsZikgLT4gT3B0aW9uYWxbZmxvYXRdOgogICAgICAgIHJldHVybiBzZWxmLl90b19mbG9hdCgKICAgICAgICAgICAgc2VsZi5nZXQoInN1YnRvdGFsIiwgInN1Yl90b3RhbCIsICJuZXRfYW1vdW50IiwgIm5ldCIsICJ0YXhhYmxlX2Ftb3VudCIpCiAgICAgICAgKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHRheF9hbW91bnQoc2VsZikgLT4gT3B0aW9uYWxbZmxvYXRdOgogICAgICAgIHJldHVybiBzZWxmLl90b19mbG9hdCgKICAgICAgICAgICAgc2VsZi5nZXQoInRheF9hbW91bnQiLCAidGF4IiwgInZhdF9hbW91bnQiLCAiZ3N0X2Ftb3VudCIsICJoc3RfYW1vdW50IiwKICAgICAgICAgICAgICAgICAgICAgInNhbGVzX3RheCIsICJ2YWx1ZV9hZGRlZF90YXgiKQogICAgICAgICkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBjdXJyZW5jeShzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmdldCgiY3VycmVuY3kiLCAiY3VycmVuY3lfY29kZSIpCgogICAgQHByb3BlcnR5CiAgICBkZWYgaWJhbihzZWxmKSAtPiBPcHRpb25hbFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmdldCgiaWJhbiIsICJiYW5rX2liYW4iKQoKCiMg4pSA4pSAIFBpcGVsaW5lIHJlc3VsdCBtb2RlbHMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFzcyBGaWVsZFZhbGlkYXRpb24oQmFzZU1vZGVsKToKICAgIGZpZWxkOiAgICAgICAgICAgc3RyCiAgICBzdGF0dXM6ICAgICAgICAgIFZhbGlkYXRpb25TdGF0dXMKICAgIG1lc3NhZ2U6ICAgICAgICAgT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgIG9yaWdpbmFsX3ZhbHVlOiAgT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgIGNvcnJlY3RlZF92YWx1ZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUKCgpjbGFzcyBFeHRyYWN0aW9uUmVzdWx0KEJhc2VNb2RlbCk6CiAgICByYXdfb2NyX3RleHQ6ICAgc3RyCiAgICBleHRyYWN0ZWRfZGF0YTogRG9jdW1lbnREYXRhCiAgICBjb25maWRlbmNlOiAgICAgZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCwgZGVmYXVsdD0wLjApCiAgICB2bG1fcmVzcG9uc2U6ICAgT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgIG9jcl91c2VkOiAgICAgICBib29sID0gRmFsc2UKCgpjbGFzcyBWYWxpZGF0aW9uUmVzdWx0KEJhc2VNb2RlbCk6CiAgICBpc192YWxpZDogICAgICAgICAgIGJvb2wKICAgIGZpZWxkX3ZhbGlkYXRpb25zOiAgbGlzdFtGaWVsZFZhbGlkYXRpb25dID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBmYWlsZWRfZmllbGRzOiAgICAgIGxpc3Rbc3RyXSAgICAgICAgICAgICA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgb3ZlcmFsbF9jb25maWRlbmNlOiBmbG9hdCAgICAgICAgICAgICAgICAgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCwgZGVmYXVsdD0wLjApCgoKY2xhc3MgUHJvY2Vzc2luZ1JlY29yZChCYXNlTW9kZWwpOgogICAgZG9jdW1lbnRfaWQ6ICAgICAgICAgICBzdHIKICAgIHNvdXJjZV9wYXRoOiAgICAgICAgICAgc3RyCiAgICBkb2NfdHlwZTogICAgICAgICAgICAgIHN0ciAgPSAidW5rbm93biIKICAgIHJhd19vY3JfdGV4dDogICAgICAgICAgc3RyCiAgICB2bG1fZXh0cmFjdGlvbjogICAgICAgIGRpY3QKICAgIGNvcnJlY3Rpb25zX2FwcGxpZWQ6ICAgbGlzdFtkaWN0XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZmluYWxfZGF0YTogICAgICAgICAgICBkaWN0CiAgICB2YWxpZGF0aW9uX3N0YXR1czogICAgIFZhbGlkYXRpb25TdGF0dXMKICAgIGV4dHJhY3Rpb25fY29uZmlkZW5jZTogZmxvYXQgPSAwLjAKICAgIHByb2Nlc3Npbmdfbm90ZXM6ICAgICAgbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCg==",
    "/content/capstone/utils/__init__.py": "",
    "/content/capstone/utils/image_processing.py": "IiIiUGlsbG93LWJhc2VkIGltYWdlIHByZXByb2Nlc3Npbmc6IGRlc2tldyBhbmQgY29udHJhc3QgZW5oYW5jZW1lbnQuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZUVuaGFuY2UsIEltYWdlRmlsdGVyCgoKZGVmIGRlc2tldyhpbWc6IEltYWdlLkltYWdlKSAtPiBJbWFnZS5JbWFnZToKICAgICIiIkRldGVjdCBhbmQgY29ycmVjdCBza2V3IHVzaW5nIHByb2plY3Rpb24gcHJvZmlsZSBtZXRob2QuIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IGN2MgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHJldHVybiBpbWcgICMgc2tpcCBzaWxlbnRseSBpZiBvcGVuY3Ygbm90IGluc3RhbGxlZAoKICAgIGdyYXkgPSBucC5hcnJheShpbWcuY29udmVydCgiTCIpKQogICAgXywgYmluYXJ5ID0gY3YyLnRocmVzaG9sZChncmF5LCAwLCAyNTUsIGN2Mi5USFJFU0hfQklOQVJZX0lOViArIGN2Mi5USFJFU0hfT1RTVSkKICAgIGNvb3JkcyA9IG5wLmNvbHVtbl9zdGFjayhucC53aGVyZShiaW5hcnkgPiAwKSkKICAgIGlmIGxlbihjb29yZHMpIDwgMTA6CiAgICAgICAgcmV0dXJuIGltZwogICAgYW5nbGUgPSBjdjIubWluQXJlYVJlY3QoY29vcmRzKVstMV0KICAgIGlmIGFuZ2xlIDwgLTQ1OgogICAgICAgIGFuZ2xlID0gOTAgKyBhbmdsZQogICAgaWYgYWJzKGFuZ2xlKSA8IDAuNToKICAgICAgICByZXR1cm4gaW1nCiAgICByZXR1cm4gaW1nLnJvdGF0ZSgtYW5nbGUsIGV4cGFuZD1UcnVlLCBmaWxsY29sb3I9KDI1NSwgMjU1LCAyNTUpKQoKCmRlZiBlbmhhbmNlX2NvbnRyYXN0KGltZzogSW1hZ2UuSW1hZ2UsIGZhY3RvcjogZmxvYXQgPSAxLjUpIC0+IEltYWdlLkltYWdlOgogICAgIiIiQXBwbHkgY29udHJhc3QgZW5oYW5jZW1lbnQgYW5kIG1pbGQgc2hhcnBlbmluZy4iIiIKICAgIGltZyA9IEltYWdlRW5oYW5jZS5Db250cmFzdChpbWcpLmVuaGFuY2UoZmFjdG9yKQogICAgaW1nID0gSW1hZ2VFbmhhbmNlLlNoYXJwbmVzcyhpbWcpLmVuaGFuY2UoMS4zKQogICAgcmV0dXJuIGltZwoKCmRlZiBpbWFnZV90b19iYXNlNjQoaW1nOiBJbWFnZS5JbWFnZSwgZm10OiBzdHIgPSAiUE5HIikgLT4gc3RyOgogICAgIiIiRW5jb2RlIGEgUElMIGltYWdlIGFzIGEgYmFzZTY0IHN0cmluZyBmb3IgQVBJIGNhbGxzLiIiIgogICAgaW1wb3J0IGJhc2U2NAogICAgaW1wb3J0IGlvCgogICAgYnVmID0gaW8uQnl0ZXNJTygpCiAgICBpbWcuc2F2ZShidWYsIGZvcm1hdD1mbXQpCiAgICByZXR1cm4gYmFzZTY0LmI2NGVuY29kZShidWYuZ2V0dmFsdWUoKSkuZGVjb2RlKCJ1dGYtOCIpCg==",
    "/content/capstone/utils/vendor_matcher.py": "IiIiVmVuZG9yIG5hbWUgc2VtYW50aWMgbWF0Y2hpbmcgdXNpbmcgQ2hyb21hREIgKyByYXBpZGZ1enouIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBUdXBsZQoKaW1wb3J0IGNocm9tYWRiCmZyb20gY2hyb21hZGIudXRpbHMgaW1wb3J0IGVtYmVkZGluZ19mdW5jdGlvbnMKZnJvbSByYXBpZGZ1enogaW1wb3J0IGZ1enosIHByb2Nlc3MgYXMgcmZwcm9jZXNzCgpmcm9tIGNvbmZpZyBpbXBvcnQgQ0hST01BX1BFUlNJU1RfRElSLCBGVVpaWV9WRU5ET1JfVEhSRVNIT0xELCBWRU5ET1JfQ09MTEVDVElPTgoKX1ZFTkRPUlNfRklMRSA9IFBhdGgoX19maWxlX18pLnBhcmVudC5wYXJlbnQgLyAiZGF0YSIgLyAidmVuZG9ycy50eHQiCgoKY2xhc3MgVmVuZG9yTWF0Y2hlcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLl9jbGllbnQgPSBjaHJvbWFkYi5QZXJzaXN0ZW50Q2xpZW50KHBhdGg9Q0hST01BX1BFUlNJU1RfRElSKQogICAgICAgIHNlbGYuX2VmICAgICA9IGVtYmVkZGluZ19mdW5jdGlvbnMuRGVmYXVsdEVtYmVkZGluZ0Z1bmN0aW9uKCkKICAgICAgICBzZWxmLl9jb2wgICAgPSBzZWxmLl9jbGllbnQuZ2V0X29yX2NyZWF0ZV9jb2xsZWN0aW9uKAogICAgICAgICAgICBuYW1lPVZFTkRPUl9DT0xMRUNUSU9OLAogICAgICAgICAgICBlbWJlZGRpbmdfZnVuY3Rpb249c2VsZi5fZWYsCiAgICAgICAgKQogICAgICAgIHNlbGYuX3NlZWRfaWZfZW1wdHkoKQoKICAgIGRlZiBfc2VlZF9pZl9lbXB0eShzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYuX2NvbC5jb3VudCgpID09IDAgYW5kIF9WRU5ET1JTX0ZJTEUuZXhpc3RzKCk6CiAgICAgICAgICAgIHZlbmRvcnMgPSBbdi5zdHJpcCgpIGZvciB2IGluIF9WRU5ET1JTX0ZJTEUucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpIGlmIHYuc3RyaXAoKV0KICAgICAgICAgICAgc2VsZi5fYWRkKHZlbmRvcnMpCgogICAgZGVmIF9hZGQoc2VsZiwgdmVuZG9yczogbGlzdFtzdHJdKSAtPiBOb25lOgogICAgICAgIGV4aXN0aW5nID0gc2V0KHNlbGYuX2NvbC5nZXQoKVsiZG9jdW1lbnRzIl0gb3IgW10pCiAgICAgICAgbmV3ID0gW3YgZm9yIHYgaW4gdmVuZG9ycyBpZiB2IG5vdCBpbiBleGlzdGluZ10KICAgICAgICBpZiBuZXc6CiAgICAgICAgICAgIG9mZnNldCA9IGxlbihleGlzdGluZykKICAgICAgICAgICAgc2VsZi5fY29sLmFkZCgKICAgICAgICAgICAgICAgIGRvY3VtZW50cz1uZXcsCiAgICAgICAgICAgICAgICBpZHM9W2YidmVuZG9yX3tvZmZzZXQgKyBpfSIgZm9yIGkgaW4gcmFuZ2UobGVuKG5ldykpXSwKICAgICAgICAgICAgKQoKICAgIGRlZiBhZGRfdmVuZG9ycyhzZWxmLCB2ZW5kb3JzOiBsaXN0W3N0cl0pIC0+IGludDoKICAgICAgICBiZWZvcmUgPSBzZWxmLl9jb2wuY291bnQoKQogICAgICAgIHNlbGYuX2FkZCh2ZW5kb3JzKQogICAgICAgIHJldHVybiBzZWxmLl9jb2wuY291bnQoKSAtIGJlZm9yZQoKICAgIGRlZiBsb2FkX2Zyb21fZmlsZShzZWxmLCBmaWxlcGF0aDogc3RyKSAtPiBOb25lOgogICAgICAgIHZlbmRvcnMgPSBbdi5zdHJpcCgpIGZvciB2IGluIFBhdGgoZmlsZXBhdGgpLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKSBpZiB2LnN0cmlwKCldCiAgICAgICAgYWRkZWQgPSBzZWxmLmFkZF92ZW5kb3JzKHZlbmRvcnMpCiAgICAgICAgcHJpbnQoZiJMb2FkZWQge2FkZGVkfSBuZXcgdmVuZG9yKHMpIGZyb20ge2ZpbGVwYXRofS4gVG90YWw6IHtzZWxmLl9jb2wuY291bnQoKX0iKQoKICAgIGRlZiBtYXRjaChzZWxmLCB2ZW5kb3JfbmFtZTogc3RyLCB0b3BfazogaW50ID0gMykgLT4gT3B0aW9uYWxbc3RyXToKICAgICAgICBpZiBzZWxmLl9jb2wuY291bnQoKSA9PSAwOgogICAgICAgICAgICByZXR1cm4gdmVuZG9yX25hbWUKICAgICAgICByZXN1bHRzID0gc2VsZi5fY29sLnF1ZXJ5KAogICAgICAgICAgICBxdWVyeV90ZXh0cz1bdmVuZG9yX25hbWVdLAogICAgICAgICAgICBuX3Jlc3VsdHM9bWluKHRvcF9rLCBzZWxmLl9jb2wuY291bnQoKSksCiAgICAgICAgKQogICAgICAgIGNhbmRpZGF0ZXMgPSByZXN1bHRzWyJkb2N1bWVudHMiXVswXSBpZiByZXN1bHRzWyJkb2N1bWVudHMiXSBlbHNlIFtdCiAgICAgICAgaWYgbm90IGNhbmRpZGF0ZXM6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgYmVzdCwgc2NvcmUsIF8gPSByZnByb2Nlc3MuZXh0cmFjdE9uZSh2ZW5kb3JfbmFtZSwgY2FuZGlkYXRlcywgc2NvcmVyPWZ1enoudG9rZW5fc29ydF9yYXRpbykKICAgICAgICByZXR1cm4gYmVzdCBpZiBzY29yZSA+PSBGVVpaWV9WRU5ET1JfVEhSRVNIT0xEIGVsc2UgTm9uZQoKICAgIGRlZiBpc19rbm93bl92ZW5kb3Ioc2VsZiwgdmVuZG9yX25hbWU6IHN0cikgLT4gVHVwbGVbYm9vbCwgT3B0aW9uYWxbc3RyXV06CiAgICAgICAgbWF0Y2hlZCA9IHNlbGYubWF0Y2godmVuZG9yX25hbWUpCiAgICAgICAgcmV0dXJuIChtYXRjaGVkIGlzIG5vdCBOb25lKSwgbWF0Y2hlZAo=",
    "/content/capstone/pipeline/__init__.py": "",
    "/content/capstone/pipeline/ingestion.py": "IiIiQ29udmVydHMgUERGL2ltYWdlIGZpbGVzIGludG8gcHJlcHJvY2Vzc2VkIFBJTCBpbWFnZXMgcmVhZHkgZm9yIE9DUiBhbmQgVkxNLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzaHV0aWwKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBVbmlvbgoKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpmcm9tIGNvbmZpZyBpbXBvcnQgT0NSX0RQSQpmcm9tIHV0aWxzLmltYWdlX3Byb2Nlc3NpbmcgaW1wb3J0IGRlc2tldywgZW5oYW5jZV9jb250cmFzdAoKCmRlZiBfZmluZF9wb3BwbGVyX3BhdGgoKSAtPiBVbmlvbltzdHIsIE5vbmVdOgogICAgIiIiUmV0dXJuIHRoZSBleHBsaWNpdCBwb3BwbGVyIGJpbiBkaXJlY3Rvcnkgc28gcGRmMmltYWdlIGFsd2F5cyBmaW5kcyBpdC4KCiAgICBPbiBtYWNPUywgSG9tZWJyZXcgaW5zdGFsbHMgdG8gL29wdC9ob21lYnJldy9iaW4gKEFwcGxlIFNpbGljb24pIG9yCiAgICAvdXNyL2xvY2FsL2JpbiAoSW50ZWwpLCBidXQgdmVudiBwcm9jZXNzZXMgbWF5IG5vdCBpbmhlcml0IHRoZSBmdWxsIFBBVEgsCiAgICBzbyB3ZSBhbHdheXMgcmV0dXJuIHRoZSBleHBsaWNpdCBwYXRoIHJhdGhlciB0aGFuIHJlbHlpbmcgb24gYXV0by1kZXRlY3QuCiAgICBPbiBMaW51eC9Db2xhYiwgcGRmdG9wcG0gaXMgb24gdGhlIHN5c3RlbSBQQVRIIGFmdGVyIGFwdC1nZXQgaW5zdGFsbC4KICAgICIiIgogICAgIyBQcmVmZXIgZXhwbGljaXQgSG9tZWJyZXcgcGF0aHMgb24gbWFjT1MgKG1vc3QgcmVsaWFibGUgdGhyb3VnaCB2ZW52KQogICAgZm9yIGNhbmRpZGF0ZSBpbiAoCiAgICAgICAgIi9vcHQvaG9tZWJyZXcvYmluIiwgICAjIEFwcGxlIFNpbGljb24gTWFjCiAgICAgICAgIi91c3IvbG9jYWwvYmluIiwgICAgICAjIEludGVsIE1hYwogICAgICAgICIvb3B0L2xvY2FsL2JpbiIsICAgICAgIyBNYWNQb3J0cwogICAgKToKICAgICAgICBpZiBQYXRoKGNhbmRpZGF0ZSwgInBkZnRvcHBtIikuZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybiBjYW5kaWRhdGUKICAgICMgTGludXggLyBDb2xhYiDigJQgb24gUEFUSCBhZnRlciBhcHQtZ2V0IGluc3RhbGwgcG9wcGxlci11dGlscwogICAgaWYgc2h1dGlsLndoaWNoKCJwZGZ0b3BwbSIpOgogICAgICAgIHJldHVybiBOb25lICAgIyBsZXQgcGRmMmltYWdlIHVzZSB0aGUgc3lzdGVtIFBBVEgKICAgIHJldHVybiBOb25lCgoKZGVmIGxvYWRfZG9jdW1lbnQocGF0aDogVW5pb25bc3RyLCBQYXRoXSkgLT4gbGlzdFtJbWFnZS5JbWFnZV06CiAgICAiIiJSZXR1cm4gYSBsaXN0IG9mIHByZXByb2Nlc3NlZCBQSUwgaW1hZ2VzIOKAlCBvbmUgcGVyIHBhZ2UgZm9yIFBERnMuIiIiCiAgICBwYXRoICAgPSBQYXRoKHBhdGgpCiAgICBzdWZmaXggPSBwYXRoLnN1ZmZpeC5sb3dlcigpCgogICAgaWYgc3VmZml4ID09ICIucGRmIjoKICAgICAgICByZXR1cm4gX3BkZl90b19pbWFnZXMocGF0aCkKICAgIGVsaWYgc3VmZml4IGluIHsiLnBuZyIsICIuanBnIiwgIi5qcGVnIiwgIi50aWZmIiwgIi50aWYiLCAiLmJtcCIsICIud2VicCJ9OgogICAgICAgIGltZyA9IEltYWdlLm9wZW4ocGF0aCkuY29udmVydCgiUkdCIikKICAgICAgICByZXR1cm4gW19wcmVwcm9jZXNzKGltZyldCiAgICBlbHNlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnN1cHBvcnRlZCBmaWxlIHR5cGU6IHtzdWZmaXh9LiBTdXBwb3J0ZWQ6IFBERiwgUE5HLCBKUEcsIFRJRkYsIEJNUCIpCgoKZGVmIF9wZGZfdG9faW1hZ2VzKHBhdGg6IFBhdGgpIC0+IGxpc3RbSW1hZ2UuSW1hZ2VdOgogICAgdHJ5OgogICAgICAgIGZyb20gcGRmMmltYWdlIGltcG9ydCBjb252ZXJ0X2Zyb21fcGF0aAogICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGU6CiAgICAgICAgcmFpc2UgSW1wb3J0RXJyb3IoInBkZjJpbWFnZSBub3QgaW5zdGFsbGVkLiBSdW46IHBpcCBpbnN0YWxsIHBkZjJpbWFnZSIpIGZyb20gZQoKICAgIHBvcHBsZXJfcGF0aCA9IF9maW5kX3BvcHBsZXJfcGF0aCgpCiAgICBrd2FyZ3MgPSB7ImRwaSI6IE9DUl9EUEl9CiAgICBpZiBwb3BwbGVyX3BhdGg6CiAgICAgICAga3dhcmdzWyJwb3BwbGVyX3BhdGgiXSA9IHBvcHBsZXJfcGF0aAoKICAgIHRyeToKICAgICAgICBwYWdlcyA9IGNvbnZlcnRfZnJvbV9wYXRoKHN0cihwYXRoKSwgKiprd2FyZ3MpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgZXJyID0gc3RyKGUpCiAgICAgICAgaWYgInBvcHBsZXIiIGluIGVyci5sb3dlcigpIG9yICJwYWdlIGNvdW50IiBpbiBlcnIubG93ZXIoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgIlBvcHBsZXIgbm90IGZvdW5kLiBJbnN0YWxsIGl0OlxuIgogICAgICAgICAgICAgICAgIiAgbWFjT1M6ICBicmV3IGluc3RhbGwgcG9wcGxlclxuIgogICAgICAgICAgICAgICAgIiAgVWJ1bnR1OiBzdWRvIGFwdC1nZXQgaW5zdGFsbCBwb3BwbGVyLXV0aWxzXG4iCiAgICAgICAgICAgICAgICAiICBDb2xhYjogICFhcHQtZ2V0IGluc3RhbGwgLXkgcG9wcGxlci11dGlscyIKICAgICAgICAgICAgKSBmcm9tIGUKICAgICAgICByYWlzZQoKICAgIHJldHVybiBbX3ByZXByb2Nlc3MocC5jb252ZXJ0KCJSR0IiKSkgZm9yIHAgaW4gcGFnZXNdCgoKZGVmIF9wcmVwcm9jZXNzKGltZzogSW1hZ2UuSW1hZ2UpIC0+IEltYWdlLkltYWdlOgogICAgaW1nID0gZGVza2V3KGltZykKICAgIGltZyA9IGVuaGFuY2VfY29udHJhc3QoaW1nKQogICAgcmV0dXJuIGltZwo=",
    "/content/capstone/pipeline/ocr.py": "IiIiVGVzc2VyYWN0IE9DUiBmYWxsYmFjayDigJQgdGV4dCBleHRyYWN0aW9uLCBib3VuZGluZyBib3hlcywgYW5kIFZMTSBtZXJnZS4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgc2h1dGlsDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbA0KDQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UNCg0KZGVmIF9maW5kX3Rlc3NlcmFjdCgpIC0+IE9wdGlvbmFsW3N0cl06DQogICAgZm9yIGNhbmRpZGF0ZSBpbiAoDQogICAgICAgICIvb3B0L2hvbWVicmV3L2Jpbi90ZXNzZXJhY3QiLA0KICAgICAgICAiL3Vzci9sb2NhbC9iaW4vdGVzc2VyYWN0IiwNCiAgICAgICAgIi91c3IvYmluL3Rlc3NlcmFjdCIsDQogICAgICAgICIvb3B0L2xvY2FsL2Jpbi90ZXNzZXJhY3QiLA0KICAgICk6DQogICAgICAgIGlmIFBhdGgoY2FuZGlkYXRlKS5leGlzdHMoKToNCiAgICAgICAgICAgIHJldHVybiBjYW5kaWRhdGUNCiAgICByZXR1cm4gc2h1dGlsLndoaWNoKCJ0ZXNzZXJhY3QiKQ0KDQp0cnk6DQogICAgaW1wb3J0IHB5dGVzc2VyYWN0DQogICAgZnJvbSBweXRlc3NlcmFjdCBpbXBvcnQgT3V0cHV0IGFzIF9PdXRwdXQNCg0KICAgIF90ZXNzX3BhdGggPSBfZmluZF90ZXNzZXJhY3QoKQ0KICAgIGlmIF90ZXNzX3BhdGg6DQogICAgICAgIHB5dGVzc2VyYWN0LnB5dGVzc2VyYWN0LnRlc3NlcmFjdF9jbWQgPSBfdGVzc19wYXRoDQoNCiAgICBweXRlc3NlcmFjdC5nZXRfdGVzc2VyYWN0X3ZlcnNpb24oKQ0KICAgIF9URVNTRVJBQ1RfQVZBSUxBQkxFID0gVHJ1ZQ0KDQpleGNlcHQgRXhjZXB0aW9uOg0KICAgIF9URVNTRVJBQ1RfQVZBSUxBQkxFID0gRmFsc2UNCg0KDQpkZWYgb2NyX2V4dHJhY3RfdGV4dChpbWFnZTogSW1hZ2UuSW1hZ2UpIC0+IHN0cjoNCiAgICBpZiBub3QgX1RFU1NFUkFDVF9BVkFJTEFCTEU6DQogICAgICAgIHJldHVybiAiIg0KICAgIHRyeToNCiAgICAgICAgcmV0dXJuIHB5dGVzc2VyYWN0LmltYWdlX3RvX3N0cmluZyhpbWFnZSwgY29uZmlnPSItLXBzbSA2IikNCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICByZXR1cm4gIiINCg0KDQpkZWYgb2NyX2V4dHJhY3Rfd2l0aF9ib3hlcyhpbWFnZTogSW1hZ2UuSW1hZ2UpIC0+IGRpY3Q6DQogICAgaWYgbm90IF9URVNTRVJBQ1RfQVZBSUxBQkxFOg0KICAgICAgICByZXR1cm4geyJ0ZXh0IjogW10sICJjb25mIjogW10sICJsZWZ0IjogW10sICJ0b3AiOiBbXSwgIndpZHRoIjogW10sICJoZWlnaHQiOiBbXX0NCiAgICB0cnk6DQogICAgICAgIHJldHVybiBweXRlc3NlcmFjdC5pbWFnZV90b19kYXRhKGltYWdlLCBvdXRwdXRfdHlwZT1fT3V0cHV0LkRJQ1QsIGNvbmZpZz0iLS1wc20gNiIpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgcmV0dXJuIHsidGV4dCI6IFtdLCAiY29uZiI6IFtdLCAibGVmdCI6IFtdLCAidG9wIjogW10sICJ3aWR0aCI6IFtdLCAiaGVpZ2h0IjogW119DQoNCg0KZGVmIGNyb3BfcmVnaW9uKGltYWdlOiBJbWFnZS5JbWFnZSwga2V5d29yZDogc3RyLCBwYWRkaW5nOiBpbnQgPSAyMCkgLT4gSW1hZ2UuSW1hZ2U6DQogICAgcmV0dXJuIGltYWdlDQoNCg0KZGVmIG1lcmdlX29jcl93aXRoX2V4dHJhY3Rpb24ob2NyX3RleHQsIGV4dHJhY3RlZCk6DQogICAgcmV0dXJuIGV4dHJhY3RlZA==",
    "/content/capstone/agents/__init__.py": "",
    "/content/capstone/agents/classification_agent.py": "IiIiQ2xhc3NpZmljYXRpb24gQWdlbnQg4oCUIGlkZW50aWZpZXMgZG9jdW1lbnQgdHlwZSBiZWZvcmUgZXh0cmFjdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCByZQoKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgppbXBvcnQgY29uZmlnCmZyb20gY29uZmlnIGltcG9ydCBBTlRIUk9QSUNfQVBJX0tFWSwgQ0xBVURFX01PREVMLCBHRU1JTklfQVBJX0tFWSwgR0VNSU5JX01PREVMCmZyb20gdXRpbHMuaW1hZ2VfcHJvY2Vzc2luZyBpbXBvcnQgaW1hZ2VfdG9fYmFzZTY0CgpfU0NIRU1BX01BUCA9IHsKICAgICJpbnZvaWNlIjogIkludm9pY2VEYXRhIiwKICAgICJyZWNlaXB0IjogIlJlY2VpcHREYXRhIiwKICAgICJmb3JtIjogICAgIkZvcm1EYXRhIiwKfQoKX0NMQVNTSUZJQ0FUSU9OX1BST01QVCA9ICIiIkxvb2sgYXQgdGhpcyBkb2N1bWVudCBpbWFnZSBhbmQgY2xhc3NpZnkgaXQgaW50byBleGFjdGx5IG9uZSB0eXBlLgoKUmV0dXJuIE9OTFkgdGhpcyBKU09OIG9iamVjdCwgbm90aGluZyBlbHNlOgp7ImRvY190eXBlIjogImludm9pY2UiLCAiY29uZmlkZW5jZSI6IDAuOTUsICJyZWFzb25pbmciOiAib25lIHNlbnRlbmNlIn0KClZhbGlkIGRvY190eXBlIHZhbHVlczoKLSAiaW52b2ljZSIgIDogZm9ybWFsIHZlbmRvciBpbnZvaWNlIHN1Y2ggYXMgYW4gQW1hem9uIEluZGlhIGludm9pY2UsIEIyQiBJbnZvaWNlLCB3aXRoIGludm9pY2UgbnVtYmVyLCBiaWxsLXRvIG5hbWUsIGxpbmUgaXRlbXMsIERhdGUsIFByaWNlCi0gInJlY2VpcHQiICA6IHBvaW50LW9mLXNhbGUgb3IgcHVyY2hhc2UgcmVjZWlwdCAoc2ltcGxlciwgb2Z0ZW4gZnJvbSByZXRhaWwpCi0gImZvcm0iICAgICA6IHN0cnVjdHVyZWQgZm9ybSBzdWNoIGFzIFctOSwgZXhwZW5zZSByZXBvcnQsIG9yIGludGFrZSBmb3JtCgpSdWxlczoKLSBSZXR1cm4gT05MWSB0aGUgSlNPTiwgbm8gZXhwbGFuYXRpb24gb3IgbWFya2Rvd24gZmVuY2VzLgotIGNvbmZpZGVuY2UgbXVzdCBiZSBhIGZsb2F0IGJldHdlZW4gMC4wIGFuZCAxLjAuCi0gZm9yIHVuY2xlYXIgaW1hZ2VzLCB0cmVhdCBkb2N1bWVudCB0eXBlID0gaW52b2ljZSwgY29uZmlkZW5jZSA9IDAuNSByZWFzb25pbmcgPSAiRGVmYXVsdCIKCiIiIgoKY2xhc3MgQ2xhc3NpZmljYXRpb25SZXN1bHQ6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZG9jX3R5cGU6IHN0ciwgY29uZmlkZW5jZTogZmxvYXQsIHNjaGVtYV9uYW1lOiBzdHIsIHJlYXNvbmluZzogc3RyID0gIiIpOgogICAgICAgIHNlbGYuZG9jX3R5cGUgICAgPSBkb2NfdHlwZQogICAgICAgIHNlbGYuY29uZmlkZW5jZSAgPSBjb25maWRlbmNlCiAgICAgICAgc2VsZi5zY2hlbWFfbmFtZSA9IHNjaGVtYV9uYW1lCiAgICAgICAgc2VsZi5yZWFzb25pbmcgICA9IHJlYXNvbmluZwoKICAgIGRlZiBfX3JlcHJfXyhzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIGYiQ2xhc3NpZmljYXRpb25SZXN1bHQoZG9jX3R5cGU9e3NlbGYuZG9jX3R5cGUhcn0sIGNvbmZpZGVuY2U9e3NlbGYuY29uZmlkZW5jZTouMmZ9KSIKCgpkZWYgcnVuKGltYWdlOiBJbWFnZS5JbWFnZSkgLT4gQ2xhc3NpZmljYXRpb25SZXN1bHQ6CiAgICBpZiBjb25maWcuVkxNX0JBQ0tFTkQgPT0gImdlbWluaSI6CiAgICAgICAgcmV0dXJuIF9jbGFzc2lmeV93aXRoX2dlbWluaShpbWFnZSkKICAgIGVsaWYgY29uZmlnLlZMTV9CQUNLRU5EID09ICJjbGF1ZGUiOgogICAgICAgIHJldHVybiBfY2xhc3NpZnlfd2l0aF9jbGF1ZGUoaW1hZ2UpCiAgICByZXR1cm4gX2ZhbGxiYWNrX2NsYXNzaWZ5KCkKCgojIOKUgOKUgCBHZW1pbmkgKGZyZWUsIHJlY29tbWVuZGVkKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmRlZiBfY2xhc3NpZnlfd2l0aF9nZW1pbmkoaW1hZ2U6IEltYWdlLkltYWdlKSAtPiBDbGFzc2lmaWNhdGlvblJlc3VsdDoKICAgIGZyb20gZ29vZ2xlIGltcG9ydCBnZW5haQogICAgZnJvbSBnb29nbGUuZ2VuYWkgaW1wb3J0IHR5cGVzCgogICAgY2xpZW50ID0gZ2VuYWkuQ2xpZW50KGFwaV9rZXk9R0VNSU5JX0FQSV9LRVkpCiAgICByZXNwb25zZSA9IGNsaWVudC5tb2RlbHMuZ2VuZXJhdGVfY29udGVudCgKICAgICAgICBtb2RlbD1HRU1JTklfTU9ERUwsCiAgICAgICAgY29udGVudHM9W2ltYWdlLCBfQ0xBU1NJRklDQVRJT05fUFJPTVBUXSwKICAgICAgICBjb25maWc9dHlwZXMuR2VuZXJhdGVDb250ZW50Q29uZmlnKAogICAgICAgICAgICBtYXhfb3V0cHV0X3Rva2Vucz01MDAsCiAgICAgICAgICAgIHRlbXBlcmF0dXJlPTAuMSwKICAgICAgICApLAogICAgKQogICAgcmV0dXJuIF9wYXJzZV9yZXNwb25zZShyZXNwb25zZS50ZXh0KQoKCiMg4pSA4pSAIENsYXVkZSAob3B0aW9uYWwsIHBhaWQpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIF9jbGFzc2lmeV93aXRoX2NsYXVkZShpbWFnZTogSW1hZ2UuSW1hZ2UpIC0+IENsYXNzaWZpY2F0aW9uUmVzdWx0OgogICAgaW1wb3J0IGFudGhyb3BpYwoKICAgIGNsaWVudCA9IGFudGhyb3BpYy5BbnRocm9waWMoYXBpX2tleT1BTlRIUk9QSUNfQVBJX0tFWSkKICAgIGI2NCA9IGltYWdlX3RvX2Jhc2U2NChpbWFnZSwgZm10PSJQTkciKQoKICAgIG1lc3NhZ2UgPSBjbGllbnQubWVzc2FnZXMuY3JlYXRlKAogICAgICAgIG1vZGVsPUNMQVVERV9NT0RFTCwKICAgICAgICBtYXhfdG9rZW5zPTUwMCwKICAgICAgICBtZXNzYWdlcz1bewogICAgICAgICAgICAicm9sZSI6ICJ1c2VyIiwKICAgICAgICAgICAgImNvbnRlbnQiOiBbCiAgICAgICAgICAgICAgICB7InR5cGUiOiAiaW1hZ2UiLCAic291cmNlIjogeyJ0eXBlIjogImJhc2U2NCIsICJtZWRpYV90eXBlIjogImltYWdlL3BuZyIsICJkYXRhIjogYjY0fX0sCiAgICAgICAgICAgICAgICB7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogX0NMQVNTSUZJQ0FUSU9OX1BST01QVH0sCiAgICAgICAgICAgIF0sCiAgICAgICAgfV0sCiAgICApCiAgICByZXR1cm4gX3BhcnNlX3Jlc3BvbnNlKG1lc3NhZ2UuY29udGVudFswXS50ZXh0KQoKCiMg4pSA4pSAIEhlbHBlcnMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgX3BhcnNlX3Jlc3BvbnNlKHRleHQ6IHN0cikgLT4gQ2xhc3NpZmljYXRpb25SZXN1bHQ6CiAgICB0ZXh0ID0gcmUuc3ViKHIiYGBgKD86anNvbik/IiwgIiIsIHRleHQpLnN0cmlwKCkKICAgIG1hdGNoID0gcmUuc2VhcmNoKHIiXHtbXHNcU10qP1x9IiwgdGV4dCkKICAgIGlmIG1hdGNoOgogICAgICAgIHRyeToKICAgICAgICAgICAgb2JqID0ganNvbi5sb2FkcyhtYXRjaC5ncm91cCgpKQogICAgICAgICAgICBkb2NfdHlwZSAgID0gb2JqLmdldCgiZG9jX3R5cGUiLCAiaW52b2ljZSIpLmxvd2VyKCkKICAgICAgICAgICAgY29uZmlkZW5jZSA9IGZsb2F0KG9iai5nZXQoImNvbmZpZGVuY2UiLCAwLjcpKQogICAgICAgICAgICByZWFzb25pbmcgID0gb2JqLmdldCgicmVhc29uaW5nIiwgIiIpCiAgICAgICAgICAgIHNjaGVtYSAgICAgPSBfU0NIRU1BX01BUC5nZXQoZG9jX3R5cGUsICJJbnZvaWNlRGF0YSIpCiAgICAgICAgICAgIHJldHVybiBDbGFzc2lmaWNhdGlvblJlc3VsdChkb2NfdHlwZSwgY29uZmlkZW5jZSwgc2NoZW1hLCByZWFzb25pbmcpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzcwogICAgcmV0dXJuIF9mYWxsYmFja19jbGFzc2lmeSgpCgoKZGVmIF9mYWxsYmFja19jbGFzc2lmeSgpIC0+IENsYXNzaWZpY2F0aW9uUmVzdWx0OgogICAgcmV0dXJuIENsYXNzaWZpY2F0aW9uUmVzdWx0KCJpbnZvaWNlIiwgMC41LCAiSW52b2ljZURhdGEiLCAiRmFsbGJhY2sg4oCUIGRlZmF1bHRpbmcgdG8gaW52b2ljZSIpCg==",
    "/content/capstone/agents/extraction_agent.py": "...",
    "/content/capstone/agents/validation_agent.py": "IiIiVmFsaWRhdGlvbiBBZ2VudCDigJQgZHluYW1pYyBjaGVja3MgdGhhdCBhZGFwdCB0byB3aGF0ZXZlciBmaWVsZHMgd2VyZSBleHRyYWN0ZWQuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgcmUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUKCmZyb20gY29uZmlnIGltcG9ydCBUT1RBTF9UT0xFUkFOQ0UKZnJvbSBtb2RlbHMuc2NoZW1hcyBpbXBvcnQgRXh0cmFjdGlvblJlc3VsdCwgRmllbGRWYWxpZGF0aW9uLCBWYWxpZGF0aW9uUmVzdWx0LCBWYWxpZGF0aW9uU3RhdHVzCmZyb20gdXRpbHMudmVuZG9yX21hdGNoZXIgaW1wb3J0IFZlbmRvck1hdGNoZXIKCl92ZW5kb3JfbWF0Y2hlciA9IFZlbmRvck1hdGNoZXIoKQoKCmRlZiBydW4oZXh0cmFjdGlvbjogRXh0cmFjdGlvblJlc3VsdCkgLT4gVmFsaWRhdGlvblJlc3VsdDoKICAgIGRhdGEgICA9IGV4dHJhY3Rpb24uZXh0cmFjdGVkX2RhdGEKICAgIGNoZWNrczogbGlzdFtGaWVsZFZhbGlkYXRpb25dID0gW10KCiAgICBjaGVja3MuZXh0ZW5kKF9jaGVja190b3RhbHMoZGF0YSkpCiAgICBjaGVja3MuZXh0ZW5kKF9jaGVja19kYXRlcyhkYXRhKSkKICAgIGNoZWNrcy5leHRlbmQoX2NoZWNrX2liYW4oZGF0YSkpCiAgICBjaGVja3MuZXh0ZW5kKF9jaGVja192ZW5kb3IoZGF0YSkpCgogICAgZmFpbGVkICAgICA9IFt2LmZpZWxkIGZvciB2IGluIGNoZWNrcyBpZiB2LnN0YXR1cyA9PSBWYWxpZGF0aW9uU3RhdHVzLkZBSUxFRF0KICAgIHBhc3NlZF9jbnQgPSBzdW0oMSBmb3IgdiBpbiBjaGVja3MgaWYgdi5zdGF0dXMgPT0gVmFsaWRhdGlvblN0YXR1cy5WQUxJRCkKICAgIGNvbmYgICAgICAgPSByb3VuZChwYXNzZWRfY250IC8gbWF4KGxlbihjaGVja3MpLCAxKSwgMikKCiAgICByZXR1cm4gVmFsaWRhdGlvblJlc3VsdCgKICAgICAgICBpc192YWxpZD1sZW4oZmFpbGVkKSA9PSAwLAogICAgICAgIGZpZWxkX3ZhbGlkYXRpb25zPWNoZWNrcywKICAgICAgICBmYWlsZWRfZmllbGRzPWZhaWxlZCwKICAgICAgICBvdmVyYWxsX2NvbmZpZGVuY2U9Y29uZiwKICAgICkKCgojIOKUgOKUgCBDaGVja3Mg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgX2NoZWNrX3RvdGFscyhkYXRhKSAtPiBsaXN0W0ZpZWxkVmFsaWRhdGlvbl06CiAgICAiIiJWZXJpZnkgc3VidG90YWwgKyB0YXggPSB0b3RhbCBpZiBhbGwgdGhyZWUgYXJlIHByZXNlbnQuIiIiCiAgICB0b3RhbCAgICA9IGRhdGEudG90YWxfYW1vdW50CiAgICBzdWJ0b3RhbCA9IGRhdGEuc3VidG90YWwKICAgIHRheCAgICAgID0gZGF0YS50YXhfYW1vdW50CgogICAgaWYgTm9uZSBpbiAodG90YWwsIHN1YnRvdGFsLCB0YXgpOgogICAgICAgIHJldHVybiBbXQoKICAgIGV4cGVjdGVkID0gcm91bmQoc3VidG90YWwgKyB0YXgsIDIpCiAgICBpZiBhYnMoZXhwZWN0ZWQgLSB0b3RhbCkgPiBUT1RBTF9UT0xFUkFOQ0U6CiAgICAgICAgcmV0dXJuIFtGaWVsZFZhbGlkYXRpb24oCiAgICAgICAgICAgIGZpZWxkPSJ0b3RhbF9hbW91bnQiLAogICAgICAgICAgICBzdGF0dXM9VmFsaWRhdGlvblN0YXR1cy5GQUlMRUQsCiAgICAgICAgICAgIG1lc3NhZ2U9ZiJUb3RhbCB7dG90YWx9IOKJoCBzdWJ0b3RhbCB7c3VidG90YWx9ICsgdGF4IHt0YXh9ID0ge2V4cGVjdGVkfSIsCiAgICAgICAgKV0KICAgIHJldHVybiBbRmllbGRWYWxpZGF0aW9uKGZpZWxkPSJ0b3RhbF9hbW91bnQiLCBzdGF0dXM9VmFsaWRhdGlvblN0YXR1cy5WQUxJRCldCgoKZGVmIF9jaGVja19kYXRlcyhkYXRhKSAtPiBsaXN0W0ZpZWxkVmFsaWRhdGlvbl06CiAgICAiIiJDaGVjayBhbnkgZmllbGQgd2hvc2UgbmFtZSBjb250YWlucyAnZGF0ZScgZm9yIElTTyA4NjAxIGZvcm1hdC4iIiIKICAgIHJlc3VsdHMgPSBbXQogICAgZGF0ZV9maWVsZHMgPSB7azogdiBmb3IgaywgdiBpbiBkYXRhLmZpZWxkcy5pdGVtcygpCiAgICAgICAgICAgICAgICAgICBpZiAiZGF0ZSIgaW4gay5sb3dlcigpIGFuZCB2IGFuZCBpc2luc3RhbmNlKHYsIHN0cil9CiAgICBmb3IgZm5hbWUsIHZhbHVlIGluIGRhdGVfZmllbGRzLml0ZW1zKCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBkYXRldGltZS5zdHJwdGltZSh2YWx1ZSwgIiVZLSVtLSVkIikKICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoRmllbGRWYWxpZGF0aW9uKGZpZWxkPWZuYW1lLCBzdGF0dXM9VmFsaWRhdGlvblN0YXR1cy5WQUxJRCkpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKEZpZWxkVmFsaWRhdGlvbigKICAgICAgICAgICAgICAgIGZpZWxkPWZuYW1lLAogICAgICAgICAgICAgICAgc3RhdHVzPVZhbGlkYXRpb25TdGF0dXMuRkFJTEVELAogICAgICAgICAgICAgICAgbWVzc2FnZT1mIid7dmFsdWV9JyBpcyBub3QgWVlZWS1NTS1ERCBmb3JtYXQiLAogICAgICAgICAgICApKQogICAgcmV0dXJuIHJlc3VsdHMKCgpkZWYgX2NoZWNrX2liYW4oZGF0YSkgLT4gbGlzdFtGaWVsZFZhbGlkYXRpb25dOgogICAgIiIiVmFsaWRhdGUgSUJBTiBmb3JtYXQgaWYgcHJlc2VudC4iIiIKICAgIGliYW4gPSBkYXRhLmliYW4KICAgIGlmIG5vdCBpYmFuOgogICAgICAgIHJldHVybiBbXQogICAgY2xlYW5lZCA9IGliYW4ucmVwbGFjZSgiICIsICIiKS51cHBlcigpCiAgICBpZiByZS5tYXRjaChyIl5bQS1aXXsyfVswLTldezJ9W0EtWjAtOV17MTEsMzB9JCIsIGNsZWFuZWQpOgogICAgICAgIHJldHVybiBbRmllbGRWYWxpZGF0aW9uKGZpZWxkPSJpYmFuIiwgc3RhdHVzPVZhbGlkYXRpb25TdGF0dXMuVkFMSUQpXQogICAgcmV0dXJuIFtGaWVsZFZhbGlkYXRpb24oCiAgICAgICAgZmllbGQ9ImliYW4iLAogICAgICAgIHN0YXR1cz1WYWxpZGF0aW9uU3RhdHVzLkZBSUxFRCwKICAgICAgICBtZXNzYWdlPWYiSW52YWxpZCBJQkFOIGZvcm1hdDoge2liYW59IiwKICAgICldCgoKZGVmIF9jaGVja192ZW5kb3IoZGF0YSkgLT4gbGlzdFtGaWVsZFZhbGlkYXRpb25dOgogICAgIiIiRnV6enktbWF0Y2ggdmVuZG9yIG5hbWUgYWdhaW5zdCBrbm93biB2ZW5kb3IgcmVnaXN0cnkuIiIiCiAgICB2ZW5kb3IgPSBkYXRhLnZlbmRvcl9uYW1lCiAgICBpZiBub3QgdmVuZG9yOgogICAgICAgIHJldHVybiBbXQoKICAgIGtub3duLCBtYXRjaGVkID0gX3ZlbmRvcl9tYXRjaGVyLmlzX2tub3duX3ZlbmRvcih2ZW5kb3IpCiAgICBpZiBrbm93biBhbmQgbWF0Y2hlZCBhbmQgbWF0Y2hlZC5sb3dlcigpICE9IHZlbmRvci5sb3dlcigpOgogICAgICAgIHJldHVybiBbRmllbGRWYWxpZGF0aW9uKAogICAgICAgICAgICBmaWVsZD0idmVuZG9yX25hbWUiLAogICAgICAgICAgICBzdGF0dXM9VmFsaWRhdGlvblN0YXR1cy5WQUxJRCwKICAgICAgICAgICAgbWVzc2FnZT1mIk1hdGNoZWQgdG8ga25vd24gdmVuZG9yOiAne21hdGNoZWR9JyIsCiAgICAgICAgICAgIGNvcnJlY3RlZF92YWx1ZT1tYXRjaGVkLAogICAgICAgICldCiAgICByZXR1cm4gW0ZpZWxkVmFsaWRhdGlvbihmaWVsZD0idmVuZG9yX25hbWUiLCBzdGF0dXM9VmFsaWRhdGlvblN0YXR1cy5WQUxJRCldCg==",
    "/content/capstone/agents/correction_agent.py": "IiIiQXV0by1Db3JyZWN0aW9uIEFnZW50IOKAlCByZS1xdWVyaWVzIFZMTSB3aXRoIGEgZm9jdXNlZCBjcm9wIHRvIGZpeCBmYWlsZWQgZmllbGRzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbCwgTGlzdAoKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgppbXBvcnQgY29uZmlnCmZyb20gY29uZmlnIGltcG9ydCBBTlRIUk9QSUNfQVBJX0tFWSwgQ0xBVURFX01PREVMLCBHRU1JTklfQVBJX0tFWSwgR0VNSU5JX01PREVMCmZyb20gbW9kZWxzLnNjaGVtYXMgaW1wb3J0IEV4dHJhY3Rpb25SZXN1bHQsIFZhbGlkYXRpb25SZXN1bHQsIFZhbGlkYXRpb25TdGF0dXMKZnJvbSBwaXBlbGluZS5vY3IgaW1wb3J0IGNyb3BfcmVnaW9uCmZyb20gdXRpbHMuaW1hZ2VfcHJvY2Vzc2luZyBpbXBvcnQgaW1hZ2VfdG9fYmFzZTY0CgpfUFJPTVBUID0gIiIiQSBmaWVsZCBleHRyYWN0ZWQgZnJvbSB0aGlzIGRvY3VtZW50IGZhaWxlZCB2YWxpZGF0aW9uLgoKRmllbGQ6IHtmaWVsZH0KQ3VycmVudCB2YWx1ZToge2N1cnJlbnRfdmFsdWV9ClZhbGlkYXRpb24gZXJyb3I6IHtlcnJvcl9tZXNzYWdlfQoKTG9vayBjYXJlZnVsbHkgYXQgdGhlIGltYWdlIGNyb3AgYW5kIGV4dHJhY3QgT05MWSB0aGUgY29ycmVjdCB2YWx1ZSBmb3IgIntmaWVsZH0iLgpSZXR1cm4gT05MWSB0aGlzIEpTT04gKG5vIG1hcmtkb3duLCBubyBleHBsYW5hdGlvbik6Cnt7ImZpZWxkIjogIntmaWVsZH0iLCAidmFsdWUiOiA8Y29ycmVjdGVkIHZhbHVlIG9yIG51bGw+fX0KClJ1bGVzOiBkYXRlcyA9IFlZWVktTU0tREQsIGFtb3VudHMgPSBwbGFpbiBudW1iZXJzLCByZXR1cm4gT05MWSBKU09OLiIiIgoKIyBLZXl3b3JkcyB0byBmaW5kIHRoZSByaWdodCBpbWFnZSByZWdpb24gcGVyIGZpZWxkIG5hbWUKX1JFR0lPTl9LRVlXT1JEUyA9IHsKICAgICJ0b3RhbF9hbW91bnQiOiAidG90YWwiLCAgInN1YnRvdGFsIjogInN1YnRvdGFsIiwgICJ0YXhfYW1vdW50IjogInRheCIsCiAgICAiaW52b2ljZV9kYXRlIjogImRhdGUiLCAgICJkdWVfZGF0ZSI6ICJkdWUiLCAgICAgICAgImludm9pY2VfbnVtYmVyIjogImludm9pY2UiLAogICAgInZlbmRvcl9uYW1lIjogICJ2ZW5kb3IiLCAiaWJhbiI6ICAgICAiaWJhbiIsICAgICAgICAiYWNjb3VudF9udW1iZXIiOiAiYWNjb3VudCIsCn0KCgpkZWYgcnVuKAogICAgaW1hZ2VzOiBMaXN0W0ltYWdlLkltYWdlXSwKICAgIGV4dHJhY3Rpb246IEV4dHJhY3Rpb25SZXN1bHQsCiAgICB2YWxpZGF0aW9uOiBWYWxpZGF0aW9uUmVzdWx0LAopIC0+IEV4dHJhY3Rpb25SZXN1bHQ6CiAgICB1cGRhdGVkID0gZXh0cmFjdGlvbi5leHRyYWN0ZWRfZGF0YS5tb2RlbF9jb3B5KGRlZXA9VHJ1ZSkKCiAgICBmb3IgZmllbGQgaW4gdmFsaWRhdGlvbi5mYWlsZWRfZmllbGRzOgogICAgICAgIGZ2ICAgICAgPSBuZXh0KCh2IGZvciB2IGluIHZhbGlkYXRpb24uZmllbGRfdmFsaWRhdGlvbnMgaWYgdi5maWVsZCA9PSBmaWVsZCksIE5vbmUpCiAgICAgICAgY3VycmVudCA9IHN0cih1cGRhdGVkLmZpZWxkcy5nZXQoZmllbGQsICIiKSkKICAgICAgICBlcnJvciAgID0gZnYubWVzc2FnZSBpZiBmdiBlbHNlICIiCiAgICAgICAga2V5d29yZCA9IF9SRUdJT05fS0VZV09SRFMuZ2V0KGZpZWxkLCBmaWVsZC5yZXBsYWNlKCJfIiwgIiAiKSkKCiAgICAgICAgY29ycmVjdGVkID0gTm9uZQogICAgICAgIAogICAgICAgIGZvciBpbWFnZSBpbiBpbWFnZXM6CiAgICAgICAgICAgIGNvcnJlY3RlZCA9IF9jb3JyZWN0X2ZpZWxkKGltYWdlLCBmaWVsZCwgY3VycmVudCwgZXJyb3IsIGtleXdvcmQpCiAgICAgICAgICAgIGlmIGNvcnJlY3RlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgCiAgICAgICAgaWYgY29ycmVjdGVkIGlzIG5vdCBOb25lOgogICAgICAgICAgICB1cGRhdGVkLmZpZWxkc1tmaWVsZF0gPSBjb3JyZWN0ZWQKCiAgICByZXR1cm4gRXh0cmFjdGlvblJlc3VsdCgKICAgICAgICByYXdfb2NyX3RleHQ9ZXh0cmFjdGlvbi5yYXdfb2NyX3RleHQsCiAgICAgICAgZXh0cmFjdGVkX2RhdGE9dXBkYXRlZCwKICAgICAgICBjb25maWRlbmNlPWV4dHJhY3Rpb24uY29uZmlkZW5jZSwKICAgICAgICB2bG1fcmVzcG9uc2U9ZXh0cmFjdGlvbi52bG1fcmVzcG9uc2UsCiAgICAgICAgb2NyX3VzZWQ9ZXh0cmFjdGlvbi5vY3JfdXNlZCwKICAgICkKCgpkZWYgX2NvcnJlY3RfZmllbGQoaW1hZ2U6IEltYWdlLkltYWdlLCBmaWVsZDogc3RyLCBjdXJyZW50OiBzdHIsCiAgICAgICAgICAgICAgICAgICBlcnJvcjogc3RyLCBrZXl3b3JkOiBzdHIpIC0+IE9wdGlvbmFsW3N0cl06CiAgICBjcm9wICAgPSBjcm9wX3JlZ2lvbihpbWFnZSwga2V5d29yZCkgb3IgaW1hZ2UKICAgIHByb21wdCA9IF9QUk9NUFQuZm9ybWF0KGZpZWxkPWZpZWxkLCBjdXJyZW50X3ZhbHVlPWN1cnJlbnQsIGVycm9yX21lc3NhZ2U9ZXJyb3IpCgogICAgaWYgY29uZmlnLlZMTV9CQUNLRU5EID09ICJnZW1pbmkiOgogICAgICAgIHJldHVybiBfY29ycmVjdF9nZW1pbmkoY3JvcCwgcHJvbXB0KQogICAgZWxpZiBjb25maWcuVkxNX0JBQ0tFTkQgPT0gImNsYXVkZSI6CiAgICAgICAgcmV0dXJuIF9jb3JyZWN0X2NsYXVkZShjcm9wLCBwcm9tcHQpCiAgICByZXR1cm4gTm9uZQoKCmRlZiBfY29ycmVjdF9nZW1pbmkoY3JvcDogSW1hZ2UuSW1hZ2UsIHByb21wdDogc3RyKSAtPiBPcHRpb25hbFtzdHJdOgogICAgZnJvbSBnb29nbGUgaW1wb3J0IGdlbmFpCiAgICBmcm9tIGdvb2dsZS5nZW5haSBpbXBvcnQgdHlwZXMKCiAgICBjbGllbnQgPSBnZW5haS5DbGllbnQoYXBpX2tleT1HRU1JTklfQVBJX0tFWSkKICAgIHIgPSBjbGllbnQubW9kZWxzLmdlbmVyYXRlX2NvbnRlbnQoCiAgICAgICAgbW9kZWw9R0VNSU5JX01PREVMLAogICAgICAgIGNvbnRlbnRzPVtjcm9wLCBwcm9tcHRdLAogICAgICAgIGNvbmZpZz10eXBlcy5HZW5lcmF0ZUNvbnRlbnRDb25maWcobWF4X291dHB1dF90b2tlbnM9MjU2LCB0ZW1wZXJhdHVyZT0wLjEpLAogICAgKQogICAgcmV0dXJuIF9wYXJzZShyLnRleHQpCgoKZGVmIF9jb3JyZWN0X2NsYXVkZShjcm9wOiBJbWFnZS5JbWFnZSwgcHJvbXB0OiBzdHIpIC0+IE9wdGlvbmFsW3N0cl06CiAgICBpbXBvcnQgYW50aHJvcGljCiAgICBjbGllbnQgPSBhbnRocm9waWMuQW50aHJvcGljKGFwaV9rZXk9QU5USFJPUElDX0FQSV9LRVkpCiAgICBiNjQgICAgPSBpbWFnZV90b19iYXNlNjQoY3JvcCwgZm10PSJQTkciKQogICAgbXNnICAgID0gY2xpZW50Lm1lc3NhZ2VzLmNyZWF0ZSgKICAgICAgICBtb2RlbD1DTEFVREVfTU9ERUwsIG1heF90b2tlbnM9MjU2LAogICAgICAgIG1lc3NhZ2VzPVt7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogWwogICAgICAgICAgICB7InR5cGUiOiAiaW1hZ2UiLCAic291cmNlIjogeyJ0eXBlIjogImJhc2U2NCIsICJtZWRpYV90eXBlIjogImltYWdlL3BuZyIsICJkYXRhIjogYjY0fX0sCiAgICAgICAgICAgIHsidHlwZSI6ICJ0ZXh0IiwgICJ0ZXh0IjogcHJvbXB0fSwKICAgICAgICBdfV0sCiAgICApCiAgICByZXR1cm4gX3BhcnNlKG1zZy5jb250ZW50WzBdLnRleHQpCgoKZGVmIF9wYXJzZSh0ZXh0OiBzdHIpIC0+IE9wdGlvbmFsW3N0cl06CiAgICB0ZXh0ID0gcmUuc3ViKHIiYGBgKD86anNvbik/IiwgIiIsIHRleHQpLnN0cmlwKCkKICAgIG0gICAgPSByZS5zZWFyY2gociJce1tcc1xTXSo/XH0iLCB0ZXh0KQogICAgaWYgbm90IG06CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHRyeToKICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhtLmdyb3VwKCkpLmdldCgidmFsdWUiKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQo=",
    "/content/capstone/database/__init__.py": "",
    "/content/capstone/database/storage.py": "IiIiU1FMaXRlIHN0b3JhZ2Ug4oCUIENSVUQsIGF1ZGl0IGxvZywgcmV2aWV3IHF1ZXVlLCBzdGF0cywgYW5kIGV4cG9ydC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCBzcWxpdGUzCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCmZyb20gY29uZmlnIGltcG9ydCBTUUxJVEVfUEFUSApmcm9tIG1vZGVscy5zY2hlbWFzIGltcG9ydCBQcm9jZXNzaW5nUmVjb3JkCgoKZGVmIF9nZXRfY29ubigpIC0+IHNxbGl0ZTMuQ29ubmVjdGlvbjoKICAgIFBhdGgoU1FMSVRFX1BBVEgpLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb25uID0gc3FsaXRlMy5jb25uZWN0KFNRTElURV9QQVRIKQogICAgY29ubi5yb3dfZmFjdG9yeSA9IHNxbGl0ZTMuUm93CiAgICByZXR1cm4gY29ubgoKCmRlZiBpbml0X2RiKCkgLT4gTm9uZToKICAgIHdpdGggX2dldF9jb25uKCkgYXMgY29ubjoKICAgICAgICBjb25uLmV4ZWN1dGUoIiIiCiAgICAgICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIGludm9pY2VfcmVjb3JkcyAoCiAgICAgICAgICAgICAgICBpZCAgICAgICAgICAgICAgICAgICAgSU5URUdFUiBQUklNQVJZIEtFWSBBVVRPSU5DUkVNRU5ULAogICAgICAgICAgICAgICAgZG9jdW1lbnRfaWQgICAgICAgICAgIFRFWFQgVU5JUVVFIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgc291cmNlX3BhdGggICAgICAgICAgIFRFWFQsCiAgICAgICAgICAgICAgICBkb2NfdHlwZSAgICAgICAgICAgICAgVEVYVCBERUZBVUxUICdpbnZvaWNlJywKICAgICAgICAgICAgICAgIHJhd19vY3JfdGV4dCAgICAgICAgICBURVhULAogICAgICAgICAgICAgICAgdmxtX2V4dHJhY3Rpb24gICAgICAgIFRFWFQsCiAgICAgICAgICAgICAgICBjb3JyZWN0aW9ucyAgICAgICAgICAgVEVYVCwKICAgICAgICAgICAgICAgIGZpbmFsX2RhdGEgICAgICAgICAgICBURVhULAogICAgICAgICAgICAgICAgdmFsaWRhdGlvbl9zdGF0dXMgICAgIFRFWFQsCiAgICAgICAgICAgICAgICBleHRyYWN0aW9uX2NvbmZpZGVuY2UgUkVBTCBERUZBVUxUIDAuMCwKICAgICAgICAgICAgICAgIHByb2Nlc3Npbmdfbm90ZXMgICAgICBURVhULAogICAgICAgICAgICAgICAgY3JlYXRlZF9hdCAgICAgICAgICAgIFRFWFQgREVGQVVMVCAoZGF0ZXRpbWUoJ25vdycpKQogICAgICAgICAgICApCiAgICAgICAgIiIiKQogICAgICAgIGNvbm4uZXhlY3V0ZSgiIiIKICAgICAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgYXVkaXRfbG9nICgKICAgICAgICAgICAgICAgIGlkICAgICAgICAgIElOVEVHRVIgUFJJTUFSWSBLRVkgQVVUT0lOQ1JFTUVOVCwKICAgICAgICAgICAgICAgIGRvY3VtZW50X2lkIFRFWFQgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICBldmVudCAgICAgICBURVhULAogICAgICAgICAgICAgICAgZGV0YWlscyAgICAgVEVYVCwKICAgICAgICAgICAgICAgIHRpbWVzdGFtcCAgIFRFWFQgREVGQVVMVCAoZGF0ZXRpbWUoJ25vdycpKQogICAgICAgICAgICApCiAgICAgICAgIiIiKQogICAgICAgICMgQWRkIG5ldyBjb2x1bW5zIHRvIGV4aXN0aW5nIERCcyB3aXRob3V0IGZhaWxpbmcKICAgICAgICBmb3IgY29sLCB0eXBlZGVmIGluIFsKICAgICAgICAgICAgKCJkb2NfdHlwZSIsICAgICAgICAgICAgICAiVEVYVCBERUZBVUxUICdpbnZvaWNlJyIpLAogICAgICAgICAgICAoImV4dHJhY3Rpb25fY29uZmlkZW5jZSIsICJSRUFMIERFRkFVTFQgMC4wIiksCiAgICAgICAgXToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY29ubi5leGVjdXRlKGYiQUxURVIgVEFCTEUgaW52b2ljZV9yZWNvcmRzIEFERCBDT0xVTU4ge2NvbH0ge3R5cGVkZWZ9IikKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MgICMgY29sdW1uIGFscmVhZHkgZXhpc3RzCgoKIyDilIDilIAgV3JpdGUg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgc2F2ZV9yZWNvcmQocmVjb3JkOiBQcm9jZXNzaW5nUmVjb3JkKSAtPiBOb25lOgogICAgd2l0aCBfZ2V0X2Nvbm4oKSBhcyBjb25uOgogICAgICAgIGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIiIiCiAgICAgICAgICAgIElOU0VSVCBJTlRPIGludm9pY2VfcmVjb3JkcwogICAgICAgICAgICAgICAgKGRvY3VtZW50X2lkLCBzb3VyY2VfcGF0aCwgZG9jX3R5cGUsIHJhd19vY3JfdGV4dCwgdmxtX2V4dHJhY3Rpb24sCiAgICAgICAgICAgICAgICAgY29ycmVjdGlvbnMsIGZpbmFsX2RhdGEsIHZhbGlkYXRpb25fc3RhdHVzLCBleHRyYWN0aW9uX2NvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICAgcHJvY2Vzc2luZ19ub3RlcykKICAgICAgICAgICAgVkFMVUVTICg/LCA/LCA/LCA/LCA/LCA/LCA/LCA/LCA/LCA/KQogICAgICAgICAgICBPTiBDT05GTElDVChkb2N1bWVudF9pZCkgRE8gVVBEQVRFIFNFVAogICAgICAgICAgICAgICAgZG9jX3R5cGUgICAgICAgICAgICAgID0gZXhjbHVkZWQuZG9jX3R5cGUsCiAgICAgICAgICAgICAgICB2bG1fZXh0cmFjdGlvbiAgICAgICAgPSBleGNsdWRlZC52bG1fZXh0cmFjdGlvbiwKICAgICAgICAgICAgICAgIGNvcnJlY3Rpb25zICAgICAgICAgICA9IGV4Y2x1ZGVkLmNvcnJlY3Rpb25zLAogICAgICAgICAgICAgICAgZmluYWxfZGF0YSAgICAgICAgICAgID0gZXhjbHVkZWQuZmluYWxfZGF0YSwKICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fc3RhdHVzICAgICA9IGV4Y2x1ZGVkLnZhbGlkYXRpb25fc3RhdHVzLAogICAgICAgICAgICAgICAgZXh0cmFjdGlvbl9jb25maWRlbmNlID0gZXhjbHVkZWQuZXh0cmFjdGlvbl9jb25maWRlbmNlLAogICAgICAgICAgICAgICAgcHJvY2Vzc2luZ19ub3RlcyAgICAgID0gZXhjbHVkZWQucHJvY2Vzc2luZ19ub3RlcwogICAgICAgICAgICAiIiIsCiAgICAgICAgICAgICgKICAgICAgICAgICAgICAgIHJlY29yZC5kb2N1bWVudF9pZCwKICAgICAgICAgICAgICAgIHJlY29yZC5zb3VyY2VfcGF0aCwKICAgICAgICAgICAgICAgIHJlY29yZC5kb2NfdHlwZSwKICAgICAgICAgICAgICAgIHJlY29yZC5yYXdfb2NyX3RleHQsCiAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHJlY29yZC52bG1fZXh0cmFjdGlvbiksCiAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHJlY29yZC5jb3JyZWN0aW9uc19hcHBsaWVkKSwKICAgICAgICAgICAgICAgIGpzb24uZHVtcHMocmVjb3JkLmZpbmFsX2RhdGEpLAogICAgICAgICAgICAgICAgcmVjb3JkLnZhbGlkYXRpb25fc3RhdHVzLnZhbHVlLAogICAgICAgICAgICAgICAgcmVjb3JkLmV4dHJhY3Rpb25fY29uZmlkZW5jZSwKICAgICAgICAgICAgICAgIGpzb24uZHVtcHMocmVjb3JkLnByb2Nlc3Npbmdfbm90ZXMpLAogICAgICAgICAgICApLAogICAgICAgICkKCgpkZWYgbG9nX2V2ZW50KGRvY3VtZW50X2lkOiBzdHIsIGV2ZW50OiBzdHIsIGRldGFpbHM6IE9wdGlvbmFsW2RpY3RdID0gTm9uZSkgLT4gTm9uZToKICAgIHdpdGggX2dldF9jb25uKCkgYXMgY29ubjoKICAgICAgICBjb25uLmV4ZWN1dGUoCiAgICAgICAgICAgICJJTlNFUlQgSU5UTyBhdWRpdF9sb2cgKGRvY3VtZW50X2lkLCBldmVudCwgZGV0YWlscykgVkFMVUVTICg/LCA/LCA/KSIsCiAgICAgICAgICAgIChkb2N1bWVudF9pZCwgZXZlbnQsIGpzb24uZHVtcHMoZGV0YWlscyBvciB7fSkpLAogICAgICAgICkKCgojIOKUgOKUgCBSZWFkIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIGdldF9yZWNvcmQoZG9jdW1lbnRfaWQ6IHN0cikgLT4gT3B0aW9uYWxbZGljdF06CiAgICB3aXRoIF9nZXRfY29ubigpIGFzIGNvbm46CiAgICAgICAgcm93ID0gY29ubi5leGVjdXRlKAogICAgICAgICAgICAiU0VMRUNUICogRlJPTSBpbnZvaWNlX3JlY29yZHMgV0hFUkUgZG9jdW1lbnRfaWQgPSA/IiwgKGRvY3VtZW50X2lkLCkKICAgICAgICApLmZldGNob25lKCkKICAgIHJldHVybiBkaWN0KHJvdykgaWYgcm93IGVsc2UgTm9uZQoKCmRlZiBsaXN0X3JlY29yZHMobGltaXQ6IGludCA9IDIwMCwgc3RhdHVzOiBPcHRpb25hbFtzdHJdID0gTm9uZSkgLT4gbGlzdFtkaWN0XToKICAgIHdpdGggX2dldF9jb25uKCkgYXMgY29ubjoKICAgICAgICBpZiBzdGF0dXM6CiAgICAgICAgICAgIHJvd3MgPSBjb25uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiU0VMRUNUICogRlJPTSBpbnZvaWNlX3JlY29yZHMgV0hFUkUgdmFsaWRhdGlvbl9zdGF0dXMgPSA/IE9SREVSIEJZIGNyZWF0ZWRfYXQgREVTQyBMSU1JVCA/IiwKICAgICAgICAgICAgICAgIChzdGF0dXMsIGxpbWl0KSwKICAgICAgICAgICAgKS5mZXRjaGFsbCgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcm93cyA9IGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgICAgICJTRUxFQ1QgKiBGUk9NIGludm9pY2VfcmVjb3JkcyBPUkRFUiBCWSBjcmVhdGVkX2F0IERFU0MgTElNSVQgPyIsIChsaW1pdCwpCiAgICAgICAgICAgICkuZmV0Y2hhbGwoKQogICAgcmV0dXJuIFtkaWN0KHIpIGZvciByIGluIHJvd3NdCgoKZGVmIGdldF9wZW5kaW5nX3JldmlldyhsaW1pdDogaW50ID0gMTAwKSAtPiBsaXN0W2RpY3RdOgogICAgcmV0dXJuIGxpc3RfcmVjb3JkcyhsaW1pdD1saW1pdCwgc3RhdHVzPSJwZW5kaW5nX3JldmlldyIpCgoKZGVmIGdldF9hdWRpdF90cmFpbChkb2N1bWVudF9pZDogc3RyKSAtPiBsaXN0W2RpY3RdOgogICAgd2l0aCBfZ2V0X2Nvbm4oKSBhcyBjb25uOgogICAgICAgIHJvd3MgPSBjb25uLmV4ZWN1dGUoCiAgICAgICAgICAgICJTRUxFQ1QgKiBGUk9NIGF1ZGl0X2xvZyBXSEVSRSBkb2N1bWVudF9pZCA9ID8gT1JERVIgQlkgdGltZXN0YW1wIEFTQyIsCiAgICAgICAgICAgIChkb2N1bWVudF9pZCwpLAogICAgICAgICkuZmV0Y2hhbGwoKQogICAgcmV0dXJuIFtkaWN0KHIpIGZvciByIGluIHJvd3NdCgoKIyDilIDilIAgUmV2aWV3IHF1ZXVlIGFjdGlvbnMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgYXBwcm92ZV9yZWNvcmQoZG9jdW1lbnRfaWQ6IHN0ciwgdXBkYXRlZF9kYXRhOiBkaWN0KSAtPiBOb25lOgogICAgd2l0aCBfZ2V0X2Nvbm4oKSBhcyBjb25uOgogICAgICAgIGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIiIiVVBEQVRFIGludm9pY2VfcmVjb3JkcwogICAgICAgICAgICAgICBTRVQgdmFsaWRhdGlvbl9zdGF0dXMgPSAndmFsaWQnLCBmaW5hbF9kYXRhID0gPwogICAgICAgICAgICAgICBXSEVSRSBkb2N1bWVudF9pZCA9ID8iIiIsCiAgICAgICAgICAgIChqc29uLmR1bXBzKHVwZGF0ZWRfZGF0YSksIGRvY3VtZW50X2lkKSwKICAgICAgICApCiAgICBsb2dfZXZlbnQoZG9jdW1lbnRfaWQsICJhcHByb3ZlZCIsIHsiYnkiOiAiaHVtYW5fcmV2aWV3ZXIifSkKCgpkZWYgcmVqZWN0X3JlY29yZChkb2N1bWVudF9pZDogc3RyKSAtPiBOb25lOgogICAgd2l0aCBfZ2V0X2Nvbm4oKSBhcyBjb25uOgogICAgICAgIGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIlVQREFURSBpbnZvaWNlX3JlY29yZHMgU0VUIHZhbGlkYXRpb25fc3RhdHVzID0gJ3JlamVjdGVkJyBXSEVSRSBkb2N1bWVudF9pZCA9ID8iLAogICAgICAgICAgICAoZG9jdW1lbnRfaWQsKSwKICAgICAgICApCiAgICBsb2dfZXZlbnQoZG9jdW1lbnRfaWQsICJyZWplY3RlZCIsIHsiYnkiOiAiaHVtYW5fcmV2aWV3ZXIifSkKCgojIOKUgOKUgCBBZ2dyZWdhdGVzIGZvciBkYXNoYm9hcmQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgZ2V0X3N0YXRzKCkgLT4gZGljdDoKICAgIHdpdGggX2dldF9jb25uKCkgYXMgY29ubjoKICAgICAgICB0b3RhbCAgID0gY29ubi5leGVjdXRlKCJTRUxFQ1QgQ09VTlQoKikgRlJPTSBpbnZvaWNlX3JlY29yZHMiKS5mZXRjaG9uZSgpWzBdCiAgICAgICAgdmFsaWQgICA9IGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIlNFTEVDVCBDT1VOVCgqKSBGUk9NIGludm9pY2VfcmVjb3JkcyBXSEVSRSB2YWxpZGF0aW9uX3N0YXR1cyBJTiAoJ3ZhbGlkJywnY29ycmVjdGVkJykiCiAgICAgICAgKS5mZXRjaG9uZSgpWzBdCiAgICAgICAgcGVuZGluZyA9IGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIlNFTEVDVCBDT1VOVCgqKSBGUk9NIGludm9pY2VfcmVjb3JkcyBXSEVSRSB2YWxpZGF0aW9uX3N0YXR1cyA9ICdwZW5kaW5nX3JldmlldyciCiAgICAgICAgKS5mZXRjaG9uZSgpWzBdCiAgICAgICAgdG9kYXkgICA9IGNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIlNFTEVDVCBDT1VOVCgqKSBGUk9NIGludm9pY2VfcmVjb3JkcyBXSEVSRSBjcmVhdGVkX2F0ID49IGRhdGUoJ25vdycpIgogICAgICAgICkuZmV0Y2hvbmUoKVswXQoKICAgICAgICBieV90eXBlID0gY29ubi5leGVjdXRlKAogICAgICAgICAgICAiU0VMRUNUIGRvY190eXBlLCBDT1VOVCgqKSBhcyBjbnQgRlJPTSBpbnZvaWNlX3JlY29yZHMgR1JPVVAgQlkgZG9jX3R5cGUiCiAgICAgICAgKS5mZXRjaGFsbCgpCgogICAgcmV0dXJuIHsKICAgICAgICAidG90YWwiOiAgICAgICAgdG90YWwsCiAgICAgICAgInN1Y2Nlc3NfcmF0ZSI6IHJvdW5kKHZhbGlkIC8gdG90YWwgKiAxMDAsIDEpIGlmIHRvdGFsIGVsc2UgMC4wLAogICAgICAgICJwZW5kaW5nIjogICAgICBwZW5kaW5nLAogICAgICAgICJ0b2RheSI6ICAgICAgICB0b2RheSwKICAgICAgICAiYnlfdHlwZSI6ICAgICAge3JbImRvY190eXBlIl06IHJbImNudCJdIGZvciByIGluIGJ5X3R5cGV9LAogICAgfQoKCiMg4pSA4pSAIEV4cG9ydCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmRlZiBleHBvcnRfanNvbihkb2N1bWVudF9pZDogc3RyKSAtPiBzdHI6CiAgICByZWNvcmQgPSBnZXRfcmVjb3JkKGRvY3VtZW50X2lkKQogICAgaWYgbm90IHJlY29yZDoKICAgICAgICByZXR1cm4gInt9IgogICAgZGF0YSA9IGpzb24ubG9hZHMocmVjb3JkLmdldCgiZmluYWxfZGF0YSIpIG9yICJ7fSIpCiAgICByZXR1cm4ganNvbi5kdW1wcyhkYXRhLCBpbmRlbnQ9MiwgZGVmYXVsdD1zdHIpCgoKZGVmIGV4cG9ydF9hbGxfY3N2KCkgLT4gc3RyOgogICAgaW1wb3J0IGNzdiwgaW8KICAgIHJlY29yZHMgPSBsaXN0X3JlY29yZHMobGltaXQ9MTAwMDApCiAgICBpZiBub3QgcmVjb3JkczoKICAgICAgICByZXR1cm4gIiIKICAgIGJ1ZiA9IGlvLlN0cmluZ0lPKCkKICAgIGZpZWxkcyA9IFsiZG9jdW1lbnRfaWQiLCAic291cmNlX3BhdGgiLCAiZG9jX3R5cGUiLCAidmFsaWRhdGlvbl9zdGF0dXMiLAogICAgICAgICAgICAgICJleHRyYWN0aW9uX2NvbmZpZGVuY2UiLCAiY3JlYXRlZF9hdCJdCiAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihidWYsIGZpZWxkbmFtZXM9ZmllbGRzLCBleHRyYXNhY3Rpb249Imlnbm9yZSIpCiAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgd3JpdGVyLndyaXRlcm93cyhyZWNvcmRzKQogICAgcmV0dXJuIGJ1Zi5nZXR2YWx1ZSgpCg==",
    "/content/capstone/graph/__init__.py": "",
    "/content/capstone/graph/workflow.py": "IiIiTGFuZ0dyYXBoIG11bHRpLWFnZW50IHdvcmtmbG93IOKAlCBjbGFzc2lmeSArIGV4dHJhY3QgaW4gb25lIFZMTSBjYWxsLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKaW1wb3J0IHV1aWQKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBUeXBlZERpY3QKCmZyb20gbGFuZ2dyYXBoLmdyYXBoIGltcG9ydCBFTkQsIFN0YXRlR3JhcGgKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpmcm9tIGFnZW50cyBpbXBvcnQgY29ycmVjdGlvbl9hZ2VudCwgZXh0cmFjdGlvbl9hZ2VudCwgdmFsaWRhdGlvbl9hZ2VudAppbXBvcnQgY29uZmlnICAgIyByZWFkIGF0IGNhbGwtdGltZSBzbyBTZXR0aW5ncyBjaGFuZ2VzIHRha2UgZWZmZWN0IGltbWVkaWF0ZWx5CmZyb20gZGF0YWJhc2UgaW1wb3J0IHN0b3JhZ2UKZnJvbSBtb2RlbHMuc2NoZW1hcyBpbXBvcnQgRXh0cmFjdGlvblJlc3VsdCwgUHJvY2Vzc2luZ1JlY29yZCwgVmFsaWRhdGlvblJlc3VsdCwgVmFsaWRhdGlvblN0YXR1cwpmcm9tIHBpcGVsaW5lIGltcG9ydCBpbmdlc3Rpb24sIG9jcgoKCiMg4pSA4pSAIFN0YXRlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgSW52b2ljZVN0YXRlKFR5cGVkRGljdCk6CiAgICBkb2N1bWVudF9pZDogICAgICAgICAgc3RyCiAgICBzb3VyY2VfcGF0aDogICAgICAgICAgc3RyCiAgICBpbWFnZXM6ICAgICAgICAgICAgICAgbGlzdAogICAgb2NyX3RleHRzOiAgICAgICAgICAgIGxpc3QKICAgIGV4dHJhY3Rpb246ICAgICAgICAgICBPcHRpb25hbFtFeHRyYWN0aW9uUmVzdWx0XQogICAgdmFsaWRhdGlvbjogICAgICAgICAgIE9wdGlvbmFsW1ZhbGlkYXRpb25SZXN1bHRdCiAgICBjb3JyZWN0aW9uX2F0dGVtcHRzOiAgaW50CiAgICBmaW5hbF9zdGF0dXM6ICAgICAgICAgT3B0aW9uYWxbVmFsaWRhdGlvblN0YXR1c10KICAgIHByb2Nlc3Npbmdfbm90ZXM6ICAgICBsaXN0CiAgICB0aW1pbmdzOiAgICAgICAgICAgICAgZGljdCAgICAgICMge25vZGVfbmFtZTogc2Vjb25kc30KICAgIHBpcGVsaW5lX3N0YXJ0OiAgICAgICBmbG9hdCAgICAjIGVwb2NoIHRpbWUgd2hlbiBwaXBlbGluZSBzdGFydGVkCgoKIyDilIDilIAgVGltaW5nIGhlbHBlciDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmRlZiBfdGltZWQoc3RhdGU6IEludm9pY2VTdGF0ZSwgbm9kZTogc3RyLCBzdGFydDogZmxvYXQpIC0+IGZsb2F0OgogICAgIiIiUmVjb3JkIGVsYXBzZWQgdGltZSBmb3IgYSBub2RlIGFuZCByZXR1cm4gaXQuIiIiCiAgICBlbGFwc2VkID0gcm91bmQodGltZS50aW1lKCkgLSBzdGFydCwgMikKICAgIHN0YXRlWyJ0aW1pbmdzIl1bbm9kZV0gPSBlbGFwc2VkCiAgICByZXR1cm4gZWxhcHNlZAoKCiMg4pSA4pSAIE5vZGVzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIG5vZGVfaW5nZXN0KHN0YXRlOiBJbnZvaWNlU3RhdGUpIC0+IEludm9pY2VTdGF0ZToKICAgIHQwICAgICA9IHRpbWUudGltZSgpCiAgICBpbWFnZXMgPSBpbmdlc3Rpb24ubG9hZF9kb2N1bWVudChzdGF0ZVsic291cmNlX3BhdGgiXSkKICAgIHN0YXRlWyJpbWFnZXMiXSA9IGltYWdlcwogICAgZWxhcHNlZCA9IF90aW1lZChzdGF0ZSwgImluZ2VzdCIsIHQwKQogICAgc3RhdGVbInByb2Nlc3Npbmdfbm90ZXMiXS5hcHBlbmQoCiAgICAgICAgZiLwn5OlIEluZ2VzdGVkIHtsZW4oaW1hZ2VzKX0gcGFnZShzKSDCtyDij7Ege2VsYXBzZWR9cyIKICAgICkKICAgIHN0b3JhZ2UubG9nX2V2ZW50KHN0YXRlWyJkb2N1bWVudF9pZCJdLCAiaW5nZXN0ZWQiLAogICAgICAgICAgICAgICAgICAgICAgeyJwYWdlcyI6IGxlbihpbWFnZXMpLCAibGF0ZW5jeV9zIjogZWxhcHNlZH0pCiAgICByZXR1cm4gc3RhdGUKCgpkZWYgbm9kZV9leHRyYWN0KHN0YXRlOiBJbnZvaWNlU3RhdGUpIC0+IEludm9pY2VTdGF0ZToKICAgICIiIkNvbWJpbmVkIGNsYXNzaWZ5ICsgZXh0cmFjdCDigJQgb25lIFZMTSBjYWxsIGlkZW50aWZpZXMgdHlwZSBhbmQgcmVhZHMgYWxsIGZpZWxkcy4iIiIKICAgIHQwICAgICAgICAgICA9IHRpbWUudGltZSgpCiAgICBjb21iaW5lZF9vY3IgPSAiXG5cbiIuam9pbihzdGF0ZS5nZXQoIm9jcl90ZXh0cyIpIG9yIFtdKQogICAgcmVzdWx0ICAgICAgID0gZXh0cmFjdGlvbl9hZ2VudC5ydW4oc3RhdGVbImltYWdlcyJdLCBjb21iaW5lZF9vY3IpCiAgICBzdGF0ZVsiZXh0cmFjdGlvbiJdID0gcmVzdWx0CiAgICBlbGFwc2VkID0gX3RpbWVkKHN0YXRlLCAiZXh0cmFjdCIsIHQwKQoKICAgIGRvYyA9IHJlc3VsdC5leHRyYWN0ZWRfZGF0YQogICAgc3ViID0gZiIgwrcgc3VidHlwZToge2RvYy5kb2Nfc3VidHlwZX0iIGlmIGRvYy5kb2Nfc3VidHlwZSBlbHNlICIiCiAgICBzdGF0ZVsicHJvY2Vzc2luZ19ub3RlcyJdLmFwcGVuZCgKICAgICAgICBmIvCflI0gQ2xhc3NpZmllZCBhcyAne2RvYy5kb2NfdHlwZX0ne3N1Yn0gwrcgIgogICAgICAgIGYie2xlbihkb2MuZmllbGRzKX0gZmllbGRzIGV4dHJhY3RlZCDCtyAiCiAgICAgICAgZiJjb25maWRlbmNlIHtyZXN1bHQuY29uZmlkZW5jZTouMCV9IMK3IOKPsSB7ZWxhcHNlZH1zIgogICAgKQogICAgc3RvcmFnZS5sb2dfZXZlbnQoc3RhdGVbImRvY3VtZW50X2lkIl0sICJleHRyYWN0ZWQiLCB7CiAgICAgICAgImRvY190eXBlIjogICAgZG9jLmRvY190eXBlLAogICAgICAgICJmaWVsZF9jb3VudCI6IGxlbihkb2MuZmllbGRzKSwKICAgICAgICAiY29uZmlkZW5jZSI6ICByZXN1bHQuY29uZmlkZW5jZSwKICAgICAgICAibGF0ZW5jeV9zIjogICBlbGFwc2VkLAogICAgfSkKICAgIHJldHVybiBzdGF0ZQoKCmRlZiBub2RlX29jcl9mYWxsYmFjayhzdGF0ZTogSW52b2ljZVN0YXRlKSAtPiBJbnZvaWNlU3RhdGU6CiAgICAiIiJSdW4gVGVzc2VyYWN0IGFuZCBtZXJnZSBvdXRwdXQgdG8gZmlsbCBnYXBzIGZyb20gVkxNIGV4dHJhY3Rpb24uIiIiCiAgICB0MCAgICAgICA9IHRpbWUudGltZSgpCiAgICB0ZXh0cyAgICA9IFtvY3Iub2NyX2V4dHJhY3RfdGV4dChpbWcpIGZvciBpbWcgaW4gc3RhdGVbImltYWdlcyJdXQogICAgY29tYmluZWQgPSAiXG5cbiIuam9pbih0ZXh0cykKICAgIHN0YXRlWyJvY3JfdGV4dHMiXSA9IHRleHRzCgogICAgaWYgc3RhdGVbImV4dHJhY3Rpb24iXToKICAgICAgICBtZXJnZWQgPSBvY3IubWVyZ2Vfb2NyX3dpdGhfZXh0cmFjdGlvbihjb21iaW5lZCwgc3RhdGVbImV4dHJhY3Rpb24iXS5leHRyYWN0ZWRfZGF0YSkKICAgICAgICBzdGF0ZVsiZXh0cmFjdGlvbiJdID0gRXh0cmFjdGlvblJlc3VsdCgKICAgICAgICAgICAgcmF3X29jcl90ZXh0PWNvbWJpbmVkLAogICAgICAgICAgICBleHRyYWN0ZWRfZGF0YT1tZXJnZWQsCiAgICAgICAgICAgIGNvbmZpZGVuY2U9c3RhdGVbImV4dHJhY3Rpb24iXS5jb25maWRlbmNlLAogICAgICAgICAgICB2bG1fcmVzcG9uc2U9c3RhdGVbImV4dHJhY3Rpb24iXS52bG1fcmVzcG9uc2UsCiAgICAgICAgICAgIG9jcl91c2VkPVRydWUsCiAgICAgICAgKQogICAgZWxhcHNlZCA9IF90aW1lZChzdGF0ZSwgIm9jcl9mYWxsYmFjayIsIHQwKQogICAgc3RhdGVbInByb2Nlc3Npbmdfbm90ZXMiXS5hcHBlbmQoCiAgICAgICAgZiLwn5OEIE9DUiBmYWxsYmFjayByYW4g4oCUIG1lcmdlZCB3aXRoIFZMTSBvdXRwdXQgwrcg4o+xIHtlbGFwc2VkfXMiCiAgICApCiAgICBzdG9yYWdlLmxvZ19ldmVudChzdGF0ZVsiZG9jdW1lbnRfaWQiXSwgIm9jcl9mYWxsYmFjayIsCiAgICAgICAgICAgICAgICAgICAgICB7ImNoYXJzIjogbGVuKGNvbWJpbmVkKSwgImxhdGVuY3lfcyI6IGVsYXBzZWR9KQogICAgcmV0dXJuIHN0YXRlCgoKZGVmIG5vZGVfdmFsaWRhdGUoc3RhdGU6IEludm9pY2VTdGF0ZSkgLT4gSW52b2ljZVN0YXRlOgogICAgdDAgICAgID0gdGltZS50aW1lKCkKICAgIHJlc3VsdCA9IHZhbGlkYXRpb25fYWdlbnQucnVuKHN0YXRlWyJleHRyYWN0aW9uIl0pCiAgICBzdGF0ZVsidmFsaWRhdGlvbiJdID0gcmVzdWx0CiAgICBlbGFwc2VkID0gX3RpbWVkKHN0YXRlLCAidmFsaWRhdGUiLCB0MCkKCiAgICBtc2cgPSAiVmFsaWRhdGlvbiBwYXNzZWQiIGlmIHJlc3VsdC5pc192YWxpZCBlbHNlIGYiVmFsaWRhdGlvbiBmYWlsZWQ6IHtyZXN1bHQuZmFpbGVkX2ZpZWxkc30iCiAgICBzdGF0ZVsicHJvY2Vzc2luZ19ub3RlcyJdLmFwcGVuZChmIuKchSB7bXNnfSDCtyDij7Ege2VsYXBzZWR9cyIpCiAgICBzdG9yYWdlLmxvZ19ldmVudChzdGF0ZVsiZG9jdW1lbnRfaWQiXSwgInZhbGlkYXRlZCIsIHsKICAgICAgICAidmFsaWQiOiAgICAgIHJlc3VsdC5pc192YWxpZCwKICAgICAgICAiZmFpbGVkIjogICAgIHJlc3VsdC5mYWlsZWRfZmllbGRzLAogICAgICAgICJsYXRlbmN5X3MiOiAgZWxhcHNlZCwKICAgIH0pCiAgICByZXR1cm4gc3RhdGUKCgpkZWYgbm9kZV9jb3JyZWN0KHN0YXRlOiBJbnZvaWNlU3RhdGUpIC0+IEludm9pY2VTdGF0ZToKICAgIHQwID0gdGltZS50aW1lKCkKICAgIHN0YXRlWyJjb3JyZWN0aW9uX2F0dGVtcHRzIl0gKz0gMQogICAgdXBkYXRlZCA9IGNvcnJlY3Rpb25fYWdlbnQucnVuKAogICAgICAgIHN0YXRlWyJpbWFnZXMiXSwKICAgICAgICBzdGF0ZVsiZXh0cmFjdGlvbiJdLAogICAgICAgIHN0YXRlWyJ2YWxpZGF0aW9uIl0sCiAgICApCiAgICBzdGF0ZVsiZXh0cmFjdGlvbiJdID0gdXBkYXRlZAogICAgZWxhcHNlZCA9IF90aW1lZChzdGF0ZSwgZiJjb3JyZWN0X3tzdGF0ZVsnY29ycmVjdGlvbl9hdHRlbXB0cyddfSIsIHQwKQogICAgc3RhdGVbInByb2Nlc3Npbmdfbm90ZXMiXS5hcHBlbmQoCiAgICAgICAgZiLwn5SEIEF1dG8tY29ycmVjdGlvbiBhdHRlbXB0IHtzdGF0ZVsnY29ycmVjdGlvbl9hdHRlbXB0cyddfSDCtyDij7Ege2VsYXBzZWR9cyIKICAgICkKICAgIHN0b3JhZ2UubG9nX2V2ZW50KHN0YXRlWyJkb2N1bWVudF9pZCJdLCAiY29ycmVjdGVkIiwKICAgICAgICAgICAgICAgICAgICAgIHsiYXR0ZW1wdCI6IHN0YXRlWyJjb3JyZWN0aW9uX2F0dGVtcHRzIl0sICJsYXRlbmN5X3MiOiBlbGFwc2VkfSkKICAgIHJldHVybiBzdGF0ZQoKCmRlZiBfc2F2ZShzdGF0ZTogSW52b2ljZVN0YXRlLCBzdGF0dXM6IFZhbGlkYXRpb25TdGF0dXMpIC0+IE5vbmU6CiAgICAiIiJTaGFyZWQgc2F2ZSBsb2dpYyBmb3Igc3RvcmUgYW5kIHJldmlld19xdWV1ZSBub2Rlcy4iIiIKICAgIHYgICA9IHN0YXRlWyJ2YWxpZGF0aW9uIl0KICAgIGV4dCA9IHN0YXRlWyJleHRyYWN0aW9uIl0KCiAgICAjIFRvdGFsIHBpcGVsaW5lIHRpbWUKICAgIHRvdGFsID0gcm91bmQodGltZS50aW1lKCkgLSBzdGF0ZVsicGlwZWxpbmVfc3RhcnQiXSwgMikKICAgIHN0YXRlWyJ0aW1pbmdzIl1bInRvdGFsIl0gPSB0b3RhbAogICAgc3RhdGVbInByb2Nlc3Npbmdfbm90ZXMiXS5hcHBlbmQoZiLij7EgVG90YWwgcGlwZWxpbmUgdGltZToge3RvdGFsfXMiKQoKICAgIHRyeToKICAgICAgc3RvcmFnZS5zYXZlX3JlY29yZChQcm9jZXNzaW5nUmVjb3JkKAogICAgICAgIGRvY3VtZW50X2lkPXN0YXRlWyJkb2N1bWVudF9pZCJdLAogICAgICAgIHNvdXJjZV9wYXRoPXN0YXRlWyJzb3VyY2VfcGF0aCJdLAogICAgICAgIGRvY190eXBlPWV4dC5leHRyYWN0ZWRfZGF0YS5kb2NfdHlwZSBpZiBleHQgZWxzZSAidW5rbm93biIsCiAgICAgICAgcmF3X29jcl90ZXh0PSJcblxuIi5qb2luKHN0YXRlLmdldCgib2NyX3RleHRzIikgb3IgW10pLAogICAgICAgIHZsbV9leHRyYWN0aW9uPXsKICAgICAgICAgICAgImZpZWxkcyI6ICAgICBleHQuZXh0cmFjdGVkX2RhdGEuZmllbGRzLAogICAgICAgICAgICAibGluZV9pdGVtcyI6IGV4dC5leHRyYWN0ZWRfZGF0YS5saW5lX2l0ZW1zLAogICAgICAgICAgICAidGltaW5ncyI6ICAgIHN0YXRlWyJ0aW1pbmdzIl0sCiAgICAgICAgfSBpZiBleHQgZWxzZSB7fSwKICAgICAgICBjb3JyZWN0aW9uc19hcHBsaWVkPVsKICAgICAgICAgICAgZnYubW9kZWxfZHVtcCgpIGZvciBmdiBpbiAodi5maWVsZF92YWxpZGF0aW9ucyBpZiB2IGVsc2UgW10pCiAgICAgICAgICAgIGlmIGZ2LnN0YXR1cyA9PSBWYWxpZGF0aW9uU3RhdHVzLkNPUlJFQ1RFRAogICAgICAgIF0sCiAgICAgICAgZmluYWxfZGF0YT17CiAgICAgICAgICAgICJkb2NfdHlwZSI6ICAgICAgICAgZXh0LmV4dHJhY3RlZF9kYXRhLmRvY190eXBlLAogICAgICAgICAgICAiZG9jX3N1YnR5cGUiOiAgICAgIGV4dC5leHRyYWN0ZWRfZGF0YS5kb2Nfc3VidHlwZSwKICAgICAgICAgICAgImZpZWxkcyI6ICAgICAgICAgICBleHQuZXh0cmFjdGVkX2RhdGEuZmllbGRzLAogICAgICAgICAgICAibGluZV9pdGVtcyI6ICAgICAgIGV4dC5leHRyYWN0ZWRfZGF0YS5saW5lX2l0ZW1zLAogICAgICAgICAgICAiZXh0cmFjdGlvbl9ub3RlcyI6IGV4dC5leHRyYWN0ZWRfZGF0YS5leHRyYWN0aW9uX25vdGVzLAogICAgICAgICAgICAidGltaW5ncyI6ICAgICAgICAgIHN0YXRlWyJ0aW1pbmdzIl0sCiAgICAgICAgfSBpZiBleHQgZWxzZSB7fSwKICAgICAgICB2YWxpZGF0aW9uX3N0YXR1cz1zdGF0dXMsCiAgICAgICAgZXh0cmFjdGlvbl9jb25maWRlbmNlPWV4dC5jb25maWRlbmNlIGlmIGV4dCBlbHNlIDAuMCwKICAgICAgICBwcm9jZXNzaW5nX25vdGVzPXN0YXRlWyJwcm9jZXNzaW5nX25vdGVzIl0sCiAgICAgICkpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiJbd29ya2Zsb3ddIEVSUk9SOiBjb3VsZCBub3Qgc2F2ZSByZWNvcmQge3N0YXRlWydkb2N1bWVudF9pZCddfToge2V9IikKICAgICAgICByYWlzZSAgICMgcmUtcmFpc2Ugc28gdGhlIHBpcGVsaW5lIHN1cmZhY2UgdGhlIGZhaWx1cmUgdmlzaWJseQoKCmRlZiBub2RlX3N0b3JlKHN0YXRlOiBJbnZvaWNlU3RhdGUpIC0+IEludm9pY2VTdGF0ZToKICAgIHYgICAgICA9IHN0YXRlWyJ2YWxpZGF0aW9uIl0KICAgIHN0YXR1cyA9IFZhbGlkYXRpb25TdGF0dXMuVkFMSUQgaWYgKHYgYW5kIHYuaXNfdmFsaWQpIGVsc2UgVmFsaWRhdGlvblN0YXR1cy5DT1JSRUNURUQKICAgIHN0YXRlWyJmaW5hbF9zdGF0dXMiXSA9IHN0YXR1cwogICAgX3NhdmUoc3RhdGUsIHN0YXR1cykKICAgIHN0b3JhZ2UubG9nX2V2ZW50KHN0YXRlWyJkb2N1bWVudF9pZCJdLCAic3RvcmVkIiwKICAgICAgICAgICAgICAgICAgICAgIHsic3RhdHVzIjogc3RhdHVzLnZhbHVlLCAidG90YWxfcyI6IHN0YXRlWyJ0aW1pbmdzIl0uZ2V0KCJ0b3RhbCIpfSkKICAgIHJldHVybiBzdGF0ZQoKCmRlZiBub2RlX3Jldmlld19xdWV1ZShzdGF0ZTogSW52b2ljZVN0YXRlKSAtPiBJbnZvaWNlU3RhdGU6CiAgICBzdGF0ZVsiZmluYWxfc3RhdHVzIl0gPSBWYWxpZGF0aW9uU3RhdHVzLlBFTkRJTkcKICAgIHN0YXRlWyJwcm9jZXNzaW5nX25vdGVzIl0uYXBwZW5kKCJTZW50IHRvIGh1bWFuIHJldmlldyBxdWV1ZSIpCiAgICBfc2F2ZShzdGF0ZSwgVmFsaWRhdGlvblN0YXR1cy5QRU5ESU5HKQogICAgc3RvcmFnZS5sb2dfZXZlbnQoc3RhdGVbImRvY3VtZW50X2lkIl0sICJyZXZpZXdfcXVldWUiLAogICAgICAgICAgICAgICAgICAgICAgeyJyZWFzb24iOiAibG93IGNvbmZpZGVuY2Ugb3IgbWF4IGNvcnJlY3Rpb25zIiwKICAgICAgICAgICAgICAgICAgICAgICAidG90YWxfcyI6IHN0YXRlWyJ0aW1pbmdzIl0uZ2V0KCJ0b3RhbCIpfSkKICAgIHJldHVybiBzdGF0ZQoKCiMg4pSA4pSAIFJvdXRlcnMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgX3Nob3VsZF9ydW5fb2NyKHN0YXRlOiBJbnZvaWNlU3RhdGUpIC0+IHN0cjoKICAgIGNvbmYgPSBzdGF0ZVsiZXh0cmFjdGlvbiJdLmNvbmZpZGVuY2UgaWYgc3RhdGVbImV4dHJhY3Rpb24iXSBlbHNlIDAuMAogICAgcmV0dXJuICJvY3JfZmFsbGJhY2siIGlmIGNvbmYgPCBjb25maWcuRVhUUkFDVElPTl9DT05GX1RIUkVTSE9MRCBlbHNlICJ2YWxpZGF0ZSIKCgpkZWYgX3JvdXRlX2FmdGVyX3ZhbGlkYXRpb24oc3RhdGU6IEludm9pY2VTdGF0ZSkgLT4gc3RyOgogICAgdiAgICA9IHN0YXRlWyJ2YWxpZGF0aW9uIl0KICAgIGNvbmYgPSBzdGF0ZVsiZXh0cmFjdGlvbiJdLmNvbmZpZGVuY2UgaWYgc3RhdGVbImV4dHJhY3Rpb24iXSBlbHNlIDAuMAoKICAgIGlmIHYgYW5kIG5vdCB2LmlzX3ZhbGlkOgogICAgICAgIGlmIHN0YXRlWyJjb3JyZWN0aW9uX2F0dGVtcHRzIl0gPCBjb25maWcuTUFYX0NPUlJFQ1RJT05fQVRURU1QVFM6CiAgICAgICAgICAgIHJldHVybiAiY29ycmVjdCIKICAgICAgICByZXR1cm4gInJldmlld19xdWV1ZSIKCiAgICByZXR1cm4gInN0b3JlIiBpZiBjb25mID49IGNvbmZpZy5BVVRPX0FQUFJPVkVfVEhSRVNIT0xEIGVsc2UgInJldmlld19xdWV1ZSIKCgojIOKUgOKUgCBCdWlsZCAmIHJ1biDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmRlZiBidWlsZF9ncmFwaCgpOgogICAgc3RvcmFnZS5pbml0X2RiKCkKICAgIGcgPSBTdGF0ZUdyYXBoKEludm9pY2VTdGF0ZSkKCiAgICBnLmFkZF9ub2RlKCJpbmdlc3QiLCAgICAgICBub2RlX2luZ2VzdCkKICAgIGcuYWRkX25vZGUoImV4dHJhY3QiLCAgICAgIG5vZGVfZXh0cmFjdCkKICAgIGcuYWRkX25vZGUoIm9jcl9mYWxsYmFjayIsIG5vZGVfb2NyX2ZhbGxiYWNrKQogICAgZy5hZGRfbm9kZSgidmFsaWRhdGUiLCAgICAgbm9kZV92YWxpZGF0ZSkKICAgIGcuYWRkX25vZGUoImNvcnJlY3QiLCAgICAgIG5vZGVfY29ycmVjdCkKICAgIGcuYWRkX25vZGUoInN0b3JlIiwgICAgICAgIG5vZGVfc3RvcmUpCiAgICBnLmFkZF9ub2RlKCJyZXZpZXdfcXVldWUiLCBub2RlX3Jldmlld19xdWV1ZSkKCiAgICBnLnNldF9lbnRyeV9wb2ludCgiaW5nZXN0IikKICAgIGcuYWRkX2VkZ2UoImluZ2VzdCIsICJleHRyYWN0IikKICAgIGcuYWRkX2NvbmRpdGlvbmFsX2VkZ2VzKCJleHRyYWN0IiwgX3Nob3VsZF9ydW5fb2NyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsib2NyX2ZhbGxiYWNrIjogIm9jcl9mYWxsYmFjayIsICJ2YWxpZGF0ZSI6ICJ2YWxpZGF0ZSJ9KQogICAgZy5hZGRfZWRnZSgib2NyX2ZhbGxiYWNrIiwgInZhbGlkYXRlIikKICAgIGcuYWRkX2NvbmRpdGlvbmFsX2VkZ2VzKCJ2YWxpZGF0ZSIsIF9yb3V0ZV9hZnRlcl92YWxpZGF0aW9uLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsiY29ycmVjdCI6ICJjb3JyZWN0IiwgInN0b3JlIjogInN0b3JlIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJldmlld19xdWV1ZSI6ICJyZXZpZXdfcXVldWUifSkKICAgIGcuYWRkX2VkZ2UoImNvcnJlY3QiLCAgICAgICJ2YWxpZGF0ZSIpCiAgICBnLmFkZF9lZGdlKCJzdG9yZSIsICAgICAgICBFTkQpCiAgICBnLmFkZF9lZGdlKCJyZXZpZXdfcXVldWUiLCBFTkQpCgogICAgcmV0dXJuIGcuY29tcGlsZSgpCgoKZGVmIF9pbml0aWFsX3N0YXRlKHBhdGg6IHN0cikgLT4gSW52b2ljZVN0YXRlOgogICAgcmV0dXJuIHsKICAgICAgICAiZG9jdW1lbnRfaWQiOiAgICAgICAgIHN0cih1dWlkLnV1aWQ0KCkpLAogICAgICAgICJzb3VyY2VfcGF0aCI6ICAgICAgICAgc3RyKHBhdGgpLAogICAgICAgICJpbWFnZXMiOiAgICAgICAgICAgICAgW10sCiAgICAgICAgIm9jcl90ZXh0cyI6ICAgICAgICAgICBbXSwKICAgICAgICAiZXh0cmFjdGlvbiI6ICAgICAgICAgIE5vbmUsCiAgICAgICAgInZhbGlkYXRpb24iOiAgICAgICAgICBOb25lLAogICAgICAgICJjb3JyZWN0aW9uX2F0dGVtcHRzIjogMCwKICAgICAgICAiZmluYWxfc3RhdHVzIjogICAgICAgIE5vbmUsCiAgICAgICAgInByb2Nlc3Npbmdfbm90ZXMiOiAgICBbXSwKICAgICAgICAidGltaW5ncyI6ICAgICAgICAgICAgIHt9LAogICAgICAgICJwaXBlbGluZV9zdGFydCI6ICAgICAgdGltZS50aW1lKCksCiAgICB9CgoKZGVmIHByb2Nlc3NfZG9jdW1lbnQocGF0aDogc3RyKSAtPiBJbnZvaWNlU3RhdGU6CiAgICByZXR1cm4gYnVpbGRfZ3JhcGgoKS5pbnZva2UoX2luaXRpYWxfc3RhdGUocGF0aCkpCgoKZGVmIHByb2Nlc3NfZG9jdW1lbnRfc3RyZWFtKHBhdGg6IHN0cik6CiAgICAiIiJZaWVsZCAobm9kZV9uYW1lLCBzdGF0ZSkgYWZ0ZXIgZWFjaCBub2RlIGNvbXBsZXRlcy4iIiIKICAgIGdyYXBoID0gYnVpbGRfZ3JhcGgoKQogICAgZm9yIGNodW5rIGluIGdyYXBoLnN0cmVhbShfaW5pdGlhbF9zdGF0ZShwYXRoKSk6CiAgICAgICAgbm9kZV9uYW1lICAgPSBsaXN0KGNodW5rLmtleXMoKSlbMF0KICAgICAgICB5aWVsZCBub2RlX25hbWUsIGNodW5rW25vZGVfbmFtZV0K",
    "/content/capstone/app.py": "aW1wb3J0IHN0cmVhbWxpdCBhcyBzdAoKc3Quc2V0X3BhZ2VfY29uZmlnKAogICAgcGFnZV90aXRsZT0iRG9jIEFnZW50IOKAlCBJbnZvaWNlIEludGVsbGlnZW5jZSIsCiAgICBwYWdlX2ljb249IvCfp74iLAogICAgbGF5b3V0PSJ3aWRlIiwKICAgIGluaXRpYWxfc2lkZWJhcl9zdGF0ZT0iZXhwYW5kZWQiLAopCgojIEltcG9ydHMgYWZ0ZXIgc2V0X3BhZ2VfY29uZmlnCmZyb20gdWkuYXV0aCBpbXBvcnQgY2hlY2tfYXV0aApmcm9tIHVpLnN0eWxlcyBpbXBvcnQgaW5qZWN0X2NzcwppbXBvcnQgdWkucGFnZXMubG9naW4gYXMgbG9naW5fcGFnZQppbXBvcnQgdWkucGFnZXMuZGFzaGJvYXJkIGFzIGRhc2hib2FyZF9wYWdlCmltcG9ydCB1aS5wYWdlcy5wcm9jZXNzIGFzIHByb2Nlc3NfcGFnZQppbXBvcnQgdWkucGFnZXMucmV2aWV3IGFzIHJldmlld19wYWdlCmltcG9ydCB1aS5wYWdlcy5oaXN0b3J5IGFzIGhpc3RvcnlfcGFnZQppbXBvcnQgdWkucGFnZXMudGVjaHN0YWNrIGFzIHRlY2hzdGFja19wYWdlCmltcG9ydCB1aS5wYWdlcy5zZXR0aW5ncyBhcyBzZXR0aW5nc19wYWdlCmZyb20gdWkuY29tcG9uZW50cy5zaWRlYmFyIGltcG9ydCByZW5kZXJfc2lkZWJhcgoKaW5qZWN0X2NzcygpCgppZiBub3QgY2hlY2tfYXV0aCgpOgogICAgbG9naW5fcGFnZS5yZW5kZXIoKQogICAgc3Quc3RvcCgpCgpwYWdlID0gcmVuZGVyX3NpZGViYXIoKQoKX1BBR0VTID0gewogICAgIkRhc2hib2FyZCI6ICAgICAgICBkYXNoYm9hcmRfcGFnZS5yZW5kZXIsCiAgICAiUHJvY2VzcyBEb2N1bWVudCI6IHByb2Nlc3NfcGFnZS5yZW5kZXIsCiAgICAiUmV2aWV3IFF1ZXVlIjogICAgIHJldmlld19wYWdlLnJlbmRlciwKICAgICJIaXN0b3J5IjogICAgICAgICAgaGlzdG9yeV9wYWdlLnJlbmRlciwKICAgICJUZWNoIFN0YWNrIjogICAgICAgdGVjaHN0YWNrX3BhZ2UucmVuZGVyLAogICAgIlNldHRpbmdzIjogICAgICAgICBzZXR0aW5nc19wYWdlLnJlbmRlciwKfQoKX1BBR0VTLmdldChwYWdlLCBkYXNoYm9hcmRfcGFnZS5yZW5kZXIpKCkK",
    "/content/capstone/ui/__init__.py": "",
    "/content/capstone/ui/auth.py": "aW1wb3J0IHN0cmVhbWxpdCBhcyBzdAoKX0NSRURFTlRJQUxTID0geyJkZW1vIjogImRlbW8ifQoKCmRlZiBjaGVja19hdXRoKCkgLT4gYm9vbDoKICAgIHJldHVybiBzdC5zZXNzaW9uX3N0YXRlLmdldCgibG9nZ2VkX2luIiwgRmFsc2UpCgoKZGVmIGxvZ2luKHVzZXJuYW1lOiBzdHIsIHBhc3N3b3JkOiBzdHIpIC0+IGJvb2w6CiAgICBpZiBfQ1JFREVOVElBTFMuZ2V0KHVzZXJuYW1lKSA9PSBwYXNzd29yZDoKICAgICAgICBzdC5zZXNzaW9uX3N0YXRlLmxvZ2dlZF9pbiA9IFRydWUKICAgICAgICBzdC5zZXNzaW9uX3N0YXRlLnVzZXJuYW1lID0gdXNlcm5hbWUKICAgICAgICByZXR1cm4gVHJ1ZQogICAgcmV0dXJuIEZhbHNlCgoKZGVmIGxvZ291dCgpIC0+IE5vbmU6CiAgICBmb3Iga2V5IGluICgibG9nZ2VkX2luIiwgInVzZXJuYW1lIiwgImN1cnJlbnRfcGFnZSIpOgogICAgICAgIHN0LnNlc3Npb25fc3RhdGUucG9wKGtleSwgTm9uZSkK",
    "/content/capstone/ui/styles.py": "...",
    "/content/capstone/ui/pages/__init__.py": "",
    "/content/capstone/ui/pages/login.py": "aW1wb3J0IHN0cmVhbWxpdCBhcyBzdApmcm9tIHVpLmF1dGggaW1wb3J0IGxvZ2luCgoKZGVmIHJlbmRlcigpIC0+IE5vbmU6CiAgICAjIFRocmVlLWNvbHVtbiBjZW50ZXJpbmcgdHJpY2sKICAgIF8sIGNlbnRlciwgXyA9IHN0LmNvbHVtbnMoWzEsIDEuMiwgMV0pCgogICAgd2l0aCBjZW50ZXI6CiAgICAgICAgc3QubWFya2Rvd24oIjxicj48YnI+IiwgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSkKICAgICAgICBzdC5tYXJrZG93bigKICAgICAgICAgICAgIiIiCiAgICAgICAgICAgIDxkaXYgc3R5bGU9InRleHQtYWxpZ246Y2VudGVyO21hcmdpbi1ib3R0b206MzJweDsiPgogICAgICAgICAgICAgICAgPGRpdiBzdHlsZT0iZm9udC1zaXplOjMuNXJlbTtsaW5lLWhlaWdodDoxOyI+8J+nvjwvZGl2PgogICAgICAgICAgICAgICAgPGgxIHN0eWxlPSJmb250LXNpemU6MnJlbTtmb250LXdlaWdodDo4MDA7Y29sb3I6IzFhMjc0NDttYXJnaW46OHB4IDAgNHB4OyI+RG9jIEFnZW50PC9oMT4KICAgICAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojNmI3YTk5O2ZvbnQtc2l6ZTouOTVyZW07bWFyZ2luOjA7Ij4KICAgICAgICAgICAgICAgICAgICBJbnZvaWNlICZhbXA7IERvY3VtZW50IEludGVsbGlnZW5jZSBQbGF0Zm9ybQogICAgICAgICAgICAgICAgPC9wPgogICAgICAgICAgICA8L2Rpdj4KICAgICAgICAgICAgIiIiLAogICAgICAgICAgICB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlLAogICAgICAgICkKCiAgICAgICAgd2l0aCBzdC5jb250YWluZXIoYm9yZGVyPVRydWUpOgogICAgICAgICAgICBzdC5tYXJrZG93bigKICAgICAgICAgICAgICAgICI8cCBzdHlsZT0nZm9udC1zaXplOjEuMXJlbTtmb250LXdlaWdodDo3MDA7Y29sb3I6IzFhMjc0NDttYXJnaW4tYm90dG9tOjIwcHg7Jz5TaWduIGluIHRvIHlvdXIgYWNjb3VudDwvcD4iLAogICAgICAgICAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgdXNlcm5hbWUgPSBzdC50ZXh0X2lucHV0KAogICAgICAgICAgICAgICAgIlVzZXJuYW1lIiwKICAgICAgICAgICAgICAgIHBsYWNlaG9sZGVyPSJFbnRlciB1c2VybmFtZSIsCiAgICAgICAgICAgICAgICBrZXk9ImxvZ2luX3VzZXJuYW1lIiwKICAgICAgICAgICAgKQogICAgICAgICAgICBwYXNzd29yZCA9IHN0LnRleHRfaW5wdXQoCiAgICAgICAgICAgICAgICAiUGFzc3dvcmQiLAogICAgICAgICAgICAgICAgdHlwZT0icGFzc3dvcmQiLAogICAgICAgICAgICAgICAgcGxhY2Vob2xkZXI9IkVudGVyIHBhc3N3b3JkIiwKICAgICAgICAgICAgICAgIGtleT0ibG9naW5fcGFzc3dvcmQiLAogICAgICAgICAgICApCgogICAgICAgICAgICBzdC5tYXJrZG93bigiPGJyLz4iLCB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlKQoKICAgICAgICAgICAgaWYgc3QuYnV0dG9uKCJTaWduIEluIiwgdHlwZT0icHJpbWFyeSIsIHVzZV9jb250YWluZXJfd2lkdGg9VHJ1ZSk6CiAgICAgICAgICAgICAgICBpZiBub3QgdXNlcm5hbWUgb3Igbm90IHBhc3N3b3JkOgogICAgICAgICAgICAgICAgICAgIHN0Lndhcm5pbmcoIlBsZWFzZSBlbnRlciBib3RoIHVzZXJuYW1lIGFuZCBwYXNzd29yZC4iKQogICAgICAgICAgICAgICAgZWxpZiBsb2dpbih1c2VybmFtZSwgcGFzc3dvcmQpOgogICAgICAgICAgICAgICAgICAgIHN0LnN1Y2Nlc3MoIkxvZ2luIHN1Y2Nlc3NmdWwhIikKICAgICAgICAgICAgICAgICAgICBzdC5yZXJ1bigpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHN0LmVycm9yKCJJbnZhbGlkIGNyZWRlbnRpYWxzLiBUcnkgZGVtbyAvIGRlbW8iKQoKICAgICAgICBzdC5tYXJrZG93bigKICAgICAgICAgICAgIjxwIHN0eWxlPSd0ZXh0LWFsaWduOmNlbnRlcjtmb250LXNpemU6Ljc4cmVtO2NvbG9yOiM5YWE1YmU7bWFyZ2luLXRvcDoxMnB4Oyc+IgogICAgICAgICAgICAiRGVtbyBjcmVkZW50aWFsczogPHN0cm9uZz5kZW1vPC9zdHJvbmc+IC8gPHN0cm9uZz5kZW1vPC9zdHJvbmc+IgogICAgICAgICAgICAiPC9wPiIsCiAgICAgICAgICAgIHVuc2FmZV9hbGxvd19odG1sPVRydWUsCiAgICAgICAgKQo=",
    "/content/capstone/ui/pages/dashboard.py": "IiIiRGFzaGJvYXJkIHBhZ2Ug4oCUIGxpdmUgS1BJcywgcmVjZW50IGFjdGl2aXR5LCBwaXBlbGluZSBoZWFsdGguIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwoKaW1wb3J0IHN0cmVhbWxpdCBhcyBzdAoKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLmdldGN3ZCgpKQoKZnJvbSB1aS5zdHlsZXMgaW1wb3J0IGJhZGdlLCBwYWdlX2hlYWRlcgpmcm9tIHVpLmNvbXBvbmVudHMubWV0cmljcyBpbXBvcnQgcmVuZGVyX2twaV9yb3cKCgpkZWYgX2xvYWRfc3RhdHMoKSAtPiBkaWN0OgogICAgdHJ5OgogICAgICAgIGZyb20gZGF0YWJhc2Uuc3RvcmFnZSBpbXBvcnQgZ2V0X3N0YXRzCiAgICAgICAgcmV0dXJuIGdldF9zdGF0cygpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiB7InRvdGFsIjogMCwgInN1Y2Nlc3NfcmF0ZSI6IDAuMCwgInBlbmRpbmciOiAwLCAidG9kYXkiOiAwLCAiYnlfdHlwZSI6IHt9fQoKCmRlZiBfbG9hZF9yZWNlbnQobGltaXQ6IGludCA9IDYpIC0+IGxpc3RbZGljdF06CiAgICB0cnk6CiAgICAgICAgZnJvbSBkYXRhYmFzZS5zdG9yYWdlIGltcG9ydCBsaXN0X3JlY29yZHMKICAgICAgICByZXR1cm4gbGlzdF9yZWNvcmRzKGxpbWl0PWxpbWl0KQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gW10KCgpkZWYgcmVuZGVyKCkgLT4gTm9uZToKICAgIHBhZ2VfaGVhZGVyKCLwn5OKIiwgIkRhc2hib2FyZCIsICJPdmVydmlldyBvZiBkb2N1bWVudCBwcm9jZXNzaW5nIGFjdGl2aXR5IikKCiAgICBzdGF0cyAgPSBfbG9hZF9zdGF0cygpCiAgICByZWNlbnQgPSBfbG9hZF9yZWNlbnQoKQoKICAgIHJlbmRlcl9rcGlfcm93KAogICAgICAgIHRvdGFsPXN0YXRzWyJ0b3RhbCJdLAogICAgICAgIHN1Y2Nlc3NfcmF0ZT1zdGF0c1sic3VjY2Vzc19yYXRlIl0sCiAgICAgICAgcGVuZGluZz1zdGF0c1sicGVuZGluZyJdLAogICAgICAgIHRvZGF5PXN0YXRzWyJ0b2RheSJdLAogICAgKQoKICAgIHN0Lm1hcmtkb3duKCI8YnIvPiIsIHVuc2FmZV9hbGxvd19odG1sPVRydWUpCiAgICBsZWZ0LCByaWdodCA9IHN0LmNvbHVtbnMoWzIsIDFdKQoKICAgIHdpdGggbGVmdDoKICAgICAgICBzdC5tYXJrZG93bigiIyMjIyBSZWNlbnQgQWN0aXZpdHkiKQogICAgICAgIGlmIG5vdCByZWNlbnQ6CiAgICAgICAgICAgIHN0LmluZm8oIk5vIGRvY3VtZW50cyBwcm9jZXNzZWQgeWV0LiBHbyB0byAqKlByb2Nlc3MgRG9jdW1lbnQqKiB0byBnZXQgc3RhcnRlZC4iKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZvciBkb2MgaW4gcmVjZW50OgogICAgICAgICAgICAgICAgZmluYWwgPSBqc29uLmxvYWRzKGRvYy5nZXQoImZpbmFsX2RhdGEiKSBvciAie30iKQogICAgICAgICAgICAgICAgc3RhdHVzID0gZG9jLmdldCgidmFsaWRhdGlvbl9zdGF0dXMiLCAidW5rbm93biIpCiAgICAgICAgICAgICAgICBjb25mICAgPSBkb2MuZ2V0KCJleHRyYWN0aW9uX2NvbmZpZGVuY2UiLCAwLjApCiAgICAgICAgICAgICAgICBiICAgICAgPSBiYWRnZShzdGF0dXMsIHN0YXR1cyBpZiBzdGF0dXMgaW4gKCJ2YWxpZCIsImZhaWxlZCIsImNvcnJlY3RlZCIsInBlbmRpbmdfcmV2aWV3IikgZWxzZSAicmV2aWV3IikKICAgICAgICAgICAgICAgIGZuYW1lICA9IChkb2MuZ2V0KCJzb3VyY2VfcGF0aCIpIG9yICIiKS5zcGxpdCgiLyIpWy0xXSBvciBkb2NbImRvY3VtZW50X2lkIl1bOjEyXQogICAgICAgICAgICAgICAgdmVuZG9yID0gZmluYWwuZ2V0KCJ2ZW5kb3JfbmFtZSIpIG9yICLigJQiCiAgICAgICAgICAgICAgICB0b3RhbCAgPSBmaW5hbC5nZXQoInRvdGFsX2Ftb3VudCIpCiAgICAgICAgICAgICAgICB0b3RhbF9zdHIgPSBmIiR7dG90YWw6LC4yZn0iIGlmIHRvdGFsIGVsc2UgIuKAlCIKICAgICAgICAgICAgICAgIGRvY190eXBlICA9IGRvYy5nZXQoImRvY190eXBlIiwgImludm9pY2UiKS50aXRsZSgpCgogICAgICAgICAgICAgICAgd2l0aCBzdC5jb250YWluZXIoYm9yZGVyPVRydWUpOgogICAgICAgICAgICAgICAgICAgIGMxLCBjMiwgYzMsIGM0ID0gc3QuY29sdW1ucyhbMi41LCAxLjUsIDEuMiwgMV0pCiAgICAgICAgICAgICAgICAgICAgd2l0aCBjMToKICAgICAgICAgICAgICAgICAgICAgICAgc3QubWFya2Rvd24oZiIqKntmbmFtZX0qKiIpCiAgICAgICAgICAgICAgICAgICAgICAgIHN0LmNhcHRpb24oZiJ7ZG9jWydkb2N1bWVudF9pZCddWzo4XX0gIMK3ICB7dmVuZG9yfSIpCiAgICAgICAgICAgICAgICAgICAgd2l0aCBjMjoKICAgICAgICAgICAgICAgICAgICAgICAgc3QuY2FwdGlvbigiVHlwZSAvIENvbmZpZGVuY2UiKQogICAgICAgICAgICAgICAgICAgICAgICBzdC5tYXJrZG93bihmIntkb2NfdHlwZX0gIMK3ICB7Y29uZioxMDA6LjBmfSUiKQogICAgICAgICAgICAgICAgICAgIHdpdGggYzM6CiAgICAgICAgICAgICAgICAgICAgICAgIHN0LmNhcHRpb24oIlRvdGFsIikKICAgICAgICAgICAgICAgICAgICAgICAgc3QubWFya2Rvd24odG90YWxfc3RyKQogICAgICAgICAgICAgICAgICAgIHdpdGggYzQ6CiAgICAgICAgICAgICAgICAgICAgICAgIHN0LmNhcHRpb24oIlN0YXR1cyIpCiAgICAgICAgICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKGIsIHVuc2FmZV9hbGxvd19odG1sPVRydWUpCgogICAgd2l0aCByaWdodDoKICAgICAgICBzdC5tYXJrZG93bigiIyMjIyBQaXBlbGluZSBIZWFsdGgiKQogICAgICAgIHdpdGggc3QuY29udGFpbmVyKGJvcmRlcj1UcnVlKToKICAgICAgICAgICAgZm9yIHN0YWdlIGluIFsiSW5nZXN0aW9uIiwgIkNsYXNzaWZpY2F0aW9uIiwgIkV4dHJhY3Rpb24iLCAiT0NSIEZhbGxiYWNrIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAiVmFsaWRhdGlvbiIsICJDb3JyZWN0aW9uIiwgIlN0b3JhZ2UiXToKICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKGYiKip7c3RhZ2V9KioiKQogICAgICAgICAgICAgICAgc3QuY2FwdGlvbigi4pyFIE9wZXJhdGlvbmFsIikKICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKCI8aHIgc3R5bGU9J21hcmdpbjo0cHggMDtib3JkZXItY29sb3I6I2YwZjRmOCcvPiIsIHVuc2FmZV9hbGxvd19odG1sPVRydWUpCgogICAgICAgIHN0Lm1hcmtkb3duKCIjIyMjIERvY3VtZW50IFR5cGVzIikKICAgICAgICB3aXRoIHN0LmNvbnRhaW5lcihib3JkZXI9VHJ1ZSk6CiAgICAgICAgICAgIGJ5X3R5cGUgPSBzdGF0cy5nZXQoImJ5X3R5cGUiLCB7fSkKICAgICAgICAgICAgdG90YWwgICA9IHN0YXRzWyJ0b3RhbCJdIG9yIDEKICAgICAgICAgICAgZm9yIGxhYmVsLCBrZXkgaW4gWygi8J+nviBJbnZvaWNlcyIsICJpbnZvaWNlIiksICgi8J+nviBSZWNlaXB0cyIsICJyZWNlaXB0IiksICgi8J+TiyBGb3JtcyIsICJmb3JtIildOgogICAgICAgICAgICAgICAgY250ICA9IGJ5X3R5cGUuZ2V0KGtleSwgMCkKICAgICAgICAgICAgICAgIHBjdCAgPSBjbnQgLyB0b3RhbAogICAgICAgICAgICAgICAgc3QubWFya2Rvd24oZiJ7bGFiZWx9ICDigJQge2NudH0gICh7cGN0KjEwMDouMGZ9JSkiKQogICAgICAgICAgICAgICAgc3QucHJvZ3Jlc3MocGN0KQo=",
    "/content/capstone/ui/pages/process.py": "...",
    "/content/capstone/ui/pages/review.py": "IiIiUmV2aWV3IFF1ZXVlIOKAlCBodW1hbiBhcHByb3ZhbCBmb3IgbG93LWNvbmZpZGVuY2UgLyBmYWlsZWQgZG9jdW1lbnRzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKCmltcG9ydCBzdHJlYW1saXQgYXMgc3QKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5nZXRjd2QoKSkKCmZyb20gdWkuc3R5bGVzIGltcG9ydCBiYWRnZSwgcGFnZV9oZWFkZXIKZnJvbSB1aS5jb21wb25lbnRzLmxhdGVuY3lfcGFuZWwgaW1wb3J0IHJlbmRlciBhcyByZW5kZXJfbGF0ZW5jeQoKCmRlZiBfbG9hZF9wZW5kaW5nKCkgLT4gbGlzdFtkaWN0XToKICAgIHRyeToKICAgICAgICBmcm9tIGRhdGFiYXNlLnN0b3JhZ2UgaW1wb3J0IGdldF9wZW5kaW5nX3JldmlldwogICAgICAgIHJldHVybiBnZXRfcGVuZGluZ19yZXZpZXcobGltaXQ9MTAwKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gW10KCgpkZWYgcmVuZGVyKCkgLT4gTm9uZToKICAgIHBhZ2VfaGVhZGVyKCLwn5GB77iPIiwgIlJldmlldyBRdWV1ZSIsICJEb2N1bWVudHMgZmxhZ2dlZCBmb3IgaHVtYW4gdmVyaWZpY2F0aW9uIikKCiAgICBwZW5kaW5nX3JlY29yZHMgPSBfbG9hZF9wZW5kaW5nKCkKCiAgICAjIOKUgOKUgCBTdGF0cyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIGMxLCBjMiwgYzMgPSBzdC5jb2x1bW5zKDMpCiAgICBjMS5tZXRyaWMoIlBlbmRpbmcgUmV2aWV3IiwgICAgICAgIGxlbihwZW5kaW5nX3JlY29yZHMpKQogICAgYzIubWV0cmljKCJBcHByb3ZlZCAodGhpcyBzZXNzaW9uKSIsIGxlbihzdC5zZXNzaW9uX3N0YXRlLmdldCgiYXBwcm92ZWRfaWRzIiwgc2V0KCkpKSkKICAgIGMzLm1ldHJpYygiUmVqZWN0ZWQgKHRoaXMgc2Vzc2lvbikiLCBsZW4oc3Quc2Vzc2lvbl9zdGF0ZS5nZXQoInJlamVjdGVkX2lkcyIsICBzZXQoKSkpKQoKICAgIGlmICJhcHByb3ZlZF9pZHMiIG5vdCBpbiBzdC5zZXNzaW9uX3N0YXRlOgogICAgICAgIHN0LnNlc3Npb25fc3RhdGUuYXBwcm92ZWRfaWRzID0gc2V0KCkKICAgIGlmICJyZWplY3RlZF9pZHMiIG5vdCBpbiBzdC5zZXNzaW9uX3N0YXRlOgogICAgICAgIHN0LnNlc3Npb25fc3RhdGUucmVqZWN0ZWRfaWRzID0gc2V0KCkKCiAgICAjIEZpbHRlciBvdXQgYWxyZWFkeSBhY3Rpb25lZAogICAgcGVuZGluZyA9IFsKICAgICAgICByIGZvciByIGluIHBlbmRpbmdfcmVjb3JkcwogICAgICAgIGlmIHJbImRvY3VtZW50X2lkIl0gbm90IGluIHN0LnNlc3Npb25fc3RhdGUuYXBwcm92ZWRfaWRzCiAgICAgICAgYW5kIHJbImRvY3VtZW50X2lkIl0gbm90IGluIHN0LnNlc3Npb25fc3RhdGUucmVqZWN0ZWRfaWRzCiAgICBdCgogICAgc3QubWFya2Rvd24oIjxici8+IiwgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSkKCiAgICBpZiBub3QgcGVuZGluZzoKICAgICAgICBzdC5zdWNjZXNzKCJSZXZpZXcgcXVldWUgaXMgZW1wdHkg4oCUIGFsbCBkb2N1bWVudHMgaGF2ZSBiZWVuIGFjdGlvbmVkLiIpCiAgICAgICAgcmV0dXJuCgogICAgZm9yIGRvYyBpbiBwZW5kaW5nOgogICAgICAgIGRvY19pZCAgID0gZG9jWyJkb2N1bWVudF9pZCJdCiAgICAgICAgZm5hbWUgICAgPSAoZG9jLmdldCgic291cmNlX3BhdGgiKSBvciAiIikuc3BsaXQoIi8iKVstMV0gb3IgZG9jX2lkWzoxMl0KICAgICAgICBjb25mICAgICA9IGRvYy5nZXQoImV4dHJhY3Rpb25fY29uZmlkZW5jZSIsIDAuMCkKICAgICAgICBkb2NfdHlwZSA9IChkb2MuZ2V0KCJkb2NfdHlwZSIpIG9yICJpbnZvaWNlIikudGl0bGUoKQogICAgICAgIG5vdGVzICAgID0ganNvbi5sb2Fkcyhkb2MuZ2V0KCJwcm9jZXNzaW5nX25vdGVzIikgb3IgIltdIikKICAgICAgICBmaW5hbCAgICA9IGpzb24ubG9hZHMoZG9jLmdldCgiZmluYWxfZGF0YSIpICAgICAgIG9yICJ7fSIpCgogICAgICAgIHJlYXNvbiA9IG5leHQoCiAgICAgICAgICAgIChuIGZvciBuIGluIHJldmVyc2VkKG5vdGVzKSBpZiAicmV2aWV3IiBpbiBuLmxvd2VyKCkgb3IgImNvbmZpZGVuY2UiIGluIG4ubG93ZXIoKSksCiAgICAgICAgICAgICJMb3cgY29uZmlkZW5jZSBvciB2YWxpZGF0aW9uIGZhaWxlZCIsCiAgICAgICAgKQoKICAgICAgICB3aXRoIHN0LmV4cGFuZGVyKAogICAgICAgICAgICBmIvCflI0gIHtmbmFtZX0gIMK3ICB7ZG9jX3R5cGV9ICDCtyAgQ29uZmlkZW5jZSB7Y29uZioxMDA6LjBmfSUgIMK3ICB7cmVhc29ufSIsCiAgICAgICAgICAgIGV4cGFuZGVkPUZhbHNlLAogICAgICAgICk6CiAgICAgICAgICAgIGZyb20gdWkuY29tcG9uZW50cy5maWVsZF9ncm91cHMgaW1wb3J0IGdyb3VwX2ZpZWxkcywgcHJldHR5X2xhYmVsCgogICAgICAgICAgICAjIEZpZWxkcyBsaXZlIHVuZGVyIGZpbmFsWyJmaWVsZHMiXSBpbiB0aGUgRG9jdW1lbnREYXRhIHNjaGVtYQogICAgICAgICAgICBmaWVsZHNfZGljdCA9IGZpbmFsLmdldCgiZmllbGRzIikgb3Ige30KICAgICAgICAgICAgbGluZV9pdGVtcyAgPSBmaW5hbC5nZXQoImxpbmVfaXRlbXMiKSBvciBbXQoKICAgICAgICAgICAgbGVmdCwgcmlnaHQgPSBzdC5jb2x1bW5zKFsyLCAxXSkKCiAgICAgICAgICAgICMg4pSA4pSAIExlZnQ6IGVkaXRhYmxlIGZpZWxkIHBhbmVsIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICAgICB3aXRoIGxlZnQ6CiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bigiKipSZXZpZXcgJiBFZGl0IEV4dHJhY3RlZCBGaWVsZHMqKiIpCiAgICAgICAgICAgICAgICBzdC5jYXB0aW9uKCJBbGwgZmllbGRzIGFyZSBwcmUtcG9wdWxhdGVkIGZyb20gdGhlIEFJIGV4dHJhY3Rpb24uICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIkNvcnJlY3QgYW55IGVycm9ycyBiZWZvcmUgYXBwcm92aW5nLiIpCgogICAgICAgICAgICAgICAgZWRpdGVkX2ZpZWxkczogZGljdCA9IHt9CgogICAgICAgICAgICAgICAgaWYgZmllbGRzX2RpY3Q6CiAgICAgICAgICAgICAgICAgICAgZ3JvdXBzID0gZ3JvdXBfZmllbGRzKGZpZWxkc19kaWN0KQogICAgICAgICAgICAgICAgICAgIGZvciBncm91cF9sYWJlbCwgZ2YgaW4gZ3JvdXBzLml0ZW1zKCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKGYiKip7Z3JvdXBfbGFiZWx9KioiKQogICAgICAgICAgICAgICAgICAgICAgICBjb2xzID0gc3QuY29sdW1ucygyKQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgaSwgKGZrZXksIGZ2YWwpIGluIGVudW1lcmF0ZShnZi5pdGVtcygpKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVkaXRlZF9maWVsZHNbZmtleV0gPSBjb2xzW2kgJSAyXS50ZXh0X2lucHV0KAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByZXR0eV9sYWJlbChma2V5KSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWx1ZT1zdHIoZnZhbCkgaWYgZnZhbCBpcyBub3QgTm9uZSBlbHNlICIiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGtleT1mInJ2X3tkb2NfaWR9X3tma2V5fSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIjxociBzdHlsZT0nbWFyZ2luOjZweCAwO2JvcmRlci1jb2xvcjojZTJlOGYwJy8+IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHVuc2FmZV9hbGxvd19odG1sPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgc3Qud2FybmluZygKICAgICAgICAgICAgICAgICAgICAgICAgIk5vIGZpZWxkcyB3ZXJlIGV4dHJhY3RlZC4gWW91IGNhbiBzdGlsbCBhcHByb3ZlIG9yIHJlamVjdCB0aGUgZG9jdW1lbnQuIgogICAgICAgICAgICAgICAgICAgICkKCiAgICAgICAgICAgICAgICAjIOKUgOKUgCBMaW5lIGl0ZW1zIChyZWFkLW9ubHkg4oCUIHNob3duIGZvciBjb250ZXh0KSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICAgICAgICAgIGlmIGxpbmVfaXRlbXM6CiAgICAgICAgICAgICAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKCIqKkxpbmUgSXRlbXMqKiAqKHJlYWQtb25seSkqIikKICAgICAgICAgICAgICAgICAgICBzdC5kYXRhZnJhbWUoCiAgICAgICAgICAgICAgICAgICAgICAgIHBkLkRhdGFGcmFtZShsaW5lX2l0ZW1zKSwKICAgICAgICAgICAgICAgICAgICAgICAgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICBoaWRlX2luZGV4PVRydWUsCiAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgIyDilIDilIAgUmlnaHQ6IHN1bW1hcnkgKyBhY3Rpb24gYnV0dG9ucyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICAgICAgd2l0aCByaWdodDoKICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKCIqKlN1bW1hcnkqKiIpCiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bihmIi0gKipEb2MgSUQ6KiogYHtkb2NfaWRbOjhdfWAiKQogICAgICAgICAgICAgICAgc3QubWFya2Rvd24oZiItICoqVHlwZToqKiB7ZG9jX3R5cGV9IikKICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICAgICAgICAgIGYiLSAqKkNvbmZpZGVuY2U6KiogIgogICAgICAgICAgICAgICAgICAgIGYiPHNwYW4gc3R5bGU9J2NvbG9yOnsnI2Q5NzcwNicgaWYgY29uZiA+PSAwLjUgZWxzZSAnI2RjMjYyNid9OyIKICAgICAgICAgICAgICAgICAgICBmImZvbnQtd2VpZ2h0OjcwMCc+e2NvbmYqMTAwOi4wZn0lPC9zcGFuPiIsCiAgICAgICAgICAgICAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKGYiLSAqKlJlYXNvbiBmbGFnZ2VkOioqIHtyZWFzb259IikKCiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bigiKipQcm9jZXNzaW5nIGxvZzoqKiIpCiAgICAgICAgICAgICAgICBmb3Igbm90ZSBpbiBub3Rlc1stNTpdOgogICAgICAgICAgICAgICAgICAgIHN0LmNhcHRpb24oZiLigKIge25vdGV9IikKCiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bigiPGJyLz4iLCB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlKQogICAgICAgICAgICAgICAgdGltaW5ncyA9IGZpbmFsLmdldCgidGltaW5ncyIpCiAgICAgICAgICAgICAgICByZW5kZXJfbGF0ZW5jeShub3RlcywgdGltaW5nc19kaWN0PXRpbWluZ3MpCgogICAgICAgICAgICAgICAgc3QubWFya2Rvd24oIjxici8+IiwgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSkKICAgICAgICAgICAgICAgIGNvbF9hLCBjb2xfciA9IHN0LmNvbHVtbnMoMikKCiAgICAgICAgICAgICAgICB3aXRoIGNvbF9hOgogICAgICAgICAgICAgICAgICAgIGlmIHN0LmJ1dHRvbigi4pyFIEFwcHJvdmUgJiBTYXZlIiwga2V5PWYiYXBwcm92ZV97ZG9jX2lkfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHVzZV9jb250YWluZXJfd2lkdGg9VHJ1ZSwgdHlwZT0icHJpbWFyeSIpOgogICAgICAgICAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBmcm9tIGRhdGFiYXNlLnN0b3JhZ2UgaW1wb3J0IGFwcHJvdmVfcmVjb3JkCgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBDb252ZXJ0IGVkaXRlZCB0ZXh0IGJhY2sgdG8gdHlwZWQgdmFsdWVzCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBfYW1vdW50X2tleXMgPSB7CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgayBmb3IgayBpbiBlZGl0ZWRfZmllbGRzCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgYW55KHcgaW4gayBmb3IgdyBpbgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoImFtb3VudCIsInRvdGFsIiwic3VidG90YWwiLCJ0YXgiLCJwcmljZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY29zdCIsImJhbGFuY2UiLCJkaXNjb3VudCIsImZlZSIsIm5ldCIsImdyb3NzIikpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBjbGVhbl9maWVsZHM6IGRpY3QgPSB7fQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGssIHYgaW4gZWRpdGVkX2ZpZWxkcy5pdGVtcygpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHYgPSB2LnN0cmlwKCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiB2ID09ICIiIG9yIHYubG93ZXIoKSA9PSAibm9uZSI6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNsZWFuX2ZpZWxkc1trXSA9IE5vbmUKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbGlmIGsgaW4gX2Ftb3VudF9rZXlzOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjbGVhbl9maWVsZHNba10gPSBmbG9hdCgKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2LnJlcGxhY2UoIiwiLCAiIikucmVwbGFjZSgiJCIsICIiKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAucmVwbGFjZSgi4oK5IiwgIiIpLnJlcGxhY2UoIuKCrCIsICIiKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNsZWFuX2ZpZWxkc1trXSA9IHYKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjbGVhbl9maWVsZHNba10gPSB2CgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBTYXZlIHRoZSBmdWxsIHVwZGF0ZWQgZmluYWxfZGF0YSAocHJlc2VydmVzIGxpbmVfaXRlbXMgZXRjLikKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHVwZGF0ZWRfZmluYWwgPSB7CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKipmaW5hbCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZmllbGRzIjogY2xlYW5fZmllbGRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgfQogICAgICAgICAgICAgICAgICAgICAgICAgICAgYXBwcm92ZV9yZWNvcmQoZG9jX2lkLCB1cGRhdGVkX2ZpbmFsKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3Quc2Vzc2lvbl9zdGF0ZS5hcHByb3ZlZF9pZHMuYWRkKGRvY19pZCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0LnRvYXN0KGYiRG9jdW1lbnQge2RvY19pZFs6OF19IGFwcHJvdmVkLiIsIGljb249IuKchSIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdC5yZXJ1bigpCiAgICAgICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0LmVycm9yKGYiQ291bGQgbm90IGFwcHJvdmU6IHtlfSIpCgogICAgICAgICAgICAgICAgd2l0aCBjb2xfcjoKICAgICAgICAgICAgICAgICAgICBpZiBzdC5idXR0b24oIuKdjCBSZWplY3QiLCBrZXk9ZiJyZWplY3Rfe2RvY19pZH0iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUpOgogICAgICAgICAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBmcm9tIGRhdGFiYXNlLnN0b3JhZ2UgaW1wb3J0IHJlamVjdF9yZWNvcmQKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlamVjdF9yZWNvcmQoZG9jX2lkKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3Quc2Vzc2lvbl9zdGF0ZS5yZWplY3RlZF9pZHMuYWRkKGRvY19pZCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0LnRvYXN0KGYiRG9jdW1lbnQge2RvY19pZFs6OF19IHJlamVjdGVkLiIsIGljb249IvCfl5HvuI8iKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3QucmVydW4oKQogICAgICAgICAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdC5lcnJvcihmIkNvdWxkIG5vdCByZWplY3Q6IHtlfSIpCg==",
    "/content/capstone/ui/pages/history.py": "IiIiSGlzdG9yeSBwYWdlIOKAlCBicm93c2UgYWxsIHByb2Nlc3NlZCByZWNvcmRzLCB2aWV3IGRldGFpbHMsIGZpbHRlciwgZXhwb3J0LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHN0cmVhbWxpdCBhcyBzdAoKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLmdldGN3ZCgpKQoKZnJvbSB1aS5zdHlsZXMgaW1wb3J0IGJhZGdlLCBwYWdlX2hlYWRlcgpmcm9tIHVpLmNvbXBvbmVudHMubGF0ZW5jeV9wYW5lbCBpbXBvcnQgcmVuZGVyIGFzIHJlbmRlcl9sYXRlbmN5CgoKZGVmIF9sb2FkX3JlY29yZHMoKSAtPiBsaXN0W2RpY3RdOgogICAgdHJ5OgogICAgICAgIGZyb20gZGF0YWJhc2Uuc3RvcmFnZSBpbXBvcnQgbGlzdF9yZWNvcmRzCiAgICAgICAgcmV0dXJuIGxpc3RfcmVjb3JkcyhsaW1pdD01MDApCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBbXQoKCmRlZiBfdG9fZGlzcGxheV9kZihyYXc6IGxpc3RbZGljdF0pIC0+IHBkLkRhdGFGcmFtZToKICAgIHJvd3MgPSBbXQogICAgZm9yIHIgaW4gcmF3OgogICAgICAgIGZpbmFsID0ganNvbi5sb2FkcyhyLmdldCgiZmluYWxfZGF0YSIpIG9yICJ7fSIpCiAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAiSUQiOiAgICAgICAgIHJbImRvY3VtZW50X2lkIl1bOjhdLAogICAgICAgICAgICAiRmlsZSI6ICAgICAgIChyLmdldCgic291cmNlX3BhdGgiKSBvciAiIikuc3BsaXQoIi8iKVstMV0gb3IgIuKAlCIsCiAgICAgICAgICAgICJUeXBlIjogICAgICAgKHIuZ2V0KCJkb2NfdHlwZSIpIG9yICJpbnZvaWNlIikudGl0bGUoKSwKICAgICAgICAgICAgIlZlbmRvciI6ICAgICBmaW5hbC5nZXQoInZlbmRvcl9uYW1lIikgb3IgIuKAlCIsCiAgICAgICAgICAgICJUb3RhbCI6ICAgICAgZmluYWwuZ2V0KCJ0b3RhbF9hbW91bnQiKSwKICAgICAgICAgICAgIlN0YXR1cyI6ICAgICByLmdldCgidmFsaWRhdGlvbl9zdGF0dXMiLCAidW5rbm93biIpLAogICAgICAgICAgICAiQ29uZmlkZW5jZSI6IGYieyhyLmdldCgnZXh0cmFjdGlvbl9jb25maWRlbmNlJykgb3IgMCkqMTAwOi4wZn0lIiwKICAgICAgICAgICAgIkRhdGUiOiAgICAgICAoci5nZXQoImNyZWF0ZWRfYXQiKSBvciAiIilbOjEwXSwKICAgICAgICB9KQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKSBpZiByb3dzIGVsc2UgcGQuRGF0YUZyYW1lKAogICAgICAgIGNvbHVtbnM9WyJJRCIsICJGaWxlIiwgIlR5cGUiLCAiVmVuZG9yIiwgIlRvdGFsIiwgIlN0YXR1cyIsICJDb25maWRlbmNlIiwgIkRhdGUiXQogICAgKQoKCmRlZiByZW5kZXIoKSAtPiBOb25lOgogICAgcGFnZV9oZWFkZXIoIvCfk4siLCAiSGlzdG9yeSIsICJBbGwgcHJvY2Vzc2VkIGRvY3VtZW50cyDigJQgY2xpY2sgYW55IHJvdyB0byB2aWV3IGZ1bGwgZGV0YWlscyIpCgogICAgcmF3ICAgICA9IF9sb2FkX3JlY29yZHMoKQogICAgZGYgICAgICA9IF90b19kaXNwbGF5X2RmKHJhdykKCiAgICBpZiBkZi5lbXB0eToKICAgICAgICBzdC5pbmZvKCJObyBkb2N1bWVudHMgcHJvY2Vzc2VkIHlldC4gR28gdG8gKipQcm9jZXNzIERvY3VtZW50KiogdG8gZ2V0IHN0YXJ0ZWQuIikKICAgICAgICByZXR1cm4KCiAgICAjIOKUgOKUgCBGaWx0ZXJzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgZjEsIGYyLCBmMyA9IHN0LmNvbHVtbnMoWzMsIDEuNSwgMS41XSkKICAgIHdpdGggZjE6CiAgICAgICAgc2VhcmNoID0gc3QudGV4dF9pbnB1dCgi8J+UjSBTZWFyY2giLCBwbGFjZWhvbGRlcj0iRmlsZSBuYW1lIG9yIHZlbmRvcuKApiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBsYWJlbF92aXNpYmlsaXR5PSJjb2xsYXBzZWQiKQogICAgd2l0aCBmMjoKICAgICAgICB0eXBlcyAgICA9IFsiQWxsIl0gKyBzb3J0ZWQoZGZbIlR5cGUiXS5kcm9wbmEoKS51bmlxdWUoKS50b2xpc3QoKSkKICAgICAgICB0X2ZpbHRlciA9IHN0LnNlbGVjdGJveCgiVHlwZSIsIHR5cGVzLCBsYWJlbF92aXNpYmlsaXR5PSJjb2xsYXBzZWQiKQogICAgd2l0aCBmMzoKICAgICAgICBzdGF0dXNlcyAgPSBbIkFsbCJdICsgc29ydGVkKGRmWyJTdGF0dXMiXS5kcm9wbmEoKS51bmlxdWUoKS50b2xpc3QoKSkKICAgICAgICBzX2ZpbHRlciAgPSBzdC5zZWxlY3Rib3goIlN0YXR1cyIsIHN0YXR1c2VzLCBsYWJlbF92aXNpYmlsaXR5PSJjb2xsYXBzZWQiKQoKICAgIGZpbHRlcmVkX2RmICA9IGRmLmNvcHkoKQogICAgZmlsdGVyZWRfcmF3ID0gbGlzdChyYXcpCgogICAgaWYgc2VhcmNoOgogICAgICAgIG1hc2sgPSAoCiAgICAgICAgICAgIGZpbHRlcmVkX2RmWyJGaWxlIl0uc3RyLmNvbnRhaW5zKHNlYXJjaCwgY2FzZT1GYWxzZSwgbmE9RmFsc2UpCiAgICAgICAgICAgIHwgZmlsdGVyZWRfZGZbIlZlbmRvciJdLnN0ci5jb250YWlucyhzZWFyY2gsIGNhc2U9RmFsc2UsIG5hPUZhbHNlKQogICAgICAgICkKICAgICAgICBmaWx0ZXJlZF9kZiAgPSBmaWx0ZXJlZF9kZlttYXNrXQogICAgICAgIGZpbHRlcmVkX3JhdyA9IFtyIGZvciByLCBrZWVwIGluIHppcChyYXcsIG1hc2spIGlmIGtlZXBdCgogICAgaWYgdF9maWx0ZXIgIT0gIkFsbCI6CiAgICAgICAgbWFzayA9IGZpbHRlcmVkX2RmWyJUeXBlIl0gPT0gdF9maWx0ZXIKICAgICAgICBmaWx0ZXJlZF9kZiAgPSBmaWx0ZXJlZF9kZlttYXNrXQogICAgICAgIGZpbHRlcmVkX3JhdyA9IFtyIGZvciByLCBrZWVwIGluIHppcChmaWx0ZXJlZF9yYXcsIG1hc2spIGlmIGtlZXBdCgogICAgaWYgc19maWx0ZXIgIT0gIkFsbCI6CiAgICAgICAgbWFzayA9IGZpbHRlcmVkX2RmWyJTdGF0dXMiXSA9PSBzX2ZpbHRlcgogICAgICAgIGZpbHRlcmVkX2RmICA9IGZpbHRlcmVkX2RmW21hc2tdCiAgICAgICAgZmlsdGVyZWRfcmF3ID0gW3IgZm9yIHIsIGtlZXAgaW4gemlwKGZpbHRlcmVkX3JhdywgbWFzaykgaWYga2VlcF0KCiAgICBmaWx0ZXJlZF9kZiA9IGZpbHRlcmVkX2RmLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCiAgICBzdC5jYXB0aW9uKGYiU2hvd2luZyB7bGVuKGZpbHRlcmVkX2RmKX0gb2Yge2xlbihkZil9IHJlY29yZHMgIMK3ICBzZWxlY3QgYSByb3cgdG8gdmlldyBkZXRhaWxzIikKCiAgICAjIOKUgOKUgCBUYWJsZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHNlbGVjdGVkID0gc3QuZGF0YWZyYW1lKAogICAgICAgIGZpbHRlcmVkX2RmLAogICAgICAgIHVzZV9jb250YWluZXJfd2lkdGg9VHJ1ZSwKICAgICAgICBoaWRlX2luZGV4PVRydWUsCiAgICAgICAgb25fc2VsZWN0PSJyZXJ1biIsCiAgICAgICAgc2VsZWN0aW9uX21vZGU9InNpbmdsZS1yb3ciLAogICAgICAgIGNvbHVtbl9jb25maWc9ewogICAgICAgICAgICAiVG90YWwiOiAgc3QuY29sdW1uX2NvbmZpZy5OdW1iZXJDb2x1bW4oIlRvdGFsIiwgZm9ybWF0PSIkJS4yZiIpLAogICAgICAgICAgICAiU3RhdHVzIjogc3QuY29sdW1uX2NvbmZpZy5UZXh0Q29sdW1uKCJTdGF0dXMiKSwKICAgICAgICB9LAogICAgKQoKICAgICMg4pSA4pSAIEV4cG9ydCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIGVjMSwgZWMyLCBfID0gc3QuY29sdW1ucyhbMSwgMSwgM10pCiAgICB3aXRoIGVjMToKICAgICAgICBzdC5kb3dubG9hZF9idXR0b24oIuKsh++4jyBFeHBvcnQgQ1NWIiwKICAgICAgICAgICAgZGF0YT1maWx0ZXJlZF9kZi50b19jc3YoaW5kZXg9RmFsc2UpLAogICAgICAgICAgICBmaWxlX25hbWU9ImRvY19hZ2VudF9oaXN0b3J5LmNzdiIsIG1pbWU9InRleHQvY3N2IiwKICAgICAgICAgICAgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLAogICAgICAgICAgICBrZXk9Imhpc3RfZXhwb3J0X2NzdiIpCiAgICB3aXRoIGVjMjoKICAgICAgICBzdC5kb3dubG9hZF9idXR0b24oIuKsh++4jyBFeHBvcnQgSlNPTiIsCiAgICAgICAgICAgIGRhdGE9ZmlsdGVyZWRfZGYudG9fanNvbihvcmllbnQ9InJlY29yZHMiLCBpbmRlbnQ9MiksCiAgICAgICAgICAgIGZpbGVfbmFtZT0iZG9jX2FnZW50X2hpc3RvcnkuanNvbiIsIG1pbWU9ImFwcGxpY2F0aW9uL2pzb24iLAogICAgICAgICAgICB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUsCiAgICAgICAgICAgIGtleT0iaGlzdF9leHBvcnRfanNvbiIpCgogICAgIyDilIDilIAgRGV0YWlsIHBhbmVsIChzaG93biB3aGVuIGEgcm93IGlzIHNlbGVjdGVkKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHJvd3Nfc2VsID0gc2VsZWN0ZWQuZ2V0KCJzZWxlY3Rpb24iLCB7fSkuZ2V0KCJyb3dzIiwgW10pCiAgICBpZiBub3Qgcm93c19zZWw6CiAgICAgICAgcmV0dXJuCgogICAgaWR4ICAgICA9IHJvd3Nfc2VsWzBdCiAgICBpZiBpZHggPj0gbGVuKGZpbHRlcmVkX3Jhdyk6CiAgICAgICAgcmV0dXJuCgogICAgcmVjb3JkICA9IGZpbHRlcmVkX3Jhd1tpZHhdCiAgICBmaW5hbCAgID0ganNvbi5sb2FkcyhyZWNvcmQuZ2V0KCJmaW5hbF9kYXRhIikgIG9yICJ7fSIpCiAgICB2bG1fcmF3ID0ganNvbi5sb2FkcyhyZWNvcmQuZ2V0KCJ2bG1fZXh0cmFjdGlvbiIpIG9yICJ7fSIpCiAgICBub3RlcyAgID0ganNvbi5sb2FkcyhyZWNvcmQuZ2V0KCJwcm9jZXNzaW5nX25vdGVzIikgb3IgIltdIikKICAgIGNvcnJlY3Rpb25zID0ganNvbi5sb2FkcyhyZWNvcmQuZ2V0KCJjb3JyZWN0aW9ucyIpIG9yICJbXSIpCiAgICBzdGF0dXMgID0gcmVjb3JkLmdldCgidmFsaWRhdGlvbl9zdGF0dXMiLCAidW5rbm93biIpCgogICAgc3QubWFya2Rvd24oIi0tLSIpCiAgICBzdC5tYXJrZG93bihmIiMjIyDwn5OEIERvY3VtZW50IERldGFpbHMg4oCUIGB7cmVjb3JkWydkb2N1bWVudF9pZCddWzo4XX1gIikKCiAgICBjb2xfc3RhdHVzLCBjb2xfY29uZiwgY29sX3R5cGUsIGNvbF9kYXRlID0gc3QuY29sdW1ucyg0KQogICAgY29sX3N0YXR1cy5tZXRyaWMoIlN0YXR1cyIsICAgICBzdGF0dXMucmVwbGFjZSgiXyIsICIgIikudGl0bGUoKSkKICAgIGNvbF9jb25mLm1ldHJpYygiQ29uZmlkZW5jZSIsICAgZiJ7KHJlY29yZC5nZXQoJ2V4dHJhY3Rpb25fY29uZmlkZW5jZScpIG9yIDApKjEwMDouMGZ9JSIpCiAgICBjb2xfdHlwZS5tZXRyaWMoIlR5cGUiLCAgICAgICAgIChyZWNvcmQuZ2V0KCJkb2NfdHlwZSIpIG9yICJpbnZvaWNlIikudGl0bGUoKSkKICAgIGNvbF9kYXRlLm1ldHJpYygiUHJvY2Vzc2VkIiwgICAgKHJlY29yZC5nZXQoImNyZWF0ZWRfYXQiKSBvciAiIilbOjEwXSkKCiAgICB0YWJfZmllbGRzLCB0YWJfaXRlbXMsIHRhYl9vY3IsIHRhYl9sYXRlbmN5LCB0YWJfYXVkaXQsIHRhYl9leHBvcnQgPSBzdC50YWJzKFsKICAgICAgICAi8J+TiyBFeHRyYWN0ZWQgRmllbGRzIiwgIvCfl5LvuI8gTGluZSBJdGVtcyIsICLwn5OEIFJhdyBPQ1IiLAogICAgICAgICLij7EgTGF0ZW5jeSIsICLwn5SNIEF1ZGl0IFRyYWlsIiwgIuKsh++4jyBFeHBvcnQiCiAgICBdKQoKICAgIHdpdGggdGFiX2ZpZWxkczoKICAgICAgICBmcm9tIHVpLmNvbXBvbmVudHMuZmllbGRfZ3JvdXBzIGltcG9ydCBmb3JtYXRfdmFsdWUsIGdyb3VwX2ZpZWxkcywgcHJldHR5X2xhYmVsCgogICAgICAgICMgRmllbGRzIGFyZSBzdG9yZWQgdW5kZXIgZmluYWxbImZpZWxkcyJdLCBub3QgYXQgdGhlIHRvcCBsZXZlbAogICAgICAgIGZpZWxkc19kaWN0ID0gZmluYWwuZ2V0KCJmaWVsZHMiKSBvciB7fQogICAgICAgIGN1cnJlbmN5ICAgID0gc3RyKGZpZWxkc19kaWN0LmdldCgiY3VycmVuY3kiKSBvciAiIikKCiAgICAgICAgaWYgbm90IGZpZWxkc19kaWN0OgogICAgICAgICAgICBzdC5pbmZvKCJObyBmaWVsZHMgd2VyZSBleHRyYWN0ZWQgZm9yIHRoaXMgZG9jdW1lbnQuIikKICAgICAgICBlbHNlOgogICAgICAgICAgICBncm91cHMgPSBncm91cF9maWVsZHMoZmllbGRzX2RpY3QpCiAgICAgICAgICAgIGZvciBncm91cF9sYWJlbCwgZ3JvdXBfZmllbGRzX2RpY3QgaW4gZ3JvdXBzLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bihmIioqe2dyb3VwX2xhYmVsfSoqIikKICAgICAgICAgICAgICAgIGNvbHMgPSBzdC5jb2x1bW5zKDIpCiAgICAgICAgICAgICAgICBmb3IgaSwgKGZrZXksIGZ2YWwpIGluIGVudW1lcmF0ZShncm91cF9maWVsZHNfZGljdC5pdGVtcygpKToKICAgICAgICAgICAgICAgICAgICBjb2xzW2kgJSAyXS50ZXh0X2lucHV0KAogICAgICAgICAgICAgICAgICAgICAgICBwcmV0dHlfbGFiZWwoZmtleSksCiAgICAgICAgICAgICAgICAgICAgICAgIHZhbHVlPWZvcm1hdF92YWx1ZShma2V5LCBmdmFsLCBjdXJyZW5jeSksCiAgICAgICAgICAgICAgICAgICAgICAgIGRpc2FibGVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgIGtleT1mImhpc3Rfe3JlY29yZFsnZG9jdW1lbnRfaWQnXX1fe2ZrZXl9IiwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bigKICAgICAgICAgICAgICAgICAgICAiPGhyIHN0eWxlPSdtYXJnaW46OHB4IDA7Ym9yZGVyLWNvbG9yOiNlMmU4ZjAnLz4iLAogICAgICAgICAgICAgICAgICAgIHVuc2FmZV9hbGxvd19odG1sPVRydWUsCiAgICAgICAgICAgICAgICApCgogICAgICAgIGlmIGNvcnJlY3Rpb25zOgogICAgICAgICAgICBzdC5tYXJrZG93bigiKipBdXRvLWNvcnJlY3Rpb25zIGFwcGxpZWQ6KioiKQogICAgICAgICAgICBmb3IgYyBpbiBjb3JyZWN0aW9uczoKICAgICAgICAgICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICAgICAgICAgIGYiLSBge2MuZ2V0KCdmaWVsZCcpfWA6ICIKICAgICAgICAgICAgICAgICAgICBmImB7Yy5nZXQoJ29yaWdpbmFsX3ZhbHVlJyl9YCDihpIgYHtjLmdldCgnY29ycmVjdGVkX3ZhbHVlJyl9YCIKICAgICAgICAgICAgICAgICkKCiAgICB3aXRoIHRhYl9pdGVtczoKICAgICAgICBpdGVtcyA9IGZpbmFsLmdldCgibGluZV9pdGVtcyIpIG9yIHZsbV9yYXcuZ2V0KCJsaW5lX2l0ZW1zIikgb3IgW10KICAgICAgICBpZiBpdGVtczoKICAgICAgICAgICAgc3QuZGF0YWZyYW1lKHBkLkRhdGFGcmFtZShpdGVtcyksIHVzZV9jb250YWluZXJfd2lkdGg9VHJ1ZSwgaGlkZV9pbmRleD1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHN0LmluZm8oIk5vIGxpbmUgaXRlbXMgaW4gdGhpcyBkb2N1bWVudC4iKQoKICAgIHdpdGggdGFiX29jcjoKICAgICAgICBvY3JfdGV4dCA9IHJlY29yZC5nZXQoInJhd19vY3JfdGV4dCIpIG9yICIiCiAgICAgICAgaWYgb2NyX3RleHQuc3RyaXAoKToKICAgICAgICAgICAgc3QudGV4dF9hcmVhKCJSYXcgT0NSIFRleHQiLCB2YWx1ZT1vY3JfdGV4dCwgaGVpZ2h0PTIyMCwgZGlzYWJsZWQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgIGtleT1mIm9jcl97cmVjb3JkWydkb2N1bWVudF9pZCddfSIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc3QuaW5mbygiT0NSIHdhcyBub3QgdXNlZCBmb3IgdGhpcyBkb2N1bWVudCAoaGlnaCBjb25maWRlbmNlIGV4dHJhY3Rpb24pLiIpCgogICAgd2l0aCB0YWJfbGF0ZW5jeToKICAgICAgICB0aW1pbmdzID0gZmluYWwuZ2V0KCJ0aW1pbmdzIikgb3Ige30KICAgICAgICByZW5kZXJfbGF0ZW5jeShub3RlcywgdGltaW5nc19kaWN0PXtrOiB2IGZvciBrLCB2IGluIHRpbWluZ3MuaXRlbXMoKSBpZiBrICE9ICJ0b3RhbCJ9KQoKICAgIHdpdGggdGFiX2F1ZGl0OgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBkYXRhYmFzZS5zdG9yYWdlIGltcG9ydCBnZXRfYXVkaXRfdHJhaWwKICAgICAgICAgICAgZXZlbnRzID0gZ2V0X2F1ZGl0X3RyYWlsKHJlY29yZFsiZG9jdW1lbnRfaWQiXSkKICAgICAgICAgICAgaWYgZXZlbnRzOgogICAgICAgICAgICAgICAgYXVkaXRfZGYgPSBwZC5EYXRhRnJhbWUoW3sKICAgICAgICAgICAgICAgICAgICAiVGltZSI6ICAgIGVbInRpbWVzdGFtcCJdLAogICAgICAgICAgICAgICAgICAgICJFdmVudCI6ICAgZVsiZXZlbnQiXSwKICAgICAgICAgICAgICAgICAgICAiRGV0YWlscyI6IGVbImRldGFpbHMiXSwKICAgICAgICAgICAgICAgIH0gZm9yIGUgaW4gZXZlbnRzXSkKICAgICAgICAgICAgICAgIHN0LmRhdGFmcmFtZShhdWRpdF9kZiwgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLCBoaWRlX2luZGV4PVRydWUpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBzdC5pbmZvKCJObyBhdWRpdCBldmVudHMgZm91bmQuIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHN0Lndhcm5pbmcoZiJDb3VsZCBub3QgbG9hZCBhdWRpdCB0cmFpbDoge2V9IikKCiAgICB3aXRoIHRhYl9leHBvcnQ6CiAgICAgICAgZG9jX2pzb24gPSBqc29uLmR1bXBzKGZpbmFsLCBpbmRlbnQ9MiwgZGVmYXVsdD1zdHIpCiAgICAgICAgYzEsIGMyID0gc3QuY29sdW1ucygyKQogICAgICAgIHdpdGggYzE6CiAgICAgICAgICAgIHN0LmRvd25sb2FkX2J1dHRvbigKICAgICAgICAgICAgICAgICLirIfvuI8gRG93bmxvYWQgSlNPTiIsCiAgICAgICAgICAgICAgICBkYXRhPWRvY19qc29uLAogICAgICAgICAgICAgICAgZmlsZV9uYW1lPWYiZG9jX3tyZWNvcmRbJ2RvY3VtZW50X2lkJ11bOjhdfS5qc29uIiwKICAgICAgICAgICAgICAgIG1pbWU9ImFwcGxpY2F0aW9uL2pzb24iLAogICAgICAgICAgICAgICAgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLAogICAgICAgICAgICAgICAga2V5PWYiZGxfanNvbl97cmVjb3JkWydkb2N1bWVudF9pZCddfSIsCiAgICAgICAgICAgICkKICAgICAgICB3aXRoIGMyOgogICAgICAgICAgICBzdC5kb3dubG9hZF9idXR0b24oCiAgICAgICAgICAgICAgICAi4qyH77iPIERvd25sb2FkIENTViIsCiAgICAgICAgICAgICAgICBkYXRhPXBkLkRhdGFGcmFtZShbe2s6IHYgZm9yIGssIHYgaW4gZmluYWwuaXRlbXMoKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBrICE9ICJsaW5lX2l0ZW1zIn1dKS50b19jc3YoaW5kZXg9RmFsc2UpLAogICAgICAgICAgICAgICAgZmlsZV9uYW1lPWYiZG9jX3tyZWNvcmRbJ2RvY3VtZW50X2lkJ11bOjhdfS5jc3YiLAogICAgICAgICAgICAgICAgbWltZT0idGV4dC9jc3YiLAogICAgICAgICAgICAgICAgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLAogICAgICAgICAgICAgICAga2V5PWYiZGxfY3N2X3tyZWNvcmRbJ2RvY3VtZW50X2lkJ119IiwKICAgICAgICAgICAgKQo=",
    "/content/capstone/ui/pages/techstack.py": "...",
    "/content/capstone/ui/pages/settings.py": "...",
    "/content/capstone/ui/components/__init__.py": "",
    "/content/capstone/ui/components/sidebar.py": "aW1wb3J0IHN0cmVhbWxpdCBhcyBzdApmcm9tIHVpLmF1dGggaW1wb3J0IGxvZ291dAoKX05BViA9IFsKICAgICgi8J+TiiIsICJEYXNoYm9hcmQiLCAgICAgICAgKSwKICAgICgi8J+UhCIsICJQcm9jZXNzIERvY3VtZW50IiwgKSwKICAgICgi8J+Rge+4jyIsICJSZXZpZXcgUXVldWUiLCAgICAgKSwKICAgICgi8J+TiyIsICJIaXN0b3J5IiwgICAgICAgICAgKSwKICAgICgi8J+boO+4jyIsICJUZWNoIFN0YWNrIiwgICAgICAgKSwKICAgICgi4pqZ77iPIiwgIlNldHRpbmdzIiwgICAgICAgICApLApdCgojIElubGluZS1zdHlsZSBjb25zdGFudHMgc28gdmlzaWJpbGl0eSBuZXZlciBkZXBlbmRzIG9uIENTUyBpbmhlcml0YW5jZQpfU0lERUJBUl9CRyAgICAgID0gIiMxYTI3NDQiCl9URVhUX0xJR0hUICAgICAgPSAiI2U4ZWRmNSIKX1RFWFRfRElNICAgICAgICA9ICJyZ2JhKDIzMiwyMzcsMjQ1LDAuNjUpIgpfQlROX0RFRkFVTFQgICAgID0gInJnYmEoMjU1LDI1NSwyNTUsMC4wNykiCl9CVE5fQUNUSVZFICAgICAgPSAicmdiYSgyNTUsMjU1LDI1NSwwLjE4KSIKX0JUTl9BQ1RJVkVfQk9SREVSID0gInJnYmEoMjU1LDI1NSwyNTUsMC4zNSkiCl9CVE5fSE9WRVIgICAgICAgPSAicmdiYSgyNTUsMjU1LDI1NSwwLjEyKSIKX0RJVklERVIgICAgICAgICA9ICJyZ2JhKDI1NSwyNTUsMjU1LDAuMTIpIgoKCmRlZiBfbmF2X2J1dHRvbihpY29uOiBzdHIsIGxhYmVsOiBzdHIsIGlzX2FjdGl2ZTogYm9vbCkgLT4gYm9vbDoKICAgIGJnICAgICA9IF9CVE5fQUNUSVZFICBpZiBpc19hY3RpdmUgZWxzZSBfQlROX0RFRkFVTFQKICAgIGJvcmRlciA9IGYiMXB4IHNvbGlkIHtfQlROX0FDVElWRV9CT1JERVJ9IiBpZiBpc19hY3RpdmUgZWxzZSAiMXB4IHNvbGlkIHJnYmEoMjU1LDI1NSwyNTUsMC4wOCkiCiAgICBmdyAgICAgPSAiNzAwIiBpZiBpc19hY3RpdmUgZWxzZSAiNTAwIgoKICAgIHN0Lm1hcmtkb3duKAogICAgICAgIGYiIiI8ZGl2IHN0eWxlPSIKICAgICAgICAgICAgYmFja2dyb3VuZDp7Ymd9OwogICAgICAgICAgICBib3JkZXI6e2JvcmRlcn07CiAgICAgICAgICAgIGJvcmRlci1yYWRpdXM6OHB4OwogICAgICAgICAgICBwYWRkaW5nOjlweCAxNHB4OwogICAgICAgICAgICBtYXJnaW46M3B4IDA7CiAgICAgICAgICAgIGN1cnNvcjpwb2ludGVyOwogICAgICAgICAgICBkaXNwbGF5OmZsZXg7CiAgICAgICAgICAgIGFsaWduLWl0ZW1zOmNlbnRlcjsKICAgICAgICAgICAgZ2FwOjEwcHg7CiAgICAgICAgICAgIGZvbnQtc2l6ZTouOTJyZW07CiAgICAgICAgICAgIGZvbnQtd2VpZ2h0Ontmd307CiAgICAgICAgICAgIGNvbG9yOntfVEVYVF9MSUdIVH07CiAgICAgICAgICAgIGxpbmUtaGVpZ2h0OjEuMzsKICAgICAgICAiPntpY29ufSZuYnNwOyZuYnNwO3tsYWJlbH08L2Rpdj4iIiIsCiAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICkKICAgIHJldHVybiBzdC5idXR0b24oCiAgICAgICAgZiJ7aWNvbn0gIHtsYWJlbH0iLAogICAgICAgIGtleT1mIm5hdl97bGFiZWx9IiwKICAgICAgICB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUsCiAgICAgICAgaGVscD1mIkdvIHRvIHtsYWJlbH0iLAogICAgKQoKCmRlZiByZW5kZXJfc2lkZWJhcigpIC0+IHN0cjoKICAgIGlmICJjdXJyZW50X3BhZ2UiIG5vdCBpbiBzdC5zZXNzaW9uX3N0YXRlOgogICAgICAgIHN0LnNlc3Npb25fc3RhdGUuY3VycmVudF9wYWdlID0gIkRhc2hib2FyZCIKCiAgICB3aXRoIHN0LnNpZGViYXI6CiAgICAgICAgIyDilIDilIAgTG9nbyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBzdC5tYXJrZG93bigKICAgICAgICAgICAgZiIiIjxkaXYgc3R5bGU9IgogICAgICAgICAgICAgICAgcGFkZGluZzoxNnB4IDRweCA0cHg7CiAgICAgICAgICAgICAgICBjb2xvcjp7X1RFWFRfTElHSFR9OwogICAgICAgICAgICAgICAgZm9udC1zaXplOjEuNHJlbTsKICAgICAgICAgICAgICAgIGZvbnQtd2VpZ2h0OjgwMDsKICAgICAgICAgICAgICAgIGxldHRlci1zcGFjaW5nOi0wLjVweDsKICAgICAgICAgICAgICAgIGxpbmUtaGVpZ2h0OjEuMjsKICAgICAgICAgICAgIj7wn6e+IERvYyBBZ2VudDwvZGl2PgogICAgICAgICAgICA8ZGl2IHN0eWxlPSIKICAgICAgICAgICAgICAgIGNvbG9yOntfVEVYVF9ESU19OwogICAgICAgICAgICAgICAgZm9udC1zaXplOi43NXJlbTsKICAgICAgICAgICAgICAgIHBhZGRpbmc6MCA0cHggMTRweDsKICAgICAgICAgICAgICAgIGJvcmRlci1ib3R0b206MXB4IHNvbGlkIHtfRElWSURFUn07CiAgICAgICAgICAgICAgICBtYXJnaW4tYm90dG9tOjEycHg7CiAgICAgICAgICAgICI+SW52b2ljZSBJbnRlbGxpZ2VuY2UgUGxhdGZvcm08L2Rpdj4iIiIsCiAgICAgICAgICAgIHVuc2FmZV9hbGxvd19odG1sPVRydWUsCiAgICAgICAgKQoKICAgICAgICAjIOKUgOKUgCBVc2VyIGluZm8g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgdXNlcm5hbWUgPSBzdC5zZXNzaW9uX3N0YXRlLmdldCgidXNlcm5hbWUiLCAidXNlciIpCiAgICAgICAgc3QubWFya2Rvd24oCiAgICAgICAgICAgIGYiIiI8ZGl2IHN0eWxlPSIKICAgICAgICAgICAgICAgIGNvbG9yOntfVEVYVF9ESU19OwogICAgICAgICAgICAgICAgZm9udC1zaXplOi44cmVtOwogICAgICAgICAgICAgICAgcGFkZGluZzowIDRweCAxNHB4OwogICAgICAgICAgICAgICAgbWFyZ2luLWJvdHRvbTo4cHg7CiAgICAgICAgICAgICI+8J+RpCB7dXNlcm5hbWV9PC9kaXY+IiIiLAogICAgICAgICAgICB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlLAogICAgICAgICkKCiAgICAgICAgIyDilIDilIAgTmF2IGxhYmVsIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICBmIiIiPGRpdiBzdHlsZT0iCiAgICAgICAgICAgICAgICBjb2xvcjp7X1RFWFRfRElNfTsKICAgICAgICAgICAgICAgIGZvbnQtc2l6ZTouNjVyZW07CiAgICAgICAgICAgICAgICB0ZXh0LXRyYW5zZm9ybTp1cHBlcmNhc2U7CiAgICAgICAgICAgICAgICBsZXR0ZXItc3BhY2luZzoxLjJweDsKICAgICAgICAgICAgICAgIHBhZGRpbmc6MCA0cHggNnB4OwogICAgICAgICAgICAiPk5hdmlnYXRpb248L2Rpdj4iIiIsCiAgICAgICAgICAgIHVuc2FmZV9hbGxvd19odG1sPVRydWUsCiAgICAgICAgKQoKICAgICAgICAjIOKUgOKUgCBOYXYgYnV0dG9ucyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBmb3IgaWNvbiwgbGFiZWwgaW4gX05BVjoKICAgICAgICAgICAgaXNfYWN0aXZlID0gc3Quc2Vzc2lvbl9zdGF0ZS5jdXJyZW50X3BhZ2UgPT0gbGFiZWwKICAgICAgICAgICAgaWYgc3QuYnV0dG9uKAogICAgICAgICAgICAgICAgZiJ7aWNvbn0gIHtsYWJlbH0iLAogICAgICAgICAgICAgICAga2V5PWYibmF2X3tsYWJlbH0iLAogICAgICAgICAgICAgICAgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLAogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgc3Quc2Vzc2lvbl9zdGF0ZS5jdXJyZW50X3BhZ2UgPSBsYWJlbAogICAgICAgICAgICAgICAgc3QucmVydW4oKQoKICAgICAgICAjIOKUgOKUgCBEaXZpZGVyICsgTG9nb3V0IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICBmIjxociBzdHlsZT0nYm9yZGVyOm5vbmU7Ym9yZGVyLXRvcDoxcHggc29saWQge19ESVZJREVSfTttYXJnaW46MTZweCAwJy8+IiwKICAgICAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICAgICApCiAgICAgICAgaWYgc3QuYnV0dG9uKCLwn5qqICBMb2dvdXQiLCB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUsIGtleT0ibmF2X2xvZ291dCIpOgogICAgICAgICAgICBsb2dvdXQoKQogICAgICAgICAgICBzdC5yZXJ1bigpCgoKICAgIHJldHVybiBzdC5zZXNzaW9uX3N0YXRlLmN1cnJlbnRfcGFnZQo=",
    "/content/capstone/ui/components/metrics.py": "aW1wb3J0IHN0cmVhbWxpdCBhcyBzdAoKCmRlZiBtZXRyaWNfY2FyZChsYWJlbDogc3RyLCB2YWx1ZTogc3RyLCBkZWx0YTogc3RyID0gIiIsIGNvbG9yOiBzdHIgPSAiYmx1ZSIpIC0+IHN0cjoKICAgIGRlbHRhX2h0bWwgPSAiIgogICAgaWYgZGVsdGE6CiAgICAgICAgY2xzID0gImRlbHRhLXVwIiBpZiBkZWx0YS5zdGFydHN3aXRoKCIrIikgZWxzZSAiZGVsdGEtZG93biIKICAgICAgICBkZWx0YV9odG1sID0gZic8ZGl2IGNsYXNzPSJtZXRyaWMtZGVsdGEge2Nsc30iPntkZWx0YX08L2Rpdj4nCiAgICByZXR1cm4gZiIiIgogICAgPGRpdiBjbGFzcz0ibWV0cmljLWNhcmQge2NvbG9yfSI+CiAgICAgICAgPGRpdiBjbGFzcz0ibWV0cmljLWxhYmVsIj57bGFiZWx9PC9kaXY+CiAgICAgICAgPGRpdiBjbGFzcz0ibWV0cmljLXZhbHVlIj57dmFsdWV9PC9kaXY+CiAgICAgICAge2RlbHRhX2h0bWx9CiAgICA8L2Rpdj4KICAgICIiIgoKCmRlZiByZW5kZXJfa3BpX3Jvdyh0b3RhbDogaW50LCBzdWNjZXNzX3JhdGU6IGZsb2F0LCBwZW5kaW5nOiBpbnQsIHRvZGF5OiBpbnQpIC0+IE5vbmU6CiAgICBjb2xzID0gc3QuY29sdW1ucyg0KQogICAgY2FyZHMgPSBbCiAgICAgICAgKGNvbHNbMF0sIG1ldHJpY19jYXJkKCJUb3RhbCBQcm9jZXNzZWQiLCBzdHIodG90YWwpLCBjb2xvcj0iYmx1ZSIpKSwKICAgICAgICAoY29sc1sxXSwgbWV0cmljX2NhcmQoIlN1Y2Nlc3MgUmF0ZSIsIGYie3N1Y2Nlc3NfcmF0ZTouMWZ9JSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICIrMi4zJSB2cyBsYXN0IHdlZWsiIGlmIHN1Y2Nlc3NfcmF0ZSA+IDAgZWxzZSAiIiwgImdyZWVuIikpLAogICAgICAgIChjb2xzWzJdLCBtZXRyaWNfY2FyZCgiUGVuZGluZyBSZXZpZXciLCBzdHIocGVuZGluZyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYieyfimqDvuI8gbmVlZHMgYXR0ZW50aW9uJyBpZiBwZW5kaW5nID4gNSBlbHNlICcnfSIsICJhbWJlciIpKSwKICAgICAgICAoY29sc1szXSwgbWV0cmljX2NhcmQoIlRvZGF5J3MgVm9sdW1lIiwgc3RyKHRvZGF5KSwgY29sb3I9ImJsdWUiKSksCiAgICBdCiAgICBmb3IgY29sLCBodG1sIGluIGNhcmRzOgogICAgICAgIHdpdGggY29sOgogICAgICAgICAgICBzdC5tYXJrZG93bihodG1sLCB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlKQo=",
    "/content/capstone/ui/components/pipeline_status.py": "IiIiUGlwZWxpbmUgc3RlcCBpbmRpY2F0b3Ig4oCUIGRyaXZlbiBieSByZWFsIExhbmdHcmFwaCBub2RlIGV2ZW50cy4iIiIKCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAppbXBvcnQgc3RyZWFtbGl0IGFzIHN0CgojIE1hcHMgTGFuZ0dyYXBoIG5vZGUgbmFtZXMg4oaSIChkaXNwbGF5IGxhYmVsLCBvcmRlciBpbmRleCkKIyBOb2RlcyB0aGF0IHNoYXJlIGEgZGlzcGxheSBsYWJlbCAoZS5nLiBjb3JyZWN0IGxvb3BzIGJhY2sgdGhyb3VnaCB2YWxpZGF0ZSkKIyBhcmUgY29sbGFwc2VkIGludG8gb25lIHN0YWdlIGZvciBkaXNwbGF5IHB1cnBvc2VzLgpfTk9ERV9TVEFHRV9NQVAgPSB7CiAgICAiaW5nZXN0IjogICAgICAgKCJJbmdlc3QiLCAgICAwKSwKICAgICJleHRyYWN0IjogICAgICAoIkV4dHJhY3QiLCAgIDEpLAogICAgIm9jcl9mYWxsYmFjayI6ICgiT0NSIENoZWNrIiwgMiksCiAgICAidmFsaWRhdGUiOiAgICAgKCJWYWxpZGF0ZSIsICAzKSwKICAgICJjb3JyZWN0IjogICAgICAoIkNvcnJlY3QiLCAgIDQpLAogICAgInN0b3JlIjogICAgICAgICgiU3RvcmUiLCAgICAgNSksCiAgICAicmV2aWV3X3F1ZXVlIjogKCJSZXZpZXciLCAgICA1KSwKfQoKX0FMTF9TVEFHRVMgPSBbCiAgICAoIvCfk6UiLCAiSW5nZXN0IiksCiAgICAoIvCflI0iLCAiRXh0cmFjdCIpLAogICAgKCLwn5OEIiwgIk9DUiBDaGVjayIpLAogICAgKCLinIUiLCAiVmFsaWRhdGUiKSwKICAgICgi8J+UhCIsICJDb3JyZWN0IiksCiAgICAoIvCfkr4iLCAiU3RvcmUiKSwKXQoKCmRlZiBfc3RhZ2VfaW5kZXgobmFtZTogc3RyKSAtPiBpbnQ6CiAgICByZXR1cm4gbmV4dCgoaSBmb3IgaSwgKF8sIG4pIGluIGVudW1lcmF0ZShfQUxMX1NUQUdFUykgaWYgbiA9PSBuYW1lKSwgLTEpCgoKZGVmIHJlbmRlcl9wcm9ncmVzcygKICAgIGNvbXBsZXRlZF9ub2Rlczogc2V0LAogICAgYWN0aXZlX25vZGU6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAogICAgc2tpcHBlZF9ub2RlczogT3B0aW9uYWxbc2V0XSA9IE5vbmUsCikgLT4gTm9uZToKICAgICIiIgogICAgUmVuZGVyIHRoZSBzdGVwcGVyIHJlZmxlY3RpbmcgcmVhbCBwaXBlbGluZSBwcm9ncmVzcy4KCiAgICBjb21wbGV0ZWRfbm9kZXMgOiBzZXQgb2YgTGFuZ0dyYXBoIG5vZGUgbmFtZXMgdGhhdCBoYXZlIGFscmVhZHkgZmluaXNoZWQKICAgIGFjdGl2ZV9ub2RlICAgICA6IG5vZGUgY3VycmVudGx5IGV4ZWN1dGluZyAoc2hvd24gYXMgc3Bpbm5pbmcgYmx1ZSkKICAgIHNraXBwZWRfbm9kZXMgICA6IG5vZGVzIHNraXBwZWQgYnkgYSBjb25kaXRpb25hbCBlZGdlIChzaG93biBkaW1tZWQpCiAgICAiIiIKICAgIHNraXBwZWRfbm9kZXMgPSBza2lwcGVkX25vZGVzIG9yIHNldCgpCgogICAgY29tcGxldGVkX3N0YWdlcyA9IHNldCgpCiAgICBmb3IgbiBpbiBjb21wbGV0ZWRfbm9kZXM6CiAgICAgICAgaWYgbiBpbiBfTk9ERV9TVEFHRV9NQVA6CiAgICAgICAgICAgIGNvbXBsZXRlZF9zdGFnZXMuYWRkKF9OT0RFX1NUQUdFX01BUFtuXVswXSkKCiAgICBhY3RpdmVfc3RhZ2UgPSBfTk9ERV9TVEFHRV9NQVAuZ2V0KGFjdGl2ZV9ub2RlLCAoTm9uZSwpKVswXSBpZiBhY3RpdmVfbm9kZSBlbHNlIE5vbmUKICAgIHNraXBwZWRfc3RhZ2VzID0gc2V0KCkKICAgIGZvciBuIGluIHNraXBwZWRfbm9kZXM6CiAgICAgICAgaWYgbiBpbiBfTk9ERV9TVEFHRV9NQVA6CiAgICAgICAgICAgIHNraXBwZWRfc3RhZ2VzLmFkZChfTk9ERV9TVEFHRV9NQVBbbl1bMF0pCgogICAgY2lyY2xlcyA9IFtdCiAgICBmb3IgaSwgKGljb24sIG5hbWUpIGluIGVudW1lcmF0ZShfQUxMX1NUQUdFUyk6CiAgICAgICAgaWYgbmFtZSBpbiBza2lwcGVkX3N0YWdlczoKICAgICAgICAgICAgY2xzLCBsYWJlbCA9ICJzdGVwLXBlbmRpbmciLCAi4oCUIgogICAgICAgIGVsaWYgbmFtZSBpbiBjb21wbGV0ZWRfc3RhZ2VzOgogICAgICAgICAgICBjbHMsIGxhYmVsID0gInN0ZXAtZG9uZSIsICLinJMiCiAgICAgICAgZWxpZiBuYW1lID09IGFjdGl2ZV9zdGFnZToKICAgICAgICAgICAgY2xzLCBsYWJlbCA9ICJzdGVwLWFjdGl2ZSIsIGljb24KICAgICAgICBlbHNlOgogICAgICAgICAgICBjbHMsIGxhYmVsID0gInN0ZXAtcGVuZGluZyIsIHN0cihpICsgMSkKCiAgICAgICAgbGluZV9jbHMgPSAiZG9uZSIgaWYgbmFtZSBpbiBjb21wbGV0ZWRfc3RhZ2VzIGFuZCBpIDwgbGVuKF9BTExfU1RBR0VTKSAtIDEgZWxzZSAiIgogICAgICAgIGxpbmVfaHRtbCA9IGYnPGRpdiBjbGFzcz0ic3RlcC1saW5lIHtsaW5lX2Nsc30iPjwvZGl2PicgaWYgaSA8IGxlbihfQUxMX1NUQUdFUykgLSAxIGVsc2UgIiIKCiAgICAgICAgY2lyY2xlcy5hcHBlbmQoCiAgICAgICAgICAgIGYnPGRpdiBjbGFzcz0ic3RlcC1pdGVtIj4nCiAgICAgICAgICAgIGYnPGRpdiBjbGFzcz0ic3RlcC1jaXJjbGUge2Nsc30iPntsYWJlbH08L2Rpdj4nCiAgICAgICAgICAgIGYnPGRpdiBjbGFzcz0ic3RlcC1uYW1lIj57bmFtZX08L2Rpdj4nCiAgICAgICAgICAgIGYnPC9kaXY+JyArIGxpbmVfaHRtbAogICAgICAgICkKCiAgICBzdC5tYXJrZG93bigKICAgICAgICAnPGRpdiBjbGFzcz0ic3RlcC1yb3ciPicgKyAiIi5qb2luKGNpcmNsZXMpICsgIjwvZGl2PiIsCiAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICkKCgpkZWYgcmVuZGVyKAogICAgY3VycmVudF9zdGFnZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUsCiAgICBmYWlsZWRfc3RhZ2U6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAopIC0+IE5vbmU6CiAgICAiIiJMZWdhY3kgcmVuZGVyIOKAlCBtYXBzIHN0YWdlIG5hbWUgc3RyaW5nIHRvIHByb2dyZXNzIGRpc3BsYXkuIiIiCiAgICBzdGFnZV9uYW1lcyA9IFtuIGZvciBfLCBuIGluIF9BTExfU1RBR0VTXQogICAgYWN0aXZlX2lkeCAgPSBzdGFnZV9uYW1lcy5pbmRleChjdXJyZW50X3N0YWdlKSBpZiBjdXJyZW50X3N0YWdlIGluIHN0YWdlX25hbWVzIGVsc2UgLTEKICAgIGZhaWxlZF9pZHggID0gc3RhZ2VfbmFtZXMuaW5kZXgoZmFpbGVkX3N0YWdlKSAgaWYgZmFpbGVkX3N0YWdlICBpbiBzdGFnZV9uYW1lcyBlbHNlIC0xCgogICAgY29tcGxldGVkID0gc2V0KHN0YWdlX25hbWVzWzphY3RpdmVfaWR4XSkgaWYgYWN0aXZlX2lkeCA+IDAgZWxzZSBzZXQoKQogICAgYWN0aXZlICAgID0gY3VycmVudF9zdGFnZQogICAgc2tpcHBlZDogc2V0ID0gc2V0KCkKCiAgICBjaXJjbGVzID0gW10KICAgIGZvciBpLCAoaWNvbiwgbmFtZSkgaW4gZW51bWVyYXRlKF9BTExfU1RBR0VTKToKICAgICAgICBpZiBmYWlsZWRfaWR4ID09IGk6CiAgICAgICAgICAgIGNscywgbGFiZWwgPSAic3RlcC1lcnJvciIsICLinJciCiAgICAgICAgZWxpZiBuYW1lIGluIGNvbXBsZXRlZDoKICAgICAgICAgICAgY2xzLCBsYWJlbCA9ICJzdGVwLWRvbmUiLCAi4pyTIgogICAgICAgIGVsaWYgbmFtZSA9PSBhY3RpdmU6CiAgICAgICAgICAgIGNscywgbGFiZWwgPSAic3RlcC1hY3RpdmUiLCBpY29uCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2xzLCBsYWJlbCA9ICJzdGVwLXBlbmRpbmciLCBzdHIoaSArIDEpCgogICAgICAgIGxpbmVfY2xzICA9ICJkb25lIiBpZiBuYW1lIGluIGNvbXBsZXRlZCBlbHNlICIiCiAgICAgICAgbGluZV9odG1sID0gZic8ZGl2IGNsYXNzPSJzdGVwLWxpbmUge2xpbmVfY2xzfSI+PC9kaXY+JyBpZiBpIDwgbGVuKF9BTExfU1RBR0VTKSAtIDEgZWxzZSAiIgoKICAgICAgICBjaXJjbGVzLmFwcGVuZCgKICAgICAgICAgICAgZic8ZGl2IGNsYXNzPSJzdGVwLWl0ZW0iPicKICAgICAgICAgICAgZic8ZGl2IGNsYXNzPSJzdGVwLWNpcmNsZSB7Y2xzfSI+e2xhYmVsfTwvZGl2PicKICAgICAgICAgICAgZic8ZGl2IGNsYXNzPSJzdGVwLW5hbWUiPntuYW1lfTwvZGl2PicKICAgICAgICAgICAgZic8L2Rpdj4nICsgbGluZV9odG1sCiAgICAgICAgKQoKICAgIHN0Lm1hcmtkb3duKAogICAgICAgICc8ZGl2IGNsYXNzPSJzdGVwLXJvdyI+JyArICIiLmpvaW4oY2lyY2xlcykgKyAiPC9kaXY+IiwKICAgICAgICB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlLAogICAgKQo=",
    "/content/capstone/ui/components/field_groups.py": "IiIiU21hcnQgZmllbGQgZ3JvdXBpbmcg4oCUIGNhdGVnb3Jpc2VzIGFueSBleHRyYWN0ZWQga2V5IGJ5IG5hbWUgcGF0dGVybi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCl9HUk9VUFMgPSB7CiAgICAi8J+ThCBEb2N1bWVudCBJbmZvIjogICAgIFsibnVtYmVyIiwgImludm9pY2UiLCAicmVjZWlwdCIsICJvcmRlciIsICJyZWZlcmVuY2UiLCAicG9fIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImRhdGUiLCAicGVyaW9kIiwgInRlcm1zIiwgImR1ZSIsICJleHBpcnkiLCAiaXNzdWUiLCAiYmlsbF9ubyJdLAogICAgIvCfj6IgVmVuZG9yIC8gU2VsbGVyIjogICBbInZlbmRvciIsICJzdXBwbGllciIsICJtZXJjaGFudCIsICJzZWxsZXIiLCAiY29tcGFueSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJpc3N1ZWRfYnkiLCAiYmlsbGVkX2J5IiwgImZyb21fIiwgInByb3ZpZGVyIl0sCiAgICAi8J+RpCBDdXN0b21lciAvIEJpbGwgVG8iOlsiY3VzdG9tZXIiLCAiY2xpZW50IiwgImJ1eWVyIiwgImJpbGxlZF90byIsICJiaWxsX3RvIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNoaXBfdG8iLCAic29sZF90byIsICJyZWNpcGllbnQiLCAicGF5ZWUiXSwKICAgICLwn5ONIENvbnRhY3QgJiBBZGRyZXNzIjogWyJhZGRyZXNzIiwgInN0cmVldCIsICJjaXR5IiwgInN0YXRlIiwgInppcCIsICJwb3N0YWwiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY291bnRyeSIsICJwaG9uZSIsICJtb2JpbGUiLCAidGVsIiwgImZheCIsICJlbWFpbCIsICJ3ZWJzaXRlIl0sCiAgICAi8J+SsCBGaW5hbmNpYWxzIjogICAgICAgIFsidG90YWwiLCAic3VidG90YWwiLCAic3ViX3RvdGFsIiwgIm5ldCIsICJncm9zcyIsICJhbW91bnQiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidGF4IiwgInZhdCIsICJnc3QiLCAiaHN0IiwgImRpc2NvdW50IiwgInByaWNlIiwgImNvc3QiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY3VycmVuY3kiLCAicmF0ZSIsICJiYWxhbmNlIiwgInBheW1lbnQiLCAiY2hhcmdlIiwgImZlZSJdLAogICAgIvCfj6YgQmFua2luZyI6ICAgICAgICAgICBbImliYW4iLCAiYWNjb3VudCIsICJiYW5rIiwgInN3aWZ0IiwgImJpYyIsICJyb3V0aW5nIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNvcnRfY29kZSIsICJicmFuY2giLCAiaWZzYyJdLAogICAgIvCfk50gTm90ZXMgJiBPdGhlciI6ICAgICBbXSwgICMgY2F0Y2gtYWxsIOKAlCBhbHdheXMgbGFzdAp9CgoKZGVmIGdyb3VwX2ZpZWxkcyhmaWVsZHM6IGRpY3QpIC0+IGRpY3Rbc3RyLCBkaWN0XToKICAgICIiIgogICAgUmV0dXJucyBhbiBvcmRlcmVkIGRpY3Qgb2Yge2dyb3VwX2xhYmVsOiB7ZmllbGRfa2V5OiB2YWx1ZX19LgogICAgRW1wdHkgZ3JvdXBzIGFyZSBvbWl0dGVkLgogICAgIiIiCiAgICByZXN1bHQ6IGRpY3Rbc3RyLCBkaWN0XSA9IHtnOiB7fSBmb3IgZyBpbiBfR1JPVVBTfQoKICAgIGZvciBrZXksIHZhbHVlIGluIGZpZWxkcy5pdGVtcygpOgogICAgICAgIGlmIHZhbHVlIGlzIE5vbmUgb3IgdmFsdWUgPT0gIiI6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYXNzaWduZWQgPSBGYWxzZQogICAgICAgIGtleV9sb3dlciA9IGtleS5sb3dlcigpCiAgICAgICAgZm9yIGdyb3VwLCBrZXl3b3JkcyBpbiBfR1JPVVBTLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIGdyb3VwID09ICLwn5OdIE5vdGVzICYgT3RoZXIiOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgYW55KGt3IGluIGtleV9sb3dlciBmb3Iga3cgaW4ga2V5d29yZHMpOgogICAgICAgICAgICAgICAgcmVzdWx0W2dyb3VwXVtrZXldID0gdmFsdWUKICAgICAgICAgICAgICAgIGFzc2lnbmVkID0gVHJ1ZQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBub3QgYXNzaWduZWQ6CiAgICAgICAgICAgIHJlc3VsdFsi8J+TnSBOb3RlcyAmIE90aGVyIl1ba2V5XSA9IHZhbHVlCgogICAgcmV0dXJuIHtnOiB2IGZvciBnLCB2IGluIHJlc3VsdC5pdGVtcygpIGlmIHZ9CgoKX0NVUlJFTkNZX1NZTUJPTFM6IGRpY3Rbc3RyLCBzdHJdID0gewogICAgIklOUiI6ICLigrkiLCAiVVNEIjogIiQiLCAgIkVVUiI6ICLigqwiLCAgIkdCUCI6ICLCoyIsCiAgICAiSlBZIjogIsKlIiwgIkNOWSI6ICLCpSIsICAiQVVEIjogIkEkIiwgIkNBRCI6ICJDJCIsCiAgICAiU0dEIjogIlMkIiwiQUVEIjogItivLtilICIsIlNBUiI6ICLvt7wgIiwgIkNIRiI6ICJDSEYgIiwKICAgICJIS0QiOiAiSEskIiwiTVlSIjogIlJNICIsIlRIQiI6ICLguL8iLCAiSURSIjogIlJwICIsCn0KCiMgS2V5cyBjb250YWluaW5nIHRoZXNlIHN1YnN0cmluZ3MgYXJlIE5PVCBtb25ldGFyeSBldmVuIGlmIHRoZXkgY29udGFpbgojICJ0b3RhbCIsICJhbW91bnQiLCBldGMuIOKAlCBlLmcuIHRvdGFsX3F1YW50aXR5X2l0ZW1zLCBhbW91bnRfaW5fd29yZHMKX05PTl9NT05FVEFSWV9TVUJTVFJJTkdTID0gKAogICAgInF1YW50aXR5IiwgInF0eSIsICJjb3VudCIsICJpdGVtcyIsICJ1bml0cyIsICJwY3MiLCAicGllY2VzIiwKICAgICJ3b3JkcyIsICJiYXRjaCIsICJzZXF1ZW5jZSIsICJyb3V0ZSIsICJzZXJpYWwiLCAibnVtYmVyIiwgIm5vXyIsCiAgICAiX25vIiwgInJlZiIsICJjb2RlIiwgImlkIiwKKQoKX01PTkVUQVJZX0tFWVdPUkRTID0gKAogICAgImFtb3VudCIsICJ0b3RhbCIsICJzdWJ0b3RhbCIsICJzdWJfdG90YWwiLCAidGF4IiwgInByaWNlIiwgImNvc3QiLAogICAgImJhbGFuY2UiLCAiZGlzY291bnQiLCAiZmVlIiwgImNoYXJnZSIsICJuZXQiLCAiZ3Jvc3MiLCAidmF0IiwgImdzdCIsCiAgICAic2dzdCIsICJjZ3N0IiwgImlnc3QiLCAiY2VzcyIsICJkdXR5IiwgInRhcmlmZiIsICJyYXRlIiwKKQoKCmRlZiBwcmV0dHlfbGFiZWwoa2V5OiBzdHIpIC0+IHN0cjoKICAgICIiIkNvbnZlcnQgc25ha2VfY2FzZSBrZXkgdG8gVGl0bGUgQ2FzZSBsYWJlbC4iIiIKICAgIHJldHVybiBrZXkucmVwbGFjZSgiXyIsICIgIikudGl0bGUoKQoKCmRlZiBmb3JtYXRfdmFsdWUoa2V5OiBzdHIsIHZhbHVlLCBjdXJyZW5jeTogc3RyID0gIiIpIC0+IHN0cjoKICAgICIiIkZvcm1hdCBhIHZhbHVlIGZvciBkaXNwbGF5IHVzaW5nIHRoZSBkb2N1bWVudCdzIG93biBjdXJyZW5jeSBzeW1ib2wuIiIiCiAgICBpZiB2YWx1ZSBpcyBOb25lOgogICAgICAgIHJldHVybiAi4oCUIgogICAga2V5X2xvd2VyID0ga2V5Lmxvd2VyKCkKCiAgICAjIFNraXAgbW9uZXRhcnkgZm9ybWF0dGluZyBmb3IgcXVhbnRpdHkgLyBub24tbW9uZXRhcnkgZmllbGRzCiAgICBpc19ub25fbW9uZXRhcnkgPSBhbnkocyBpbiBrZXlfbG93ZXIgZm9yIHMgaW4gX05PTl9NT05FVEFSWV9TVUJTVFJJTkdTKQoKICAgIGlzX21vbmV0YXJ5ID0gKG5vdCBpc19ub25fbW9uZXRhcnkpIGFuZCBhbnkoCiAgICAgICAgayBpbiBrZXlfbG93ZXIgZm9yIGsgaW4gX01PTkVUQVJZX0tFWVdPUkRTCiAgICApCgogICAgaWYgaXNfbW9uZXRhcnk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBzeW1ib2wgPSBfQ1VSUkVOQ1lfU1lNQk9MUy5nZXQoKGN1cnJlbmN5IG9yICIiKS51cHBlcigpLnN0cmlwKCksICIiKQogICAgICAgICAgICAjIEZhbGwgYmFjayB0byB0aGUgcmF3IGN1cnJlbmN5IHN0cmluZyBpZiBzeW1ib2wgbm90IG1hcHBlZAogICAgICAgICAgICBpZiBub3Qgc3ltYm9sIGFuZCBjdXJyZW5jeToKICAgICAgICAgICAgICAgIHN5bWJvbCA9IGN1cnJlbmN5LnN0cmlwKCkgKyAiICIKICAgICAgICAgICAgcmV0dXJuIGYie3N5bWJvbH17ZmxvYXQodmFsdWUpOiwuMmZ9IgogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVHlwZUVycm9yKToKICAgICAgICAgICAgcGFzcwoKICAgIHJldHVybiBzdHIodmFsdWUpCg==",
    "/content/capstone/ui/components/latency_panel.py": "IiIiTGF0ZW5jeSBicmVha2Rvd24gcGFuZWwg4oCUIHBhcnNlcyB0aW1pbmcgZnJvbSBwcm9jZXNzaW5nIG5vdGVzIGFuZCByZW5kZXJzIGEgYmFyIGNoYXJ0LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmltcG9ydCBzdHJlYW1saXQgYXMgc3QKCiMgTWFwcyBkaXNwbGF5IGxhYmVsIOKGkiBlbW9qaSBpY29uCl9TVEFHRV9JQ09OUyA9IHsKICAgICJpbmdlc3QiOiAgICAgICAi8J+TpSIsCiAgICAiZXh0cmFjdCI6ICAgICAgIvCflI0iLAogICAgIm9jcl9mYWxsYmFjayI6ICLwn5OEIiwKICAgICJ2YWxpZGF0ZSI6ICAgICAi4pyFIiwKICAgICJjb3JyZWN0IjogICAgICAi8J+UhCIsCiAgICAic3RvcmUiOiAgICAgICAgIvCfkr4iLAogICAgInJldmlld19xdWV1ZSI6ICLwn5GB77iPIiwKfQoKX1NUQUdFX0xBQkVMUyA9IHsKICAgICJpbmdlc3QiOiAgICAgICAiSW5nZXN0aW9uIiwKICAgICJleHRyYWN0IjogICAgICAiVkxNIEV4dHJhY3Rpb24iLAogICAgIm9jcl9mYWxsYmFjayI6ICJPQ1IgRmFsbGJhY2siLAogICAgInZhbGlkYXRlIjogICAgICJWYWxpZGF0aW9uIiwKICAgICJjb3JyZWN0IjogICAgICAiQXV0by1Db3JyZWN0aW9uIiwKICAgICJzdG9yZSI6ICAgICAgICAiU3RvcmFnZSIsCiAgICAicmV2aWV3X3F1ZXVlIjogIlJldmlldyBRdWV1ZSIsCn0KCgpkZWYgX3BhcnNlX3RpbWluZ3Mobm90ZXM6IGxpc3Rbc3RyXSwgdGltaW5nc19kaWN0OiBkaWN0IHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiIKICAgIEV4dHJhY3QgcGVyLW5vZGUgc2Vjb25kcyBmcm9tIHByb2Nlc3Npbmcgbm90ZXMuCiAgICBOb3RlcyBjb250YWluIHBhdHRlcm5zIGxpa2UgICfCtyDij7EgMy4ycycgIG9yICAn4o+xIFRvdGFsIHBpcGVsaW5lIHRpbWU6IDQuOHMnCiAgICBGYWxscyBiYWNrIHRvIHRpbWluZ3NfZGljdCAoc3RvcmVkIGluIGZpbmFsX2RhdGEpIGlmIGF2YWlsYWJsZS4KICAgICIiIgogICAgaWYgdGltaW5nc19kaWN0OgogICAgICAgIHJldHVybiB7azogdiBmb3IgaywgdiBpbiB0aW1pbmdzX2RpY3QuaXRlbXMoKSBpZiBrICE9ICJ0b3RhbCJ9CgogICAgIyBQYXJzZSBmcm9tIG5vdGVzIHRleHQKICAgIHJlc3VsdCA9IHt9CiAgICBzdGFnZV9rZXlzID0gbGlzdChfU1RBR0VfTEFCRUxTLmtleXMoKSkKCiAgICBmb3Igbm90ZSBpbiBub3RlczoKICAgICAgICBtID0gcmUuc2VhcmNoKHIi4o+xXHMqKFtcZC5dKylzIiwgbm90ZSkKICAgICAgICBpZiBub3QgbToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzZWNzID0gZmxvYXQobS5ncm91cCgxKSkKICAgICAgICBub3RlX2xvd2VyID0gbm90ZS5sb3dlcigpCiAgICAgICAgZm9yIGtleSBpbiBzdGFnZV9rZXlzOgogICAgICAgICAgICBsYWJlbCA9IF9TVEFHRV9MQUJFTFNba2V5XS5sb3dlcigpCiAgICAgICAgICAgICMgbWF0Y2ggYnkga2V5d29yZCBpbiBub3RlCiAgICAgICAgICAgIGlmIGtleSBpbiBub3RlX2xvd2VyIG9yIGFueSh3IGluIG5vdGVfbG93ZXIgZm9yIHcgaW4gbGFiZWwuc3BsaXQoKSk6CiAgICAgICAgICAgICAgICByZXN1bHRba2V5XSA9IHNlY3MKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiByZW5kZXIobm90ZXM6IGxpc3Rbc3RyXSwgdGltaW5nc19kaWN0OiBkaWN0IHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6CiAgICAiIiIKICAgIFJlbmRlciBhIGxhdGVuY3kgYnJlYWtkb3duIHBhbmVsLgoKICAgIG5vdGVzICAgICAgICA6IHByb2Nlc3Npbmdfbm90ZXMgbGlzdCBmcm9tIHRoZSBwaXBlbGluZSBzdGF0ZQogICAgdGltaW5nc19kaWN0IDogb3B0aW9uYWwgcHJlLXBhcnNlZCB0aW1pbmdzIGRpY3QgZnJvbSBmaW5hbF9kYXRhWyd0aW1pbmdzJ10KICAgICIiIgogICAgIyBFeHRyYWN0IHRvdGFsIGZyb20gbm90ZXMKICAgIHRvdGFsX3MgPSBOb25lCiAgICBmb3Igbm90ZSBpbiBub3RlczoKICAgICAgICBtID0gcmUuc2VhcmNoKHIiVG90YWwgcGlwZWxpbmUgdGltZTpccyooW1xkLl0rKXMiLCBub3RlKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIHRvdGFsX3MgPSBmbG9hdChtLmdyb3VwKDEpKQogICAgICAgICAgICBicmVhawoKICAgICMgUGFyc2UgcGVyLXN0YWdlIHRpbWluZ3MKICAgIHRpbWluZ3MgPSBfcGFyc2VfdGltaW5ncyhub3RlcywgdGltaW5nc19kaWN0KQoKICAgIGlmIG5vdCB0aW1pbmdzIGFuZCB0b3RhbF9zIGlzIE5vbmU6CiAgICAgICAgc3QuY2FwdGlvbigiTm8gdGltaW5nIGRhdGEgYXZhaWxhYmxlIGZvciB0aGlzIGRvY3VtZW50LiIpCiAgICAgICAgcmV0dXJuCgogICAgc3QubWFya2Rvd24oIioq4o+xIExhdGVuY3kgQnJlYWtkb3duKioiKQoKICAgIGlmIHRpbWluZ3M6CiAgICAgICAgbWF4X3QgPSBtYXgodGltaW5ncy52YWx1ZXMoKSkgaWYgdGltaW5ncyBlbHNlIDEuMAoKICAgICAgICBmb3Iga2V5LCBsYWJlbCBpbiBfU1RBR0VfTEFCRUxTLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIGtleSBub3QgaW4gdGltaW5nczoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNlY3MgID0gdGltaW5nc1trZXldCiAgICAgICAgICAgIGljb24gID0gX1NUQUdFX0lDT05TLmdldChrZXksICLilqrvuI8iKQogICAgICAgICAgICBwY3QgICA9IHNlY3MgLyBtYXgobWF4X3QsIDAuMDEpCgogICAgICAgICAgICBjb2xfbGFiZWwsIGNvbF9iYXIsIGNvbF92YWwgPSBzdC5jb2x1bW5zKFsyLCA1LCAxXSkKICAgICAgICAgICAgd2l0aCBjb2xfbGFiZWw6CiAgICAgICAgICAgICAgICBzdC5tYXJrZG93bigKICAgICAgICAgICAgICAgICAgICBmIjxkaXYgc3R5bGU9J2ZvbnQtc2l6ZTouODJyZW07Y29sb3I6IzM3NDE1MTtwYWRkaW5nLXRvcDo0cHg7Jz4iCiAgICAgICAgICAgICAgICAgICAgZiJ7aWNvbn0ge2xhYmVsfTwvZGl2PiIsCiAgICAgICAgICAgICAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgd2l0aCBjb2xfYmFyOgogICAgICAgICAgICAgICAgIyBDb2xvdXIgYmFyOiByZWQgZm9yIHNsb3dlc3QsIGdyZWVuIGZvciBmYXN0ZXN0CiAgICAgICAgICAgICAgICBodWUgPSBpbnQoMTIwICogKDEgLSBwY3QpKSAgICMgMD1yZWQsIDEyMD1ncmVlbgogICAgICAgICAgICAgICAgYmFyX2NvbG9yID0gZiJoc2woe2h1ZX0sNzAlLDQ1JSkiCiAgICAgICAgICAgICAgICBiYXJfd2lkdGggID0gbWF4KGludChwY3QgKiAxMDApLCA0KQogICAgICAgICAgICAgICAgc3QubWFya2Rvd24oCiAgICAgICAgICAgICAgICAgICAgZiI8ZGl2IHN0eWxlPSdiYWNrZ3JvdW5kOiNmMWY1Zjk7Ym9yZGVyLXJhZGl1czo0cHg7bWFyZ2luLXRvcDo2cHg7Jz4iCiAgICAgICAgICAgICAgICAgICAgZiI8ZGl2IHN0eWxlPSdiYWNrZ3JvdW5kOntiYXJfY29sb3J9O3dpZHRoOntiYXJfd2lkdGh9JTsiCiAgICAgICAgICAgICAgICAgICAgZiJoZWlnaHQ6MTBweDtib3JkZXItcmFkaXVzOjRweDsnPjwvZGl2PjwvZGl2PiIsCiAgICAgICAgICAgICAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgd2l0aCBjb2xfdmFsOgogICAgICAgICAgICAgICAgc3QubWFya2Rvd24oCiAgICAgICAgICAgICAgICAgICAgZiI8ZGl2IHN0eWxlPSdmb250LXNpemU6LjgycmVtO2ZvbnQtd2VpZ2h0OjYwMDtjb2xvcjojMWEyNzQ0OyIKICAgICAgICAgICAgICAgICAgICBmInRleHQtYWxpZ246cmlnaHQ7cGFkZGluZy10b3A6NHB4Oyc+e3NlY3N9czwvZGl2PiIsCiAgICAgICAgICAgICAgICAgICAgdW5zYWZlX2FsbG93X2h0bWw9VHJ1ZSwKICAgICAgICAgICAgICAgICkKCiAgICBpZiB0b3RhbF9zIGlzIG5vdCBOb25lOgogICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICBmIjxkaXYgc3R5bGU9J21hcmdpbi10b3A6MTBweDtwYWRkaW5nOjEwcHggMTRweDsiCiAgICAgICAgICAgIGYiYmFja2dyb3VuZDojZjBmZGY0O2JvcmRlci1yYWRpdXM6OHB4O2JvcmRlci1sZWZ0OjRweCBzb2xpZCAjMTZhMzRhOyc+IgogICAgICAgICAgICBmIjxzcGFuIHN0eWxlPSdmb250LXNpemU6Ljg1cmVtO2NvbG9yOiMxNjY1MzQ7Zm9udC13ZWlnaHQ6NzAwOyc+IgogICAgICAgICAgICBmIuKPsSBUb3RhbCBwaXBlbGluZSB0aW1lOiA8c3Ryb25nPnt0b3RhbF9zfXM8L3N0cm9uZz48L3NwYW4+IgogICAgICAgICAgICBmIjwvZGl2PiIsCiAgICAgICAgICAgIHVuc2FmZV9hbGxvd19odG1sPVRydWUsCiAgICAgICAgKQogICAgZWxpZiB0aW1pbmdzOgogICAgICAgIHRvdGFsX2NvbXB1dGVkID0gcm91bmQoc3VtKHRpbWluZ3MudmFsdWVzKCkpLCAyKQogICAgICAgIHN0Lm1hcmtkb3duKAogICAgICAgICAgICBmIjxkaXYgc3R5bGU9J21hcmdpbi10b3A6MTBweDtwYWRkaW5nOjEwcHggMTRweDsiCiAgICAgICAgICAgIGYiYmFja2dyb3VuZDojZjBmZGY0O2JvcmRlci1yYWRpdXM6OHB4O2JvcmRlci1sZWZ0OjRweCBzb2xpZCAjMTZhMzRhOyc+IgogICAgICAgICAgICBmIjxzcGFuIHN0eWxlPSdmb250LXNpemU6Ljg1cmVtO2NvbG9yOiMxNjY1MzQ7Zm9udC13ZWlnaHQ6NzAwOyc+IgogICAgICAgICAgICBmIuKPsSBNZWFzdXJlZCBzdGFnZXMgdG90YWw6IHt0b3RhbF9jb21wdXRlZH1zPC9zcGFuPiIKICAgICAgICAgICAgZiI8L2Rpdj4iLAogICAgICAgICAgICB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlLAogICAgICAgICkK",
    "/content/capstone/data/vendors.txt": "QWNtZSBDb3JwCkdsb2JhbCBTdXBwbGllcyBJbmMKVGVjaCBTb2x1dGlvbnMgTHRkCk9mZmljZSBEZXBvdApBbWF6b24gQnVzaW5lc3MKU3RhcGxlcwpEZWxsIFRlY2hub2xvZ2llcwpNaWNyb3NvZnQgQ29ycG9yYXRpb24KQWRvYmUgU3lzdGVtcwpTYWxlc2ZvcmNlIEluYwpHb29nbGUgTExDCkFwcGxlIEluYwpJQk0gQ29ycG9yYXRpb24KT3JhY2xlIENvcnBvcmF0aW9uClNBUCBTRQpDaXNjbyBTeXN0ZW1zCkhQIEluYwpMZW5vdm8gR3JvdXAKU2Ftc3VuZyBFbGVjdHJvbmljcwpJbnRlbCBDb3Jwb3JhdGlvbgpOdmlkaWEgQ29ycG9yYXRpb24KWm9vbSBWaWRlbyBDb21tdW5pY2F0aW9ucwpTbGFjayBUZWNobm9sb2dpZXMKRHJvcGJveCBJbmMKRG9jdVNpZ24gSW5jCkF0bGFzc2lhbiBDb3Jwb3JhdGlvbgpTZXJ2aWNlTm93IEluYwpXb3JrZGF5IEluYwpQYXlsb2NpdHkgQ29ycApBRFAgTExDCkZlZEV4IENvcnBvcmF0aW9uClVQUyBTdXBwbHkgQ2hhaW4gU29sdXRpb25zCkRITCBFeHByZXNzClcgVyBHcmFpbmdlciBJbmMKRmFzdGVuYWwgQ29tcGFueQpNY01hc3Rlci1DYXJyIFN1cHBseQpVbGluZSBJbmMKUXVpbGwgQ29ycG9yYXRpb24KR2xvYmFsIEluZHVzdHJpYWwgQ29tcGFueQpJbnNpZ2h0IERpcmVjdApDRFcgQ29ycG9yYXRpb24KU0hJIEludGVybmF0aW9uYWwKQ29ubmVjdGlvbiBJbmMKWm9uZXMgTExDClBDIE1hbGwgSW5jClNvZnRjaG9pY2UgQ29ycG9yYXRpb24KSW5zaWdodCBFbnRlcnByaXNlcwpQcmVzaWRpbyBJbmMKZVBsdXMgVGVjaG5vbG9neQpXb3JsZCBXaWRlIFRlY2hub2xvZ3kK",
    "/content/capstone/.streamlit/config.toml": "W3RoZW1lXQpiYXNlICAgICAgICAgICAgICAgICAgICAgPSAibGlnaHQiCnByaW1hcnlDb2xvciAgICAgICAgICAgICA9ICIjMjU2M2ViIgpiYWNrZ3JvdW5kQ29sb3IgICAgICAgICAgPSAiI2YwZjRmOCIKc2Vjb25kYXJ5QmFja2dyb3VuZENvbG9yID0gIiMxYTI3NDQiCnRleHRDb2xvciAgICAgICAgICAgICAgICA9ICIjMWEyNzQ0Igpmb250ICAgICAgICAgICAgICAgICAgICAgPSAic2FucyBzZXJpZiIK",
}

for path, b64 in FILES.items():
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
    print(f'  Written: {path}')

# Create empty data directories
for d in ['/content/capstone/data/samples', '/content/capstone/data/output',
          '/content/capstone/data/chroma']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print('\nAll project files written successfully.')


## Step 4 — Initialize & Verify


In [ ]:
import sys
sys.path.insert(0, '/content/capstone')

# Write the .env so config.py picks up env vars
import pathlib
env_vars = {
    'VLM_BACKEND':       os.environ.get('VLM_BACKEND', 'gemini'),
    'GEMINI_API_KEY':    os.environ.get('GEMINI_API_KEY', ''),
    'GEMINI_MODEL':      os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash'),
    'ANTHROPIC_API_KEY': os.environ.get('ANTHROPIC_API_KEY', ''),
    'CLAUDE_MODEL':      os.environ.get('CLAUDE_MODEL', 'claude-sonnet-4-6'),
    'SQLITE_PATH':       '/content/capstone/data/invoices.db',
    'CHROMA_PERSIST_DIR':'/content/capstone/data/chroma',
}
pathlib.Path('/content/capstone/.env').write_text(
    '\n'.join(f'{k}={v}' for k, v in env_vars.items()) + '\n'
)

# Import core modules
import config
from database import storage
from graph.workflow import build_graph, process_document
from utils.vendor_matcher import VendorMatcher

# Initialize database
storage.init_db()
print(f'SQLite DB: {config.SQLITE_PATH}')

# Initialize vendor matcher (loads vendors.txt)
vm = VendorMatcher()
print(f'Vendor matcher ready — {vm._col.count()} vendors loaded.')

# Build LangGraph
GRAPH = build_graph()
print(f'LangGraph workflow compiled.')
print(f'VLM backend: {config.VLM_BACKEND}')


## Step 5 — Upload & Process a Document

Upload any invoice, receipt, or form: **PDF, PNG, JPG, or TIFF**.

> If you don't have a document handy, the next cell generates a simple synthetic invoice image.


In [ ]:
# [OPTIONAL] Generate a synthetic test invoice if you don't have a file to upload
from PIL import Image, ImageDraw, ImageFont
import pathlib

def make_test_invoice():
    img = Image.new('RGB', (794, 1123), color='white')  # A4 at 96dpi
    d = ImageDraw.Draw(img)
    lines = [
        (50,  40,  'INVOICE',                             28),
        (50,  90,  'Acme Corp',                           18),
        (50, 115,  '123 Business St, San Francisco, CA',  12),
        (50, 160,  'Invoice Number:  INV-2024-001',        13),
        (50, 180,  'Invoice Date:    2024-03-15',           13),
        (50, 200,  'Due Date:        2024-04-15',           13),
        (50, 230,  'Bill To:  Tech Solutions Ltd',          13),
        (50, 260,  'Description                Qty  Price   Total',  13),
        (50, 280,  '--------------------------------------------',   11),
        (50, 300,  'Cloud Hosting Services     1    500.00   500.00',  12),
        (50, 320,  'Support Package            2    150.00   300.00',  12),
        (50, 340,  'Setup Fee                  1     75.00    75.00',  12),
        (50, 375,  'Subtotal:  875.00',                    13),
        (50, 395,  'Tax (10%):  87.50',                    13),
        (50, 420,  'TOTAL:  962.50 USD',                   16),
        (50, 460,  'Payment Terms: Net 30',                12),
        (50, 480,  'IBAN: GB29NWBK60161331926819',          12),
    ]
    for x, y, text, size in lines:
        try:
            font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', size)
        except Exception:
            font = ImageFont.load_default()
        d.text((x, y), text, fill='black', font=font)
    path = '/content/test_invoice.png'
    img.save(path, dpi=(150, 150))
    print(f'Synthetic invoice saved: {path}')
    return path

TEST_FILE = make_test_invoice()


In [ ]:
# Upload your own document (PDF / PNG / JPG / TIFF)
# Skip this cell if you want to use the synthetic test invoice above
from google.colab import files
print('Select your invoice file to upload:')
uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]
    dest = f'/content/{filename}'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    INPUT_FILE = dest
    print(f'Uploaded: {INPUT_FILE}')
else:
    INPUT_FILE = TEST_FILE   # fall back to synthetic invoice
    print(f'No file uploaded — using: {INPUT_FILE}')


In [ ]:
# ── Run the full LangGraph pipeline ──────────────────────────────────────
import time

print(f'Processing: {INPUT_FILE}')
print(f'Backend:    {config.VLM_BACKEND}')
print('-' * 50)

t0     = time.time()
result = process_document(INPUT_FILE)
elapsed = round(time.time() - t0, 1)

print('-' * 50)
print(f'Done in {elapsed}s')
print()

# Show pipeline log
for note in result['processing_notes']:
    print(note)


## Step 6 — View Results


In [ ]:
import pandas as pd, json
from IPython.display import display

ext        = result['extraction']
val        = result['validation']
doc        = ext.extracted_data
status     = result['final_status']
timings    = result.get('timings', {})

print('=' * 60)
print(f'  STATUS      : {status.value.upper()}')
print(f'  Doc Type    : {doc.doc_type}'  + (f' ({doc.doc_subtype})' if doc.doc_subtype else ''))
print(f'  Confidence  : {ext.confidence:.0%}')
print(f'  Document ID : {result["document_id"]}')
print(f'  Corrections : {result["correction_attempts"]}')
print(f'  OCR Used    : {ext.ocr_used}')
print('=' * 60)

# Extracted fields
print('\nExtracted Fields:')
if doc.fields:
    df_fields = pd.DataFrame(list(doc.fields.items()), columns=['Field', 'Value'])
    display(df_fields)
else:
    print('  (no fields extracted)')

# Line items
if doc.line_items:
    print('\nLine Items:')
    display(pd.DataFrame(doc.line_items))

# Validation details
if val and val.field_validations:
    print('\nValidation:')
    vdf = pd.DataFrame([v.model_dump() for v in val.field_validations])
    display(vdf[['field', 'status', 'message']].dropna(subset=['message']))

# Timing breakdown
if timings:
    print('\nTiming Breakdown:')
    for k, v in timings.items():
        print(f'  {k:<20} {v}s')

# Extraction notes
if doc.extraction_notes:
    print(f'\nNotes: {doc.extraction_notes}')


In [ ]:
# Show the preprocessed document image
import matplotlib.pyplot as plt

images = result['images']
n = len(images)
fig, axes = plt.subplots(1, n, figsize=(8 * n, 11))
if n == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, images)):
    ax.imshow(img)
    ax.set_title(f'Page {i + 1} — preprocessed (deskewed + contrast enhanced)')
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Export extracted data as formatted JSON
import json
from database import storage as db

doc_id = result['document_id']
print(f'Extracted JSON for document {doc_id}:')
print(db.export_json(doc_id))


## Step 7 — Browse the Database

All processed documents are stored in SQLite with a full audit trail.


In [ ]:
import pandas as pd, json
from database import storage as db
from IPython.display import display

records = db.list_records()
stats   = db.get_stats()

print(f'Total documents : {stats["total"]}')
print(f'Success rate    : {stats["success_rate"]}%')
print(f'Pending review  : {stats["pending"]}')
print(f'Processed today : {stats["today"]}')
if stats['by_type']:
    print(f'By type         : {stats["by_type"]}')

if records:
    print()
    df = pd.DataFrame(records)
    display(df[['document_id', 'doc_type', 'validation_status',
                'extraction_confidence', 'created_at']].head(20))


In [ ]:
# View the full audit trail for the last processed document
from database import storage as db

trail = db.get_audit_trail(result['document_id'])
print(f'Audit trail ({len(trail)} events):')
for event in trail:
    details = json.loads(event.get('details') or '{}')
    print(f'  [{event["timestamp"]}] {event["event"]:20s}  {details}')


In [ ]:
# Export all records to CSV and download
from google.colab import files
from database import storage as db

csv_content = db.export_all_csv()
if csv_content:
    csv_path = '/content/invoice_records.csv'
    with open(csv_path, 'w') as f:
        f.write(csv_content)
    files.download(csv_path)
    print('Downloaded invoice_records.csv')
else:
    print('No records to export yet.')


In [ ]:
# Download the SQLite database
from google.colab import files
import config
files.download(config.SQLITE_PATH)
print(f'Downloaded {config.SQLITE_PATH}')


## Step 8 — Batch Process Multiple Documents

Upload multiple files and process them all in sequence.


In [ ]:
from google.colab import files as colab_files
from graph.workflow import process_document
import pandas as pd, time
from IPython.display import display

print('Upload multiple invoice files:')
uploaded = colab_files.upload()

if not uploaded:
    print('No files uploaded — skipping batch run.')
else:
    results = []
    for fname, data in uploaded.items():
        path = f'/content/{fname}'
        with open(path, 'wb') as f:
            f.write(data)

        print(f'\nProcessing {fname}...')
        t0  = time.time()
        res = process_document(path)
        elapsed = round(time.time() - t0, 1)

        ext = res['extraction']
        doc = ext.extracted_data if ext else None
        results.append({
            'file':        fname,
            'doc_type':    doc.doc_type if doc else 'unknown',
            'status':      res['final_status'].value if res['final_status'] else 'unknown',
            'confidence':  f"{ext.confidence:.0%}" if ext else 'n/a',
            'vendor':      doc.vendor_name if doc else '',
            'total':       doc.total_amount if doc else '',
            'corrections': res['correction_attempts'],
            'time_s':      elapsed,
        })
        print(f'  Done in {elapsed}s — {res["final_status"].value}')

    print('\nBatch Summary:')
    display(pd.DataFrame(results))


## Step 9 — Launch Full Streamlit UI (Optional)

The full UI includes:
- **Dashboard** — live stats and doc-type breakdown
- **Process Document** — drag-and-drop upload with real-time pipeline status
- **Review Queue** — approve/reject low-confidence extractions
- **History** — searchable record browser
- **Settings** — switch VLM backend without restarting

> **Login credentials:** `demo` / `demo`

The cell below starts Streamlit and opens a **public Cloudflare Tunnel URL**.
Share the URL with teammates — it stays active while this Colab session runs.


In [ ]:
import subprocess, threading, time, re, sys

# 1. Download cloudflared (static binary, no account needed)
!wget -q -O /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

# 2. Start Streamlit in the background
streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    cwd='/content/capstone',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print('Starting Streamlit...')
time.sleep(5)

# 3. Open Cloudflare Tunnel and capture the public URL
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group()
        break
    time.sleep(1)

if public_url:
    print(f"""
╔══════════════════════════════════════════════════════════╗
║  Doc Agent is LIVE                                       ║
║                                                          ║
║  URL  : {public_url:<46}║
║  Login: demo / demo                                      ║
║                                                          ║
║  Share with your team. Active while session runs.        ║
╚══════════════════════════════════════════════════════════╝
""")
else:
    print('Could not get Cloudflare URL. Trying localtunnel...')
    import urllib.request
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
    print(f'When prompted for a password, enter: {ip}')
    !npm install -g localtunnel --silent
    !lt --port 8501


In [ ]:
# Restart Streamlit (run this cell if you edit source files and need a reload)
import subprocess, sys, time

!pkill -f streamlit || true
time.sleep(2)

subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    cwd='/content/capstone',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f'Streamlit restarted — refresh the tunnel URL.')


## Step 10 — Vendor Registry Management

Add your own vendor names to improve vendor-name matching accuracy.


In [ ]:
from utils.vendor_matcher import VendorMatcher

vm = VendorMatcher()
print(f'Current vendor count: {vm._col.count()}')

# Add custom vendors
MY_VENDORS = [
    # 'Your Vendor Name',
    # 'Another Supplier Ltd',
]
if MY_VENDORS:
    added = vm.add_vendors(MY_VENDORS)
    print(f'Added {added} vendor(s). Total: {vm._col.count()}')

# Test matching
test_name = 'Acme Corporation'
known, matched = vm.is_known_vendor(test_name)
print(f'Match test: "{test_name}" -> known={known}, matched="{matched}"')
